In [1]:
# Developed with assistance from Claude (Anthropic).
# Adım 1a — orbitrary.yml dosyasını proje köküne yazar (belge 17.1 madde 3).
from pathlib import Path
from datetime import datetime

KOK = Path.home() / "orbitrary"
# Klasör yapısı önceki sohbette kuruldu; burada yalnız varlığı kontrol edilir
for alt in ["veri", "temiz", "kod", "ara", "sonuc", "docs", "makale", "kayit"]:
    assert (KOK / alt).is_dir(), f"Eksik klasör: {KOK / alt}"

# Paket listesi belge 17.1 madde 3'ten birebir alındı.
# "pip" yalnız alttaki pip: bölümünün çalışması için gerekli (conda kuralı).
# nodefaults: Anaconda'nın kendi kanalı karışmasın, her şey conda-forge'dan gelsin.
yml = """# Developed with assistance from Claude (Anthropic).
# Orbitrary v1.2 ortamı — belge Bölüm 17.1 madde 3
name: orbitrary
channels:
  - conda-forge
  - nodefaults
dependencies:
  - python=3.12
  - numpy
  - scipy
  - pandas
  - pyarrow
  - gdal
  - rasterio
  - pyproj
  - geopandas
  - pyogrio
  - shapely
  - xarray
  - netcdf4
  - h5py
  - libgdal-hdf4
  - libgdal-hdf5
  - libgdal-netcdf
  - earthaccess
  - gudhi
  - matplotlib
  - jupyterlab
  - pytest
  - tqdm
  - pip
  - pip:
      - inflate64
"""

hedef = KOK / "orbitrary.yml"
hedef.write_text(yml, encoding="utf-8", newline="\n")

# Geri okuyup doğrula
geri = hedef.read_text(encoding="utf-8")
assert geri == yml, "Dosya yazılan metinle aynı değil"
beklenen = ["python=3.12", "numpy", "scipy", "pandas", "pyarrow", "gdal", "rasterio",
            "pyproj", "geopandas", "pyogrio", "shapely", "xarray", "netcdf4", "h5py",
            "libgdal-hdf4", "libgdal-hdf5", "libgdal-netcdf", "earthaccess", "gudhi",
            "matplotlib", "jupyterlab", "pytest", "tqdm", "inflate64"]
satirlar = [s.strip().lstrip("- ").strip() for s in geri.splitlines()]
eksik = [p for p in beklenen if p not in satirlar]
assert not eksik, f"Eksik paket: {eksik}"

# Gün sonu raporu için adım kaydı
with open(KOK / "kayit" / "13_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(f"{datetime.now():%Y-%m-%d %H:%M:%S}  1a orbitrary.yml yazildi ({hedef.stat().st_size} bayt, {len(beklenen)} paket)\n")

print("OK  ->", hedef)
print(f"Boyut: {hedef.stat().st_size} bayt, paket sayısı: {len(beklenen)} (+ pip)")
print("-" * 40)
print(geri)

OK  -> C:\Users\USER\orbitrary\orbitrary.yml
Boyut: 506 bayt, paket sayısı: 24 (+ pip)
----------------------------------------
# Developed with assistance from Claude (Anthropic).
# Orbitrary v1.2 ortamı — belge Bölüm 17.1 madde 3
name: orbitrary
channels:
  - conda-forge
  - nodefaults
dependencies:
  - python=3.12
  - numpy
  - scipy
  - pandas
  - pyarrow
  - gdal
  - rasterio
  - pyproj
  - geopandas
  - pyogrio
  - shapely
  - xarray
  - netcdf4
  - h5py
  - libgdal-hdf4
  - libgdal-hdf5
  - libgdal-netcdf
  - earthaccess
  - gudhi
  - matplotlib
  - jupyterlab
  - pytest
  - tqdm
  - pip
  - pip:
      - inflate64



In [1]:
# Developed with assistance from Claude (Anthropic).
# Adım D0 — ortam testini kod\d0_ortam.py olarak yazar ve çalıştırır (belge 17.1 madde 4).
from pathlib import Path
import runpy, sys

KOK = Path.home() / "orbitrary"
assert "orbitrary" in Path(sys.executable).parts, (
    f"Yanlış çekirdek: {sys.executable}\nJupyterLab'ı (orbitrary) yazan pencereden açmalısın.")

kod = r'''# Developed with assistance from Claude (Anthropic).
# D0 ortam testi — Orbitrary v1.2 belge Bölüm 17.1 madde 4.
# Ne yapar: paket sürümleri, GDAL sürücüleri (HDF4, HDF5, netCDF, GTiff, GPKG,
# ESRI Shapefile, OpenFileGDB), küçük yaz/oku denemeleri, GUDHI CubicalComplex
# (cevabı bilinen örnek), earthaccess login(strategy="netrc") ve bir CMR araması.
# Çıktı: kayit\ortam.json. Bir test geçmezse silinmez; FAIL olarak yazılır.
import sys, os, json, platform, shutil, subprocess, ctypes
from pathlib import Path
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError
import numpy as np

KOK = Path.home() / "orbitrary"
KAYIT = KOK / "kayit"
GECICI = KOK / "ara" / "d0_gecici"
shutil.rmtree(GECICI, ignore_errors=True)
GECICI.mkdir(parents=True)

sonuc = {"adim": "D0", "tarih": datetime.now().isoformat(timespec="seconds"), "testler": {}}
T = sonuc["testler"]

def kaydet(ad, gecti, detay=""):
    T[ad] = {"durum": "PASS" if gecti else "FAIL", "detay": str(detay)}
    print(("[PASS] " if gecti else "[FAIL] ") + ad + ": " + str(detay))

def dene(ad, fonk):
    # Test hata verirse FAIL olarak kaydedilir, diğer testler devam eder
    try:
        gecti, detay = fonk()
        kaydet(ad, gecti, detay)
    except Exception as e:
        kaydet(ad, False, type(e).__name__ + ": " + str(e))

# ---------- 0) Doğru ortam ----------
exe = Path(sys.executable)
kaydet("ortam_python", ("envs" in exe.parts and "orbitrary" in exe.parts and sys.version_info[:2] == (3, 12)),
       str(exe) + " | Python " + sys.version.split()[0])

# ---------- 1) Makine bilgisi (makale Implementation, t11_sure için) ----------
def ram_gb():
    try:
        class M(ctypes.Structure):
            _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong),
                        ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                        ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                        ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                        ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]
        m = M(); m.dwLength = ctypes.sizeof(M)
        ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(m))
        return round(m.ullTotalPhys / 1e9, 1)
    except Exception:
        return None

conda_surum = None
if os.environ.get("CONDA_EXE"):
    try:
        conda_surum = subprocess.run([os.environ["CONDA_EXE"], "--version"], capture_output=True,
                                     text=True, timeout=60).stdout.strip()
    except Exception:
        pass
disk = shutil.disk_usage(KOK)
sonuc["makine"] = {"platform": platform.platform(), "islemci": platform.processor(),
                   "cekirdek": os.cpu_count(), "ram_gb": ram_gb(),
                   "disk_bos_gb": round(disk.free / 1e9, 1), "conda": conda_surum,
                   "python": sys.version, "python_yolu": str(exe)}
print("Makine:", sonuc["makine"])

# ---------- 2) Paket sürümleri ----------
# pip/metadata adı -> içe aktarma adı (metadata yoksa modülün __version__'ı okunur)
paketler = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "pyarrow": "pyarrow",
            "GDAL": "osgeo.gdal", "rasterio": "rasterio", "pyproj": "pyproj", "geopandas": "geopandas",
            "pyogrio": "pyogrio", "shapely": "shapely", "xarray": "xarray", "netCDF4": "netCDF4",
            "h5py": "h5py", "earthaccess": "earthaccess", "gudhi": "gudhi", "matplotlib": "matplotlib",
            "jupyterlab": "jupyterlab", "pytest": "pytest", "tqdm": "tqdm", "inflate64": "inflate64"}
surumler = {}
for ad, modul in paketler.items():
    try:
        surumler[ad] = version(ad)
    except PackageNotFoundError:
        try:
            m = __import__(modul, fromlist=["x"])
            surumler[ad] = getattr(m, "__version__", "bilinmiyor")
        except Exception as e:
            surumler[ad] = None
    # Paket gerçekten içe aktarılabiliyor mu
    try:
        __import__(modul, fromlist=["x"])
        aktarildi = True
    except Exception as e:
        aktarildi = False
    kaydet("paket_" + ad, surumler[ad] is not None and aktarildi, surumler[ad])
sonuc["paket_surumleri"] = surumler

# Conda paketleri (GDAL eklentileri pip'te görünmez; conda-meta'dan okunur)
meta = {}
for f in (Path(sys.prefix) / "conda-meta").glob("*.json"):
    try:
        d = json.loads(f.read_text(encoding="utf-8"))
        meta[d["name"]] = d["version"]
    except Exception:
        pass
conda_p = {a: meta.get(a) for a in ["python", "gdal", "libgdal-core", "libgdal-hdf4", "libgdal-hdf5", "libgdal-netcdf"]}
sonuc["conda_paketleri"] = conda_p
for a in ["libgdal-hdf4", "libgdal-hdf5", "libgdal-netcdf"]:
    kaydet("conda_" + a, conda_p[a] is not None, conda_p[a])

# ---------- 3) GDAL: tek sürüm ve sürücüler ----------
from osgeo import gdal
gdal.UseExceptions()
import rasterio, pyogrio

def tek_gdal():
    gv = gdal.__version__
    rv = rasterio.__gdal_version__
    pv = pyogrio.__gdal_version_string__
    kisa = lambda s: ".".join(str(s).split(".")[:3])
    return kisa(gv) == kisa(rv) == kisa(pv), "osgeo " + gv + " | rasterio " + rv + " | pyogrio " + pv
dene("gdal_tek_surum", tek_gdal)
sonuc["gdal_driver_path"] = os.environ.get("GDAL_DRIVER_PATH")

suruculer = ["HDF4", "HDF4Image", "HDF5", "HDF5Image", "netCDF", "GTiff", "GPKG", "ESRI Shapefile", "OpenFileGDB"]
for d in suruculer:
    drv = gdal.GetDriverByName(d)
    kaydet("surucu_" + d, drv is not None, drv.LongName if drv is not None else "yok")

ogr_list = pyogrio.list_drivers()
for d in ["GPKG", "ESRI Shapefile", "OpenFileGDB"]:
    kaydet("pyogrio_surucu_" + d, d in ogr_list, ogr_list.get(d, "yok"))

# ---------- 4) Küçük yaz/oku denemeleri ----------
from rasterio.transform import from_origin

def t_gtiff():
    a = np.arange(16, dtype="float32").reshape(4, 4); a[0, 0] = np.nan
    p = GECICI / "t.tif"
    with rasterio.open(p, "w", driver="GTiff", height=4, width=4, count=1, dtype="float32",
                       crs="EPSG:4326", transform=from_origin(0, 4, 1, 1), nodata=np.nan) as ds:
        ds.write(a, 1)
    with rasterio.open(p) as ds:
        b = ds.read(1)
    return np.array_equal(a, b, equal_nan=True), "4x4 float32, NaN veri-yok geri okundu"
dene("yazoku_GTiff", t_gtiff)

def t_bigtiff():
    # LOLA ve PGDA dosyaları BigTIFF (rapor M1, M7: ilk baytlar 49 49 2B 00)
    a = np.arange(16, dtype="int16").reshape(4, 4)
    p = GECICI / "t_big.tif"
    with rasterio.open(p, "w", driver="GTiff", height=4, width=4, count=1, dtype="int16",
                       crs="EPSG:4326", transform=from_origin(0, 4, 1, 1), BIGTIFF="YES") as ds:
        ds.write(a, 1)
    bas = p.read_bytes()[:4]
    with rasterio.open(p) as ds:
        b = ds.read(1)
    return bas == bytes([0x49, 0x49, 0x2B, 0x00]) and np.array_equal(a, b), "ilk baytlar " + bas.hex(" ")
dene("yazoku_BigTIFF", t_bigtiff)

import geopandas as gpd
from shapely.geometry import box

def vektor_testi(surucu, uzanti):
    def f():
        g = gpd.GeoDataFrame({"birim": ["a"]}, geometry=[box(0, 0, 2, 1)], crs="EPSG:4326")
        p = GECICI / ("t" + uzanti)
        g.to_file(p, driver=surucu, engine="pyogrio")
        g2 = gpd.read_file(p, engine="pyogrio")
        ok = len(g2) == 1 and g2["birim"].iloc[0] == "a" and abs(g2.geometry.iloc[0].area - 2.0) < 1e-12
        return ok, "1 poligon yazıldı/okundu, alan " + str(g2.geometry.iloc[0].area)
    return f
dene("yazoku_GPKG", vektor_testi("GPKG", ".gpkg"))
dene("yazoku_ESRI_Shapefile", vektor_testi("ESRI Shapefile", ".shp"))
dene("yazoku_OpenFileGDB", vektor_testi("OpenFileGDB", ".gdb"))

import h5py
def t_hdf5():
    # ASTER GED gibi: boşluklu öznitelik adı "Scale Factor" (rapor E4)
    a = np.arange(12, dtype="uint8").reshape(3, 4)
    p = GECICI / "t.h5"
    with h5py.File(p, "w") as f:
        ds = f.create_dataset("Emissivity", data=a)
        ds.attrs["Scale Factor"] = 0.002
    with h5py.File(p, "r") as f:
        sf = float(f["Emissivity"].attrs["Scale Factor"])
    g = gdal.Open(str(p))
    alt = g.GetSubDatasets()
    src = gdal.Open(alt[0][0]) if alt else g
    b = src.ReadAsArray()
    src = None; g = None
    return np.array_equal(a, b) and sf == 0.002, "GDAL HDF5 ile okundu; 'Scale Factor' = " + str(sf)
dene("yazoku_HDF5", t_hdf5)

import netCDF4
def t_netcdf():
    a = np.arange(12, dtype="float32").reshape(3, 4)
    p = GECICI / "t.nc"
    with netCDF4.Dataset(p, "w") as n:
        n.createDimension("y", 3); n.createDimension("x", 4)
        v = n.createVariable("z", "f4", ("y", "x")); v[:] = a
    src = gdal.Open('NETCDF:"' + p.as_posix() + '":z')
    b = src.ReadAsArray()
    src = None
    if np.array_equal(a, b):
        return True, "aynı yönde okundu"
    if np.array_equal(a, np.flipud(b)):
        return True, "GDAL satırları ters çevirerek okudu (koordinatsız netCDF'te beklenen; kayda geçti)"
    return False, "değerler tutmuyor"
dene("yazoku_netCDF", t_netcdf)
# HDF4: Python'da HDF4 yazıcısı yok; gerçek okuma testi D2'de MCD12C1 dosyasıyla yapılır.
T["yazoku_HDF4"] = {"durum": "SONRA", "detay": "yalnız sürücü kontrol edildi; okuma testi D2'de MCD12C1 ile"}
print("[SONRA] yazoku_HDF4: okuma testi D2'de MCD12C1 ile")

# ---------- 5) GUDHI CubicalComplex, cevabı bilinen örnek ----------
import gudhi
def t_gudhi():
    # Dört köşe hücre 0, 1, 2, 3; aralar 5. Alt-seviye filtrasyonunda dört ayrı çukur
    # doğar ve hepsi 5'te birleşir. Beklenen H0: (0, sonsuz), (1, 5), (2, 5), (3, 5).
    hucre = np.array([[0, 5, 1], [5, 5, 5], [2, 5, 3]], dtype=float)
    cc = gudhi.CubicalComplex(top_dimensional_cells=hucre)
    p = cc.persistence()
    h0 = sorted((float(b), float(d)) for dim, (b, d) in p if dim == 0)
    beklenen = [(0.0, float("inf")), (1.0, 5.0), (2.0, 5.0), (3.0, 5.0)]
    return h0 == beklenen, "H0 = " + str(h0)
dene("gudhi_CubicalComplex", t_gudhi)

# ---------- 6) Earthdata: netrc ile giriş ve CMR araması ----------
NETRC = Path.home() / "_netrc"
kaydet("netrc_dosyasi", NETRC.is_file(), str(NETRC) + " (içerik okunmaz, yazılmaz)")
os.environ.setdefault("NETRC", str(NETRC))   # earthaccess/requests Windows'ta _netrc'yi bulsun
import earthaccess
def t_login():
    auth = earthaccess.login(strategy="netrc")
    return bool(getattr(auth, "authenticated", False)), "authenticated = " + str(getattr(auth, "authenticated", None))
dene("earthaccess_login_netrc", t_login)

def t_arama():
    # Rapor E2: CMR'de 13 Kasım'da hits = 1
    s = earthaccess.search_data(short_name="MCD12C1", version="061", temporal=("2023-01-01", "2023-12-31"))
    return len(s) == 1, str(len(s)) + " granül (rapor: 1)"
dene("earthaccess_arama_MCD12C1", t_arama)

# ---------- 7) Kayıt ----------
shutil.rmtree(GECICI, ignore_errors=True)
gecmeyen = [k for k, v in T.items() if v["durum"] == "FAIL"]
sonuc["ozet"] = {"toplam": len(T), "PASS": sum(v["durum"] == "PASS" for v in T.values()),
                 "FAIL": len(gecmeyen), "gecmeyenler": gecmeyen}
(KAYIT / "ortam.json").write_text(json.dumps(sonuc, ensure_ascii=False, indent=2), encoding="utf-8")
with open(KAYIT / "13_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(datetime.now().strftime("%Y-%m-%d %H:%M:%S") + "  D0 ortam testi: "
            + str(sonuc["ozet"]["PASS"]) + " PASS, " + str(len(gecmeyen)) + " FAIL " + str(gecmeyen) + "\n")
print("=" * 50)
print("Yazıldı:", KAYIT / "ortam.json")
print("Özet:", sonuc["ozet"])
assert not gecmeyen, "Geçmeyen testler var: " + str(gecmeyen)
print("D0 TAMAM")
'''

dosya = KOK / "kod" / "d0_ortam.py"
dosya.write_text(kod.lstrip("\n"), encoding="utf-8")
assert dosya.read_text(encoding="utf-8").startswith("# Developed with assistance from Claude (Anthropic).")
print("Yazıldı:", dosya, "\n")
runpy.run_path(str(dosya), run_name="__main__")

Yazıldı: C:\Users\USER\orbitrary\kod\d0_ortam.py 

[PASS] ortam_python: C:\Users\USER\anaconda3\envs\orbitrary\python.exe | Python 3.12.16
Makine: {'platform': 'Windows-11-10.0.26200-SP0', 'islemci': 'AMD64 Family 25 Model 97 Stepping 2, AuthenticAMD', 'cekirdek': 12, 'ram_gb': 34.1, 'disk_bos_gb': 184.2, 'conda': 'conda 26.5.3', 'python': '3.12.16 | packaged by conda-forge | (main, Oct 26 2026, 11:29:01) [MSC v.1944 64 bit (AMD64)]', 'python_yolu': 'C:\\Users\\USER\\anaconda3\\envs\\orbitrary\\python.exe'}
[PASS] paket_numpy: 2.5.4
[PASS] paket_scipy: 1.18.1
[PASS] paket_pandas: 3.1.1
[PASS] paket_pyarrow: 26.0.1
[PASS] paket_GDAL: 3.13.3
[PASS] paket_rasterio: 1.5.2
[PASS] paket_pyproj: 3.8.0
[PASS] paket_geopandas: 1.2.0
[PASS] paket_pyogrio: 0.13.0
[PASS] paket_shapely: 2.2.0
[PASS] paket_xarray: 2026.9.0
[PASS] paket_netCDF4: 1.7.5
[PASS] paket_h5py: 3.16.0


C:\Users\USER\anaconda3\envs\orbitrary\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[PASS] paket_earthaccess: 0.19.0
[PASS] paket_gudhi: 3.13.0
[PASS] paket_matplotlib: 3.11.2
[PASS] paket_jupyterlab: 4.6.6
[PASS] paket_pytest: 9.2.0
[PASS] paket_tqdm: 4.70.1
[PASS] paket_inflate64: 1.0.4
[PASS] conda_libgdal-hdf4: 3.13.3
[PASS] conda_libgdal-hdf5: 3.13.3
[PASS] conda_libgdal-netcdf: 3.13.3
[PASS] gdal_tek_surum: osgeo 3.13.3 | rasterio 3.13.3 | pyogrio 3.13.3
[PASS] surucu_HDF4: Hierarchical Data Format Release 4
[PASS] surucu_HDF4Image: HDF4 Dataset
[PASS] surucu_HDF5: Hierarchical Data Format Release 5
[PASS] surucu_HDF5Image: HDF5 Dataset
[PASS] surucu_netCDF: Network Common Data Format
[PASS] surucu_GTiff: GeoTIFF
[PASS] surucu_GPKG: GeoPackage
[PASS] surucu_ESRI Shapefile: ESRI Shapefile
[PASS] surucu_OpenFileGDB: ESRI FileGeodatabase (using OpenFileGDB)
[PASS] pyogrio_surucu_GPKG: raw
[PASS] pyogrio_surucu_ESRI Shapefile: raw
[PASS] pyogrio_surucu_OpenFileGDB: raw
[PASS] yazoku_GTiff: 4x4 float32, NaN veri-yok geri okundu
[PASS] yazoku_BigTIFF: ilk baytlar 49 4

{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'C:\\Users\\USER\\orbitrary\\kod\\d0_ortam.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, locals=None, fromlist=(), lev

In [3]:
# Developed with assistance from Claude (Anthropic).
# Adım 2a-ön — indirme raporunun gerçek adını ve yerini bulur. Hiçbir dosyayı taşımaz/değiştirmez.
from pathlib import Path
import hashlib

KOK = Path.home() / "orbitrary"

print("1) kod\\ klasörünün içi (tam adlarıyla):")
for p in sorted((KOK / "kod").iterdir()):
    print(f"   {p.name!r:60} {p.stat().st_size:>10} bayt" if p.is_file() else f"   {p.name!r}  (klasör)")

print("\n2) 'indirme_raporu' geçen dosyalar (proje, İndirilenler, Masaüstü, Belgeler):")
aranan = [KOK, Path.home() / "Downloads", Path.home() / "Desktop", Path.home() / "Documents",
          Path.home() / "OneDrive" / "Desktop", Path.home() / "OneDrive" / "Documents"]
bulunan = []
for kok in aranan:
    if not kok.is_dir():
        continue
    for p in kok.rglob("*indirme_raporu*"):
        if p.is_file():
            bulunan.append(p)
for p in bulunan:
    ilk = p.read_bytes()[:80].decode("utf-8", errors="replace").splitlines()[0]
    sha = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
    print(f"   {str(p)!r}\n      {p.stat().st_size} bayt | SHA-256 ilk 16: {sha} | ilk satır: {ilk!r}")
if not bulunan:
    print("   hiç bulunamadı")

1) kod\ klasörünün içi (tam adlarıyla):
   '.ipynb_checkpoints'  (klasör)
   '13_kasim_veri.ipynb'                                             93466 bayt
   'd0_ortam.py'                                                     11842 bayt

2) 'indirme_raporu' geçen dosyalar (proje, İndirilenler, Masaüstü, Belgeler):
   'C:\\Users\\USER\\Desktop\\NASA SPACE APPS CHALLANGE\\13 kasım\\indirme_raporu3.md'
      122107 bayt | SHA-256 ilk 16: 7940e0720f2ff1de | ilk satır: '# Orbitrary v1.2 — Veri indirme ve analog raporu (Bölüm 16 + B eki)'


In [5]:
# Developed with assistance from Claude (Anthropic).
# Adım 2a-kopya — indirme raporunu masaüstünden kod\indirme_raporu3.md olarak kopyalar.
# Asıl dosya yerinde kalır. Kopyanın bayt bayt aynı olduğu doğrulanır.
from pathlib import Path
from datetime import datetime
import shutil, hashlib

KOK = Path.home() / "orbitrary"
KAYNAK = Path.home() / "Desktop" / "NASA SPACE APPS CHALLANGE" / "13 kasım" / "indirme_raporu3.md"
HEDEF = KOK / "kod" / "indirme_raporu3.md"

assert KAYNAK.is_file(), f"Kaynak bulunamadı: {KAYNAK}"
ilk = KAYNAK.read_text(encoding="utf-8").splitlines()[0]
assert ilk == "# Orbitrary v1.2 — Veri indirme ve analog raporu (Bölüm 16 + B eki)", f"Beklenmeyen ilk satır: {ilk!r}"
assert "indirme_raporu3.md (13 Kasım 2026, 21:00)" in KAYNAK.read_text(encoding="utf-8"), "Sürüm satırı yok (rapor3 değil mi?)"

shutil.copy2(KAYNAK, HEDEF)

sha_k = hashlib.sha256(KAYNAK.read_bytes()).hexdigest()
sha_h = hashlib.sha256(HEDEF.read_bytes()).hexdigest()
assert sha_k == sha_h, "Kopya kaynakla aynı değil"
with open(KOK / "kayit" / "13_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(f"{datetime.now():%Y-%m-%d %H:%M:%S}  2a-kopya indirme_raporu3.md -> kod\\ ({HEDEF.stat().st_size} bayt, sha256 {sha_h})\n")

print("OK  ->", HEDEF)
print("Boyut:", HEDEF.stat().st_size, "bayt")
print("SHA-256:", sha_h)

OK  -> C:\Users\USER\orbitrary\kod\indirme_raporu3.md
Boyut: 122107 bayt
SHA-256: 7940e0720f2ff1dead902f36d3d07c0bd6d0c127f39b8da71f2df2965f2ed551


In [6]:
# Developed with assistance from Claude (Anthropic).
# Adım 2a — kod\indirme_listesi.csv ve kod\liste_denetim.py yazılır, denetim çalıştırılır.
# Kaynak: kod\indirme_raporu3.md (her satırın "set" sütunu rapordaki bloğu gösterir).
import csv, runpy, sys
from pathlib import Path

KOK = Path.home() / "orbitrary"
assert "orbitrary" in Path(sys.executable).parts, "Çekirdek orbitrary ortamında değil"
assert (KOK / "kod" / "indirme_raporu3.md").is_file(), "kod\\indirme_raporu3.md yok"

ALANLAR = ["set", "govde", "yontem", "url", "yedek_url", "hedef", "beklenen_bayt", "beklenen_adet",
           "cmr_mb", "granul", "short_name", "version", "temporal", "saglama_turu", "saglama_degeri",
           "saglama_kaynak", "saglama_anahtar", "grup", "not"]
satirlar = []
KLASOR = {"E1": "etopo", "E2": "mcd12c1", "E3": "glim", "E4": "aster_ged", "E5": "myd11c3",
          "E6": "mcd43c3", "E7": "permafrost_kuzey", "E8": "permafrost_antarktika", "E9": "aridity",
          "E10": "soilgrids", "M1": "lola", "M2": "ay_jeoloji", "M3": "diviner_ghrm",
          "M4": "diviner_prp_gcp", "M5": "lp_grs", "M6": "lp_hidrojen", "M7": "pgda",
          "R1": "mola", "R2": "sim3292", "R3": "tes_mineral", "R4": "tes_ti_albedo", "R5": "tes_dci",
          "R6": "odyssey_grs", "R7": "swim"}
GOVDE = {"E": "dunya", "M": "ay", "R": "mars"}

def ekle(s, url="", ad=None, bayt="", grup=2, yontem=None, **k):
    """Bir indirme satırı ekler. hedef = <govde>/<set>_<ad>/<dosya> (veri\\ altında)."""
    govde = GOVDE[s[0]]
    dosya = ad if ad else url.rsplit("/", 1)[-1]
    if yontem is None:
        yontem = "http" if url.startswith("http://") else "https"
    r = {a: "" for a in ALANLAR}
    r.update(set=s, govde=govde, yontem=yontem, url=url,
             hedef=f"{govde}/{s}_{KLASOR[s]}/{dosya}", beklenen_bayt=str(bayt), grup=str(grup))
    for a, v in k.items():
        r[a] = str(v)
    satirlar.append(r)

# ---------------- DÜNYA ----------------
# E1: klasör listesinden 288 karo (ad kalıbı raporda); toplam bayt raporda
ekle("E1", "https://www.ngdc.noaa.gov/mgg/global/relief/ETOPO2022/data/15s/15s_surface_elev_gtif/",
     ad="", yontem="klasor", bayt=5230677335, beklenen_adet=288, grup=1,
     saglama_turu="yok", **{"not": "kalip ETOPO_2022_v1_15s_{N|S}{YY}{E|W}{XXX}_surface.tif; her karo HEAD boyutu"})
satirlar[-1]["hedef"] = "dunya/E1_etopo/"

# E2, E4, E5, E6: Earthdata (earthaccess)
ekle("E2", ad="MCD12C1.A2023001.061.2024251212901.hdf", yontem="earthaccess", grup=2,
     granul="MCD12C1.A2023001.061.2024251212901", cmr_mb="1186,6", short_name="MCD12C1",
     version="061", temporal="2023-01-01|2023-12-31", saglama_turu="cmr_boyut")
for ddd in ["001", "032", "060", "091", "121", "152", "182", "213", "244", "274", "305", "335"]:
    ekle("E4", ad=f"ASTER_GEDv4.1_A2015{ddd}.h5", yontem="earthaccess", grup=1,
         granul=f"ASTER_GEDv4.1_A2015{ddd}", cmr_mb="321,366", short_name="AG5KMMOH",
         version="041", temporal="2015-01-01|2015-12-31", saglama_turu="cmr_boyut")
E5 = [("MYD11C3.A2023001.061.2023033164343", "68,37"), ("MYD11C3.A2023032.061.2023060194236", "69,44"),
      ("MYD11C3.A2023060.061.2023093152049", "73,14"), ("MYD11C3.A2023091.061.2023122195853", "71,80"),
      ("MYD11C3.A2023121.061.2023153211939", "71,78"), ("MYD11C3.A2023152.061.2023193054214", "70,43"),
      ("MYD11C3.A2023182.061.2023215202151", "69,88"), ("MYD11C3.A2023213.061.2023250152704", "70,85"),
      ("MYD11C3.A2023244.061.2023274184351", "71,59"), ("MYD11C3.A2023274.061.2023305220624", "74,07"),
      ("MYD11C3.A2023305.061.2023336214254", "73,88"), ("MYD11C3.A2023335.061.2024006163117", "70,19")]
for g, mb in E5:
    ekle("E5", ad=g + ".hdf", yontem="earthaccess", grup=2, granul=g, cmr_mb=mb, short_name="MYD11C3",
         version="061", temporal="2023-01-01|2023-12-31", saglama_turu="cmr_boyut")
E6 = [("MCD43C3.A2023015.061.2023024104905", "182,27"), ("MCD43C3.A2023046.061.2023057224937", "220,62"),
      ("MCD43C3.A2023074.061.2023083070952", "232,46"), ("MCD43C3.A2023105.061.2023114063822", "196,75"),
      ("MCD43C3.A2023135.061.2023144071438", "189,63"), ("MCD43C3.A2023166.061.2023175104425", "185,03"),
      ("MCD43C3.A2023196.061.2023215170711", "182,22"), ("MCD43C3.A2023227.061.2023236071848", "180,75"),
      ("MCD43C3.A2023258.061.2023267065355", "200,24"), ("MCD43C3.A2023288.061.2023299045841", "223,74"),
      ("MCD43C3.A2023319.061.2023328224454", "196,33"), ("MCD43C3.A2023349.061.2023361073709", "174,97")]
for g, mb in E6:
    ekle("E6", ad=g + ".hdf", yontem="earthaccess", grup=2, granul=g, cmr_mb=mb, short_name="MCD43C3",
         version="061", temporal="2023-01-01|2023-12-31", saglama_turu="cmr_boyut",
         **{"not": "ada gore secilir (tarih aramasi 16 granul dondurur)"})

# E3 GLiM (Dropbox; kesin boyut raporda yok -> 3 test: >=1 GB, 'PK', testzip)
ekle("E3", "https://www.dropbox.com/s/9vuowtebp9f1iud/LiMW_GIS%202015.gdb.zip?dl=1",
     ad="LiMW_GIS 2015.gdb.zip", grup=3, saglama_turu="glim_uc_test",
     yedek_url="https://www.dropbox.com/scl/fi/5v00i8op7a9brmn4qeg8b/LiMW_GIS-2015.gdb.zip",
     **{"not": "yedek adrese ?dl=1 eklenir (rapor E3)"})
# E7, E8 PANGAEA (zip CRC; ana tif CRC'si raporda)
ekle("E7", "https://store.pangaea.de/Publications/ObuJ-etal_2018/UiO_PEX_PERPROB_5.0_20181128_2000_2016_NH.zip",
     bayt=106076765, saglama_turu="zip_crc", saglama_degeri="620a0b80",
     saglama_anahtar="UiO_PEX_PERPROB_5.0_20181128_2000_2016_NH.tif")
ekle("E8", "https://store.pangaea.de/Publications/ObuJ-etal_2019/UiO_PEX_PERPROB_5.0_20181205_2000_2017_ANT.zip",
     bayt=733901, saglama_turu="zip_crc", saglama_degeri="a6ec5a8c",
     saglama_anahtar="UiO_PEX_PERPROB_5.0_20181205_2000_2017_ANT.tif")
# E9 figshare (MD5 figshare API'den, raporda)
ekle("E9", "https://ndownloader.figshare.com/files/56300327", ad="Global-AI_ET0__annual_v3_1.zip",
     bayt=645783906, saglama_turu="md5", saglama_degeri="ef3f6f53a082fb07365954b468cca55f")
ekle("E9", "https://ndownloader.figshare.com/files/56300216", ad="Global AI_ET0 v3.1 - Readme.docx",
     bayt=645897, saglama_turu="md5", saglama_degeri="ad026eba14bd5f50c6be2abc603ed74f")
# E10 SoilGrids (üretici SHA-256 dosyası + rapordaki değer)
for oz, b, sha in [("clay", 191530251, "20f699cc64686bb5696e395f709232e5957ea3c6a4b39f6676517a502033f424"),
                   ("silt", 194638176, "4c72f9303aa27fcb5f2527bb1a64b307e9ef37885afc08cdcee063e99c495295")]:
    base = f"https://files.isric.org/soilgrids/latest/data_aggregated/1000m/{oz}/"
    ekle("E10", base + "checksum.sha256.txt", ad=f"{oz}/checksum.sha256.txt", saglama_turu="yok",
         **{"not": "uretici SHA-256 listesi"})
    ekle("E10", base + f"{oz}_0-5cm_mean_1000.tif", ad=f"{oz}/{oz}_0-5cm_mean_1000.tif", bayt=b,
         saglama_turu="sha256", saglama_degeri=sha, saglama_kaynak=f"dunya/E10_soilgrids/{oz}/checksum.sha256.txt",
         saglama_anahtar=f"{oz}_0-5cm_mean_1000.tif")

# ---------------- AY ----------------
U = "https://planetarymaps.usgs.gov/mosaic/"
ekle("M1", U + "Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif.md5", saglama_turu="yok", grup=1)
ekle("M1", U + "Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.lbl", saglama_turu="yok", grup=1)
ekle("M1", U + "Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif", bayt=8494203833, grup=1,
     saglama_turu="md5", saglama_degeri="40b61c7051c744582ec0a2d153757b5e",
     saglama_kaynak="ay/M1_lola/Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif.md5",
     saglama_anahtar="Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif")
ekle("M2", "https://asc-astropedia.s3.us-west-2.amazonaws.com/Moon/Geology/Unified_Geologic_Map_of_the_Moon_GIS_v2.zip",
     bayt=224413040, saglama_turu="zip_crc")

DIV = "https://pds-geosciences.wustl.edu/lro/urn-nasa-pds-lro_diviner_derived1/"
DIV_MD5 = "ay/M3_diviner_ghrm/urn-nasa-pds-lro_diviner_derived1.md5"
ekle("M3", DIV + "urn-nasa-pds-lro_diviner_derived1.md5", bayt=45050842, saglama_turu="yok",
     **{"not": "M3 ve M4 icin uretici MD5 listesi; anahtarlar ters egik cizgili"})
for ad, b, md5, grp in [("dghrm_ra_sam_70s70n_tif.tif", 3303158255, "F502BB2837DAC380A2794D56BDEA755C", 1),
                        ("dghrm_ra_sam_70s70n_tif.xml", "", "D44AD4DDE044CAB441829A89A1B12EA7", 2),
                        ("dghrm_tbol_sam_70s70n_tif.tif", 3303158255, "8DE5FA42F510B6079A0DB622ABE6208D", 1),
                        ("dghrm_tbol_sam_70s70n_tif.xml", "", "836089C3BF4F2FA4C5D6106F2FF81E65", 2)]:
    ekle("M3", DIV + "data_derived_ghrm/geotiff/" + ad, bayt=b, grup=grp, saglama_turu="md5",
         saglama_degeri=md5, saglama_kaynak=DIV_MD5, saglama_anahtar="data_derived_ghrm/geotiff/" + ad)
M4 = [("data_derived_prp/dlre_prp_north.tab", 604800210, "200108632FB75045597862887E841C01"),
      ("data_derived_prp/dlre_prp_north.lbl", 2167, "04611C68D7C1D8CF677DAFC61C944343"),
      ("data_derived_prp/dlre_prp_north.xml", 14561, "D02698E6FF8DC18B11535B3037CEAAC1"),
      ("data_derived_prp/dlre_prp_south.tab", 604800210, "80CDFE83592A3419C74B1B55B636711E"),
      ("data_derived_prp/dlre_prp_south.lbl", 2167, "C209EDAAA10C7AF19D973E252AC589EC"),
      ("data_derived_prp/dlre_prp_south.xml", 14561, "C0BE50F837784FA478DECB741C003482"),
      ("data_derived_gcp/global_cumul_avg_cyl_70n80n_002.tab", 156211313, "C132354928AD9B09A3DA974B6AF9F215"),
      ("data_derived_gcp/global_cumul_avg_cyl_70n80n_002.lbl", 3462, "E8908730B7CB7F7D1647E1A93099DD11"),
      ("data_derived_gcp/global_cumul_avg_cyl_70n80n_002.xml", 16003, "A9FF44AF068B3517298BE2DDF08DE90C"),
      ("data_derived_gcp/global_cumul_avg_cyl_80n90n_002.tab", 156211313, "396E9215AB3925B3B1FACF6D487050C4"),
      ("data_derived_gcp/global_cumul_avg_cyl_80n90n_002.lbl", 3462, "BABBEDE556C76F829731E00F07A1215E"),
      ("data_derived_gcp/global_cumul_avg_cyl_80n90n_002.xml", 16003, "81B738FADCDE4618BFAAD9A5E744FC63"),
      ("data_derived_gcp/global_cumul_avg_cyl_80s70s_002.tab", 156211313, "D26AA3C85B82105DCB15825DE076A4CE"),
      ("data_derived_gcp/global_cumul_avg_cyl_80s70s_002.lbl", 3464, "5B2B9B51C46836ADFF77AE5478F1038C"),
      ("data_derived_gcp/global_cumul_avg_cyl_80s70s_002.xml", 16005, "9C7B40ECEC295E89EA95AA0A465F0D78"),
      ("data_derived_gcp/global_cumul_avg_cyl_90s80s_002.tab", 156211313, "3200DD1267B676CAD89300D344A88010"),
      ("data_derived_gcp/global_cumul_avg_cyl_90s80s_002.lbl", 3413, "3A97121C320B450E324FD335B1F6F48B"),
      ("data_derived_gcp/global_cumul_avg_cyl_90s80s_002.xml", 16005, "DC72C34810B3A90E2900BA1D7B09EC42")]
for yol, b, md5 in M4:
    ekle("M4", DIV + yol, bayt=b, saglama_turu="md5", saglama_degeri=md5,
         saglama_kaynak=DIV_MD5, saglama_anahtar=yol)
LP = "https://pds-geosciences.wustl.edu/lunar/lp-l-grs-5-elem-abundance-v1/lp_9001/data/"
for ad, b in [("lpgrs_high1_elem_abundance_2deg.tab", 9949280), ("lpgrs_high1_elem_abundance_2deg.lbl", 8102),
              ("lpgrs_high1_elem_abundance_2deg.xml", 37946), ("lpgrs_elem_abundance.fmt", 23574)]:
    ekle("M5", LP + ad, bayt=b, saglama_turu="yok")
H = "https://pds-geosciences.wustl.edu/missions/lunarp/reduced/"
for ad, b in [("hydrogenhd.txt", 14256401), ("hydrogenhd.dat", 518400), ("hydrogenlow.txt", 622233),
              ("hydrogenlow.dat", 518400), ("epis.txt", 757837), ("epis.dat", 518400)]:
    ekle("M6", H + ad, bayt=b, saglama_turu="yok")
for ad, b in [("AVGVISIB_65S_240M_201608.TIF", 60394652), ("AVGVISIB_65N_240M_201608.TIF", 61184010),
              ("LPSR_65S_240M_201608.TIF", 582074), ("LPSR_65N_240M_201608.TIF", 613892)]:
    ekle("M7", "https://pgda.gsfc.nasa.gov/data/MoonIllumination/" + ad, bayt=b, saglama_turu="yok")
for ad, b in [("AVGVISIB_65S_240M_201608.LBL", 4096), ("AVGVISIB_65N_240M_201608.LBL", 4093),
              ("LPSR_65S_240M_201608.LBL", 3917), ("LPSR_65N_240M_201608.LBL", 3914)]:
    ekle("M7", "https://imbrium.mit.edu/EXTRAS/ILLUMINATION/IMG/" + ad, bayt=b, saglama_turu="yok")

# ---------------- MARS ----------------
ekle("R1", U + "Mars_MGS_MOLA_DEM_mosaic_global_463m.tif.md5", saglama_turu="yok")
ekle("R1", U + "Mars_MGS_MOLA_DEM_mosaic_global_463m.lbl", saglama_turu="yok")
ekle("R1", U + "Mars_MGS_MOLA_DEM_mosaic_global_463m.tif", bayt=2125771142, saglama_turu="md5",
     saglama_degeri="e7920a303f46707d57fe72eab5f6b1aa",
     saglama_kaynak="mars/R1_mola/Mars_MGS_MOLA_DEM_mosaic_global_463m.tif.md5",
     saglama_anahtar="Mars_MGS_MOLA_DEM_mosaic_global_463m.tif")
ekle("R2", "https://pubs.usgs.gov/sim/3292/downloads/sim3292_database.zip", bayt=808910442, saglama_turu="zip_crc")
ekle("R2", "https://pubs.usgs.gov/sim/3292/downloads/sim3292_readme.txt", saglama_turu="yok",
     **{"not": "boyut raporda kesin degil; indirme sonrasi kaydedilir"})
for ad, b in [("TES_mineral_maps_4_ppd.envi", 53913600), ("TES_mineral_maps_4_ppd.envi.hdr", 314),
              ("TES_mineral_maps_4_ppd.isis", 53914624), ("endmember_list.ascii", 1105)]:
    ekle("R3", "http://tes.asu.edu/mineral_map/" + ad, bayt=b, grup=3, saglama_turu="yok")
for ad, b in [("ti_nmap2003.bin", 51847200), ("global_albedo_8ppd.img", 16588800), ("global_albedo_8ppd.lbl", 2334)]:
    ekle("R4", "http://tes.asu.edu/mineral_map/" + ad, bayt=b, grup=3, saglama_turu="yok")
ekle("R4", "https://se.psi.edu/inertia/nmsk2003.bin", bayt=25927200, grup=3, saglama_turu="yok",
     **{"not": "boyut PSI bicim tanimindan hesaplanmis (rapor R4)"})
ekle("R5", "http://www.mars.asu.edu/~ruff/DCI/dci_lo_ice_dust_16ppd_shifted.pds", bayt=66356224, grup=3,
     saglama_turu="yok",
     yedek_url="https://web.archive.org/web/2024id_/http://www.mars.asu.edu/~ruff/DCI/dci_lo_ice_dust_16ppd_shifted.pds")
OD = "https://pds-geosciences.wustl.edu/ody/ody-m-grs-5-elements-v1/"
ekle("R6", OD + "odgm_xxxx_130911.md5", bayt=7079, saglama_turu="yok")
R6 = {"h2o": (186624, 4653, "C6B4A8F9387B886EE99EF2F518D5ABE8", "595A0199FA7C6C0CEBE343235100532F"),
      "fe":  (186624, 4647, "FA90A7E62F40BB81BB893524471E2156", "3BA2FEF040DA91F1F618C0FEAD4B51D0"),
      "si":  (186624, 4665, "F24A42FA2C11FEC04D59D4B5FAB20F3F", "DA4B22AC036EBCD69ADB622B7AD5E06B"),
      "k":   (171072, 4642, "06A60428199F8915F6769E51CAFED9EA", "9AE05D831F4812C473987423B25CFE17"),
      "th":  (171072, 4645, "70A7FD2E68CFF9AE9D907B57F1A3BDE4", "58DD0E5E128BE96AEC8E9A08C86B8363"),
      "cl":  (186624, 4649, "6419661A648F4B9D18789B3B86B8BE1D", "0839F9C3DF458127A12B735A362A8441")}
for el, (bt, bl, mt, ml) in R6.items():
    for uz, b, md5 in [("tab", bt, mt), ("lbl", bl, ml)]:
        yol = f"odgm_xxxx/data/smoothed/{el}_sr_5x5.{uz}"
        ekle("R6", OD + yol, bayt=b, saglama_turu="md5", saglama_degeri=md5,
             saglama_kaynak="mars/R6_odyssey_grs/odgm_xxxx_130911.md5", saglama_anahtar=yol)
for ad, b in [("SWIM4MIM_Ci_0_1.tif", 70995807), ("SWIM4MIM_Ci_1_5.tif", 70995807), ("SWIM4MIM_Ci_5.tif", 70995794)]:
    ekle("R7", "https://swim.psi.edu/output/SWIM4MIM/Global/Composite/" + ad, bayt=b, grup=3, saglama_turu="yok")

# ---------------- CSV yaz ----------------
LISTE = KOK / "kod" / "indirme_listesi.csv"
with open(LISTE, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=ALANLAR)
    w.writeheader()
    w.writerows(satirlar)
print("Yazıldı:", LISTE, "-", len(satirlar), "satır")

# ---------------- Denetim betiği ----------------
denetim = r'''# Developed with assistance from Claude (Anthropic).
# Liste denetimi (belge 17.2 girdisi): kod\indirme_listesi.csv'deki her adres, boyut,
# sağlama değeri, granül adı ve CMR boyutu kod\indirme_raporu3.md içinde geçiyor mu?
# indir_hepsi.py her başlangıçta bunu çalıştırır; tek bir hata varsa indirme başlamaz.
import csv, json, re, hashlib
from pathlib import Path
from datetime import datetime

KOK = Path.home() / "orbitrary"
RAPOR = KOK / "kod" / "indirme_raporu3.md"
LISTE = KOK / "kod" / "indirme_listesi.csv"
BEKLENEN_DOSYA = 409                 # rapor, Ek bölüm 2 "Toplam" tablosu
SETLER = [f"E{i}" for i in range(1, 11)] + [f"M{i}" for i in range(1, 8)] + [f"R{i}" for i in range(1, 8)]

def _genislet(s):
    # "a.{tab,lbl}" -> ["a.tab", "a.lbl"]; iç içe olmayan, virgüllü süslü parantezler
    m = re.search(r"\{([^{}]*,[^{}]*)\}", s)
    if not m:
        return [s]
    out = []
    for p in m.group(1).split(","):
        out += _genislet(s[:m.start()] + p.strip() + s[m.end():])
    return out

def denetle():
    ham = RAPOR.read_bytes()
    R = ham.decode("utf-8")
    # Boşluklu sayılar birleştirilir: "604 800 210" -> "604800210"
    RN = R
    for _ in range(6):
        RN = re.sub(r"(?<=\d)[ \u00a0\u202f\u2009](?=\d{3}(?!\d))", "", RN)
    GEN = set()
    for tok in re.findall(r"[^\s`'\"()<>\u00b7]*\{[^{}\s]+\}[^\s`'\"()<>\u00b7]*", R):
        for g in _genislet(tok):
            GEN.add(g)
            GEN.add(g.split("://", 1)[-1])

    def adres(u):
        if u in R:
            return "tam"
        if u in GEN:
            return "genisletilmis"
        d, ad = u.rsplit("/", 1)
        if (ad in R or ad in GEN) and d.split("://", 1)[-1] in R:
            return "klasor+ad"
        return None

    def sayi_var(n):
        return re.search(r"(?<![\d.,])" + re.escape(str(n)) + r"(?!\d)", RN) is not None

    with open(LISTE, encoding="utf-8", newline="") as f:
        satirlar = list(csv.DictReader(f))
    hatalar, kurallar = [], {}
    for i, r in enumerate(satirlar, start=2):           # 1. satır başlık
        etiket = f"satir {i} {r['set']} {r['hedef']}"
        def hata(m):
            hatalar.append(etiket + ": " + m)
        if r["set"] not in SETLER:
            hata("bilinmeyen set")
        if r["url"]:
            k = adres(r["url"])
            if k is None:
                hata("adres raporda yok: " + r["url"])
            kurallar[k] = kurallar.get(k, 0) + 1
        if r["yedek_url"] and adres(r["yedek_url"]) is None:
            hata("yedek adres raporda yok: " + r["yedek_url"])
        if r["beklenen_bayt"] and not sayi_var(r["beklenen_bayt"]):
            hata("boyut raporda yok: " + r["beklenen_bayt"])
        if r["beklenen_adet"] and not sayi_var(r["beklenen_adet"]):
            hata("adet raporda yok: " + r["beklenen_adet"])
        if r["saglama_degeri"] and r["saglama_degeri"] not in R:
            hata("saglama degeri raporda yok: " + r["saglama_degeri"])
        if r["yontem"] == "klasor" and "ETOPO_2022_v1_15s_{N|S}{YY}{E|W}{XXX}_surface.tif" not in R:
            hata("E1 ad kalibi raporda yok")
        if r["yontem"] == "earthaccess":
            if f'short_name="{r["short_name"]}", version="{r["version"]}"' not in R:
                hata("short_name/version raporda yok")
            a, b = r["temporal"].split("|")
            if f'temporal=("{a}","{b}")' not in R:
                hata("temporal raporda yok")
            if r["set"] == "E4":
                if "ASTER_GEDv4.1_A2015{DDD}.h5" not in R or \
                   "001, 032, 060, 091, 121, 152, 182, 213, 244, 274, 305, 335" not in R:
                    hata("E4 ad kalibi ya da DDD listesi raporda yok")
                if r["cmr_mb"] not in R:
                    hata("CMR boyutu raporda yok")
            elif r["set"] == "E2":
                if r["granul"] not in R or r["cmr_mb"] not in R:
                    hata("granul adi ya da CMR boyutu raporda yok")
            else:
                if f"`{r['granul']}` {r['cmr_mb']}" not in R:
                    hata("granul adi + CMR boyutu cifti raporda yok")
    hedefler = [r["hedef"] for r in satirlar]
    if len(set(hedefler)) != len(hedefler):
        hatalar.append("ayni hedef yolu birden fazla satirda")
    if not all(h.split("/")[0] in ("dunya", "ay", "mars") for h in hedefler):
        hatalar.append("hedef dunya/ay/mars disinda")
    eksik_set = [s for s in SETLER if s not in {r["set"] for r in satirlar}]
    if eksik_set:
        hatalar.append("listede olmayan set: " + str(eksik_set))
    dosya_sayisi = sum(int(r["beklenen_adet"]) if r["beklenen_adet"] else 1 for r in satirlar)
    if dosya_sayisi != BEKLENEN_DOSYA:
        hatalar.append(f"dosya sayisi {dosya_sayisi}, rapor {BEKLENEN_DOSYA}")
    set_sayim = {}
    for r in satirlar:
        set_sayim[r["set"]] = set_sayim.get(r["set"], 0) + (int(r["beklenen_adet"]) if r["beklenen_adet"] else 1)
    return {"tarih": datetime.now().isoformat(timespec="seconds"),
            "rapor": str(RAPOR), "rapor_sha256": hashlib.sha256(ham).hexdigest(),
            "liste": str(LISTE), "liste_sha256": hashlib.sha256(LISTE.read_bytes()).hexdigest(),
            "satir": len(satirlar), "dosya": dosya_sayisi, "set_basina_dosya": set_sayim,
            "adres_kurali": kurallar, "hatalar": hatalar, "durum": "PASS" if not hatalar else "FAIL"}

if __name__ == "__main__":
    s = denetle()
    (KOK / "kayit" / "liste_denetim.json").write_text(json.dumps(s, ensure_ascii=False, indent=2), encoding="utf-8")
    (KOK / "kayit" / "rapor_sha256.txt").write_text(s["rapor_sha256"] + "  kod/indirme_raporu3.md\n", encoding="utf-8")
    with open(KOK / "kayit" / "13_kasim_log.txt", "a", encoding="utf-8") as f:
        f.write(datetime.now().strftime("%Y-%m-%d %H:%M:%S") + f"  2a liste denetimi {s['durum']}: "
                f"{s['satir']} satir, {s['dosya']} dosya, {len(s['hatalar'])} hata\n")
    print("Rapor SHA-256:", s["rapor_sha256"])
    print("Satır:", s["satir"], "| Dosya:", s["dosya"], "(rapor: 409)")
    print("Set başına dosya:", s["set_basina_dosya"])
    print("Adres kuralı:", s["adres_kurali"])
    for h in s["hatalar"]:
        print("HATA:", h)
    print("Durum:", s["durum"])
    assert not s["hatalar"], f"{len(s['hatalar'])} hata var; indirme başlamaz"
'''
DEN = KOK / "kod" / "liste_denetim.py"
DEN.write_text(denetim, encoding="utf-8")
print("Yazıldı:", DEN, "\n")
_ = runpy.run_path(str(DEN), run_name="__main__")

Yazıldı: C:\Users\USER\orbitrary\kod\indirme_listesi.csv - 122 satır
Yazıldı: C:\Users\USER\orbitrary\kod\liste_denetim.py 

Rapor SHA-256: 7940e0720f2ff1dead902f36d3d07c0bd6d0c127f39b8da71f2df2965f2ed551
Satır: 122 | Dosya: 409 (rapor: 409)
Set başına dosya: {'E1': 288, 'E2': 1, 'E4': 12, 'E5': 12, 'E6': 12, 'E3': 1, 'E7': 1, 'E8': 1, 'E9': 2, 'E10': 4, 'M1': 3, 'M2': 1, 'M3': 5, 'M4': 18, 'M5': 4, 'M6': 6, 'M7': 8, 'R1': 3, 'R2': 2, 'R3': 4, 'R4': 4, 'R5': 1, 'R6': 13, 'R7': 3}
Adres kuralı: {'tam': 52, 'klasor+ad': 15, 'genisletilmis': 18}
Durum: PASS


In [7]:
# Developed with assistance from Claude (Anthropic).
# Adım 2b — kod\indir_hepsi.py yazılır ve sözdizimi denetlenir. Çalıştırma Anaconda Prompt'ta.
from pathlib import Path
import hashlib

KOK = Path.home() / "orbitrary"
kod = r'''# Developed with assistance from Claude (Anthropic).
# indir_hepsi.py — Orbitrary v1.2, belge Bölüm 17.2 (veri indirme).
# Girdi : kod/indirme_listesi.csv; her açılışta kod/liste_denetim.py ile raporla denetlenir.
# Çıktı : veri/<govde>/<set>_<ad>/...  ham veri (doğrulanınca salt okunur yapılır)
#         kayit/indirme.csv     her dosya: adres, bayt, SHA-256, MD5, süre, deneme, durum, not
#         kayit/indirme_ozet.json, kayit/E1_karolar.csv, kayit/indirme_kuru.csv (--kuru)
# Kullanım (Anaconda Prompt, orbitrary ortamı, proje klasöründe):
#   python kod\indir_hepsi.py --kuru        indirmeden sunucuları, boyutları ve CMR'yi sınar
#   python kod\indir_hepsi.py               hepsini indirir; yeniden çalıştırılırsa kaldığı yerden devam eder
#   python kod\indir_hepsi.py --set E3 R5   yalnız seçilen setler
# Durumlar: PASS    boyut ve/veya sağlama tuttu
#           FAIL    tutmadı ya da inmedi (kayda yazılır, silinmez)
#           KONTROL raporda sağlama/boyut yok ya da yaklaşık (CMR); D1'de içerik testiyle kapanır
import argparse, csv, hashlib, json, os, re, shutil, stat, sys, threading, time, zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from html.parser import HTMLParser
from pathlib import Path

import requests

KOK = Path.home() / "orbitrary"
VERI = KOK / "veri"
KAYIT = KOK / "kayit"
GECICI = KOK / "ara" / "indirme_gecici"          # Earthdata için geçici klasör (proje içinde)
LISTE = KOK / "kod" / "indirme_listesi.csv"

BEKLEMELER = [5, 15, 45, 120, 300]   # belge 17.2: artan bekleme (s); ilk deneme + 5 tekrar
KURU_BEKLEMELER = [5, 15]            # kuru koşuda kısa tekrar
ESZAMANLI = 4                        # belge 17.2: en fazla 4 eşzamanlı indirme
PARCA = 4 * 1024 * 1024
ZAMAN_ASIMI = (30, 120)              # (bağlantı, okuma) saniye
EN_AZ_BOS_DISK = 45e9                # belge 17.1: ham veri yaklaşık 45 GB (rapor toplamı 36,5 GB)
BASLIK = {"User-Agent": "Orbitrary/1.2 (NASA Space Apps 2026; research data download; python-requests)",
          "Accept-Encoding": "identity"}   # sunucu sıkıştırmasın: boyut ve Range ham bayt üzerinden

KILIT = threading.Lock()
ED_KILIT = threading.Lock()          # earthaccess.download aynı anda tek iş
SONUCLAR = []
_yerel = threading.local()
_indirilen = [0]

LOG_ALANLAR = ["zaman", "set", "hedef", "adres", "bayt", "beklenen_bayt", "sha256", "md5",
               "sure_s", "deneme", "durum", "not"]


class KaliciHata(Exception):
    """Tekrar denemenin anlamsız olduğu hata (ör. HTTP 404); varsa yedek adrese geçilir."""


def say(msg):
    with KILIT:
        print(f"{datetime.now():%H:%M:%S} {msg}", flush=True)


def ilerleme(n):
    with KILIT:
        _indirilen[0] += n


def oturum():
    # requests.Session iş parçacıkları arasında paylaşılmaz; her iş parçacığına bir oturum
    if not hasattr(_yerel, "s"):
        s = requests.Session()
        s.headers.update(BASLIK)
        _yerel.s = s
    return _yerel.s


def tekrarla(ad, fonk, beklemeler=BEKLEMELER):
    """fonk()'u çalıştırır; hata olursa artan beklemeyle tekrar dener. (sonuç, deneme sayısı) döner."""
    for i in range(len(beklemeler) + 1):
        try:
            return fonk(), i + 1
        except KaliciHata:
            raise
        except Exception as e:
            if i == len(beklemeler):
                raise
            say(f"   tekrar {i + 1}/{len(beklemeler)} {ad}: {type(e).__name__}: {str(e)[:150]} -> {beklemeler[i]} s bekleniyor")
            time.sleep(beklemeler[i])


def log_yaz(k):
    yol = KAYIT / "indirme.csv"
    with KILIT:
        yeni = not yol.exists()
        with open(yol, "a", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=LOG_ALANLAR)
            if yeni:
                w.writeheader()
            w.writerow({a: k.get(a, "") for a in LOG_ALANLAR})
        SONUCLAR.append(k)


def bitir(r, yol, durum, notlar, md5, sha, t0, deneme, adres, beklenen):
    k = {"zaman": datetime.now().isoformat(timespec="seconds"), "set": r["set"], "hedef": r["hedef"],
         "adres": adres, "bayt": yol.stat().st_size if (yol is not None and yol.exists()) else "",
         "beklenen_bayt": beklenen or "", "sha256": sha, "md5": md5,
         "sure_s": round(time.time() - t0, 1), "deneme": deneme, "durum": durum, "not": " | ".join(notlar)}
    log_yaz(k)
    say(f"[{durum}] {r['set']} {r['hedef']} ({k['bayt']} bayt, {k['sure_s']} s) {k['not'][:180]}")
    return durum


# ---------------------------------------------------------------- HTTP indirme (Range ile devam)
def http_indir(url, part, beklenen):
    """belge 17.2: var olan parçada Range: bytes=n- ; 206 ve Content-Range başlangıcı n ise eklenir;
    200 gelirse sıfırdan yazılır; 416 = tamam (doğrulama yine yapılır)."""
    s = oturum()
    n = part.stat().st_size if part.exists() else 0
    if beklenen and n > beklenen:
        part.unlink()
        n = 0
    if beklenen and n == beklenen:
        return "parca zaten tam boyutta"
    h = {"Range": f"bytes={n}-"} if n > 0 else {}
    with s.get(url, headers=h, stream=True, timeout=ZAMAN_ASIMI, allow_redirects=True) as r:
        if r.status_code == 416:
            return "416 (parca tam)"
        if r.status_code >= 500 or r.status_code in (408, 429):
            raise IOError(f"HTTP {r.status_code}")
        if r.status_code >= 400:
            raise KaliciHata(f"HTTP {r.status_code}")
        toplam = None
        cr = r.headers.get("Content-Range", "")
        if r.status_code == 206 and n > 0:
            m = re.match(r"bytes (\d+)-\d+/(\d+|\*)", cr)
            if not m or int(m.group(1)) != n:
                part.unlink()
                raise IOError(f"Content-Range uyusmuyor: {cr!r} (beklenen baslangic {n}); parca silindi")
            kip = "ab"
            if m.group(2) != "*":
                toplam = int(m.group(2))
        else:
            kip, n = "wb", 0                      # 200: sunucu Range'i yok saydı -> sıfırdan
            m = re.match(r"bytes \d+-\d+/(\d+)", cr)
            if m:
                toplam = int(m.group(1))
            elif r.headers.get("Content-Length"):
                toplam = int(r.headers["Content-Length"])
        with open(part, kip) as f:
            for c in r.iter_content(PARCA):
                if c:
                    f.write(c)
                    ilerleme(len(c))
    son = part.stat().st_size
    hedef_boyut = beklenen or toplam
    if hedef_boyut and son < hedef_boyut:
        raise IOError(f"baglanti erken kesildi: {son}/{hedef_boyut} bayt (kaldigi yerden devam edilecek)")
    return "tamam"


# ---------------------------------------------------------------- doğrulama
def ozetle(p):
    m5, s2 = hashlib.md5(), hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(PARCA), b""):
            m5.update(c)
            s2.update(c)
    return m5.hexdigest(), s2.hexdigest()


def uretici_deger(kaynak, anahtar, uzunluk):
    """Üreticinin md5/sha dosyasından anahtarın değerini bulur. Yollar '\\' ya da '/' ile olabilir."""
    for satir in kaynak.read_text(encoding="utf-8", errors="replace").splitlines():
        parcalar = satir.replace("*", " ").split()
        hexler = [p for p in parcalar if re.fullmatch(r"[0-9A-Fa-f]{%d}" % uzunluk, p)]
        if not hexler:
            continue
        for y in parcalar:
            if y in hexler:
                continue
            yn = y.replace("\\", "/")
            while yn.startswith("./") or yn.startswith("/"):
                yn = yn[2:] if yn.startswith("./") else yn[1:]
            if yn == anahtar or yn.endswith("/" + anahtar):
                return hexler[0]
    return None


def dogrula(r, p, beklenen):
    """(durum, notlar, md5, sha256). 'Dosya var' tamam sayılmaz: boyut ve sağlama kontrol edilir."""
    notlar, durum = [], "PASS"
    boyut = p.stat().st_size
    md5, sha = ozetle(p)
    with open(p, "rb") as f:
        bas = f.read(512)
    if boyut == 0:
        return "FAIL", ["0 bayt"], md5, sha
    if bas.lstrip()[:15].lower().startswith((b"<!doctype", b"<html")):
        return "FAIL", ["HTML sayfasi geldi (ilk baytlar <!DOCTYPE/<html)"], md5, sha
    boyut_tuttu = (not beklenen) or boyut == beklenen
    if beklenen and not boyut_tuttu:
        notlar.append(f"boyut {boyut} != beklenen {beklenen}")
    tur = r["saglama_turu"]
    if tur in ("md5", "sha256"):
        hesap = md5 if tur == "md5" else sha
        ok_rapor = hesap.lower() == r["saglama_degeri"].lower()
        notlar.append(f"{tur} rapor " + ("tuttu" if ok_rapor else "TUTMADI"))
        ok_uretici = True
        if r["saglama_kaynak"]:
            kp = VERI / r["saglama_kaynak"]
            u = uretici_deger(kp, r["saglama_anahtar"], 32 if tur == "md5" else 64) if kp.exists() else None
            ok_uretici = u is not None and u.lower() == hesap.lower()
            notlar.append(f"{tur} uretici dosyasi " + ("tuttu" if ok_uretici else ("YOK" if u is None else "TUTMADI")))
        durum = "PASS" if (ok_rapor and ok_uretici) else "FAIL"
        if durum == "PASS" and not boyut_tuttu:
            notlar.append("saglama tuttugu icin boyut farki raporun boyut yazimindan")
    elif tur in ("zip_crc", "glim_uc_test"):
        if tur == "glim_uc_test" and boyut < 1_000_000_000:
            notlar.append("1 GB'tan kucuk")
            durum = "FAIL"
        if bas[:2] != b"PK":
            notlar.append("ilk baytlar PK degil")
            durum = "FAIL"
        else:
            with zipfile.ZipFile(p) as z:
                bilgi = z.infolist()
                yontemler = sorted({i.compress_type for i in bilgi})
                notlar.append(f"zip: {len(bilgi)} girdi, yontemler {yontemler}")
                if 9 in yontemler:
                    notlar.append("Deflate64 var; CRC D1'de inflate64 ile")
                    if durum == "PASS":
                        durum = "KONTROL"
                else:
                    bozuk = z.testzip()
                    if bozuk:
                        notlar.append(f"CRC bozuk: {bozuk}")
                        durum = "FAIL"
                    else:
                        notlar.append("testzip temiz (CRC)")
                if r["saglama_anahtar"]:
                    es = [i for i in bilgi if i.filename.endswith(r["saglama_anahtar"])]
                    if len(es) == 1 and f"{es[0].CRC:08x}" == r["saglama_degeri"].lower():
                        notlar.append("ana dosya CRC rapor tuttu")
                    else:
                        notlar.append("ana dosya CRC rapor TUTMADI")
                        durum = "FAIL"
        if not boyut_tuttu:
            durum = "FAIL"
    elif tur == "cmr_boyut":
        mb = float(r["cmr_mb"].replace(",", "."))
        fark = min(abs(boyut - mb * 1e6) / (mb * 1e6), abs(boyut - mb * 2 ** 20) / (mb * 2 ** 20))
        notlar.append(f"CMR {r['cmr_mb']} MB ile fark %{fark * 100:.2f}")
        if fark > 0.01:
            durum = "KONTROL"
            notlar.append("CMR boyutu yaklasik olabilir; D1'de HDF acilis testi")
    else:   # 'yok': yalnız boyut
        if beklenen:
            durum = "PASS" if boyut_tuttu else "FAIL"
        else:
            durum = "KONTROL"
            notlar.append("raporda boyut ve saglama yok; D1'de icerik testi")
    return durum, notlar, md5, sha


# ---------------------------------------------------------------- işler
def dosya_isi(r, url, beklenen, yedek=""):
    hedef = VERI / r["hedef"]
    hedef.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    if hedef.exists():
        d, n, m5, s2 = dogrula(r, hedef, beklenen)
        return bitir(r, hedef, d, ["zaten vardi, yeniden dogrulandi"] + n, m5, s2, t0, 0, "", beklenen)
    part = hedef.with_name(hedef.name + ".part")
    adresler = [url] + ([yedek] if yedek else [])
    notlar, m5, s2, deneme, kullanilan = [], "", "", 0, ""
    for tur in (1, 2):               # 2. tur: doğrulama geçmezse parça silinip bir kez sıfırdan
        kullanilan, son_hata = None, None
        for a in adresler:
            try:
                _, deneme = tekrarla(f"{r['set']} {hedef.name}", lambda a=a: http_indir(a, part, beklenen))
                kullanilan = a
                break
            except Exception as e:
                son_hata = e
                say(f"   ! {r['set']} {hedef.name}: {a} -> {type(e).__name__}: {str(e)[:200]}")
        if kullanilan is None:
            return bitir(r, part if part.exists() else None, "FAIL", [f"indirilemedi: {son_hata}"],
                         "", "", t0, deneme, " ; ".join(adresler), beklenen)
        d, notlar, m5, s2 = dogrula(r, part, beklenen)
        if d != "FAIL":
            if kullanilan != url:
                notlar.append("YEDEK adresten indi")
            os.replace(part, hedef)
            os.chmod(hedef, stat.S_IREAD)            # veri\ salt okunur (belge 17.2)
            return bitir(r, hedef, d, notlar, m5, s2, t0, deneme, kullanilan, beklenen)
        if tur == 1:
            say(f"   ! {r['set']} {hedef.name} dogrulama gecmedi ({'; '.join(notlar)}); sifirdan bir kez daha")
            part.unlink()
    return bitir(r, part, "FAIL", notlar + ["ikinci indirmede de gecmedi; .part birakildi"],
                 m5, s2, t0, deneme, kullanilan, beklenen)


def earthdata_isi(r, granul):
    import earthaccess
    hedef = VERI / r["hedef"]
    hedef.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    if hedef.exists():
        d, n, m5, s2 = dogrula(r, hedef, None)
        return bitir(r, hedef, d, ["zaten vardi, yeniden dogrulandi"] + n, m5, s2, t0, 0, "earthaccess", None)
    g, link = granul
    tmp = GECICI / hedef.stem

    def bir_kez():
        # Earthdata dosyaları belge 17.2'deki gibi earthaccess.download ile iner; Range ile devam yok,
        # bu yüzden her denemede geçici klasör temizlenir, tamamlanan dosya doğrulanınca veri\'ye taşınır.
        shutil.rmtree(tmp, ignore_errors=True)
        tmp.mkdir(parents=True)
        with ED_KILIT:
            yollar = earthaccess.download([g], local_path=str(tmp), threads=1)
        p = tmp / hedef.name
        if not p.exists() or p.stat().st_size == 0:
            raise IOError(f"earthaccess dosyayi indirmedi: {yollar}")
        return p

    try:
        p, deneme = tekrarla(f"{r['set']} {hedef.name}", bir_kez)
    except Exception as e:
        return bitir(r, None, "FAIL", [f"indirilemedi: {e}"], "", "", t0, len(BEKLEMELER) + 1, link, None)
    d, notlar, m5, s2 = dogrula(r, p, None)
    if d != "FAIL":
        os.replace(p, hedef)
        os.chmod(hedef, stat.S_IREAD)
        shutil.rmtree(tmp, ignore_errors=True)
        return bitir(r, hedef, d, notlar, m5, s2, t0, deneme, link, None)
    return bitir(r, p, "FAIL", notlar, m5, s2, t0, deneme, link, None)


# ---------------------------------------------------------------- E1 karoları
class _Href(HTMLParser):
    def __init__(self):
        super().__init__()
        self.h = []

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            for k, v in attrs:
                if k == "href" and v:
                    self.h.append(v)


def e1_karolar(r, beklemeler=BEKLEMELER):
    """Klasör listesinden _surface.tif karolarını okur; sayı 288, HEAD boyutlarının toplamı raporla aynı olmalı."""
    def liste():
        x = oturum().get(r["url"], timeout=ZAMAN_ASIMI)
        x.raise_for_status()
        return x.text
    html, _ = tekrarla("E1 klasor listesi", liste, beklemeler)
    p = _Href()
    p.feed(html)
    adlar = sorted({h.rsplit("/", 1)[-1] for h in p.h
                    if re.fullmatch(r"ETOPO_2022_v1_15s_[NS]\d\d[EW]\d{3}_surface\.tif", h.rsplit("/", 1)[-1])})
    assert len(adlar) == int(r["beklenen_adet"]), f"E1 karo sayisi {len(adlar)}, rapor {r['beklenen_adet']}"

    def boyut(ad):
        def head():
            x = oturum().head(r["url"] + ad, timeout=ZAMAN_ASIMI, allow_redirects=True)
            if x.status_code >= 400:
                raise IOError(f"HTTP {x.status_code}")
            return int(x.headers["Content-Length"])
        return tekrarla(f"E1 HEAD {ad}", head, beklemeler)[0]

    with ThreadPoolExecutor(ESZAMANLI) as ex:
        boyutlar = list(ex.map(boyut, adlar))
    toplam = sum(boyutlar)
    assert toplam == int(r["beklenen_bayt"]), f"E1 toplam {toplam} bayt, rapor {r['beklenen_bayt']}"
    with open(KAYIT / "E1_karolar.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["ad", "adres", "bayt_HEAD"])
        for a, b in zip(adlar, boyutlar):
            w.writerow([a, r["url"] + a, b])
    say(f"E1: {len(adlar)} karo, HEAD toplami {toplam} bayt (rapor {r['beklenen_bayt']}) -> kayit/E1_karolar.csv")
    karolar = []
    for a, b in zip(adlar, boyutlar):
        k = dict(r)
        k.update(hedef="dunya/E1_etopo/" + a, url=r["url"] + a, beklenen_bayt=str(b), beklenen_adet="",
                 yontem="https", saglama_turu="yok")
        karolar.append(k)
    return karolar


# ---------------------------------------------------------------- Earthdata arama
def earthdata_hazirla(satirlar, beklemeler=BEKLEMELER):
    import earthaccess
    os.environ.setdefault("NETRC", str(Path.home() / "_netrc"))   # şifre okunmaz, yazılmaz
    auth = earthaccess.login(strategy="netrc")
    assert getattr(auth, "authenticated", False), "Earthdata girisi basarisiz (_netrc)"
    gruplar = {}
    for r in satirlar:
        gruplar.setdefault((r["short_name"], r["version"], r["temporal"]), []).append(r)
    eslesme = {}
    for (sn, v, t), rs in gruplar.items():
        a, b = t.split("|")
        sonuc, _ = tekrarla(f"CMR {sn}", lambda: earthaccess.search_data(short_name=sn, version=v, temporal=(a, b)),
                            beklemeler)
        harita = {}
        for g in sonuc:
            for l in g.data_links():
                harita.setdefault(l.rsplit("/", 1)[-1], (g, l))
        for r in rs:
            ad = r["hedef"].rsplit("/", 1)[-1]
            assert ad in harita, f"{r['set']} {ad} CMR aramasinda bulunamadi"
            eslesme[r["hedef"]] = harita[ad]
        say(f"CMR {sn} v{v} {a}..{b}: {len(sonuc)} granul bulundu, {len(rs)} dosya ada gore eslendi")
    return eslesme


# ---------------------------------------------------------------- kuru koşu
def yokla(url):
    """İlk 16 baytı ister (Range 0-15): HTTP durumu, sunucudaki toplam boyut, ilk baytlar."""
    with oturum().get(url, headers={"Range": "bytes=0-15"}, stream=True, timeout=ZAMAN_ASIMI,
                      allow_redirects=True) as x:
        if x.status_code >= 500 or x.status_code in (408, 429):
            raise IOError(f"HTTP {x.status_code}")
        if x.status_code >= 400:
            raise KaliciHata(f"HTTP {x.status_code}")
        bas = x.raw.read(16)
        m = re.match(r"bytes \d+-\d+/(\d+)", x.headers.get("Content-Range", ""))
        if m:
            toplam = int(m.group(1))
        elif x.status_code == 200 and x.headers.get("Content-Length"):
            toplam = int(x.headers["Content-Length"])
        else:
            toplam = None
        return x.status_code, toplam, bas


def kuru(satirlar):
    say("KURU KOSU: dosya indirilmez; sunucular, boyutlar ve Earthdata aramasi sinanir")
    alanlar = ["set", "hedef", "adres", "http", "sunucu_boyut", "beklenen_bayt", "boyut", "ilk_baytlar", "durum", "hata"]
    cikti = []
    for r in [x for x in satirlar if x["yontem"] == "klasor"]:
        try:
            e1_karolar(r, KURU_BEKLEMELER)
            cikti.append({"set": "E1", "hedef": r["hedef"], "adres": r["url"], "durum": "OK",
                          "boyut": "288 karo ve toplam bayt rapora esit"})
        except Exception as e:
            cikti.append({"set": "E1", "hedef": r["hedef"], "adres": r["url"], "durum": "SORUN", "hata": str(e)})
    ed = [x for x in satirlar if x["yontem"] == "earthaccess"]
    if ed:
        try:
            es = earthdata_hazirla(ed, KURU_BEKLEMELER)
            for r in ed:
                g, link = es[r["hedef"]]
                try:
                    mb = round(float(g.size()), 3)
                except Exception:
                    mb = ""
                cikti.append({"set": r["set"], "hedef": r["hedef"], "adres": link, "sunucu_boyut": f"{mb} MB (CMR)",
                              "beklenen_bayt": r["cmr_mb"] + " MB (rapor)", "durum": "OK"})
        except Exception as e:
            cikti.append({"set": "Earthdata", "durum": "SORUN", "hata": f"{type(e).__name__}: {e}"})
    for r in [x for x in satirlar if x["yontem"] in ("https", "http")]:
        adresler = [r["url"]] + ([yedek_adresi(r)] if r["yedek_url"] else [])
        satir = {"set": r["set"], "hedef": r["hedef"], "beklenen_bayt": r["beklenen_bayt"]}
        for a in adresler:
            try:
                (kod, toplam, bas), _ = tekrarla(f"{r['set']} yokla", lambda a=a: yokla(a), KURU_BEKLEMELER)
                html = bas.lstrip()[:15].lower().startswith((b"<!doctype", b"<html"))
                if r["beklenen_bayt"] and toplam is not None:
                    bd = "ayni" if int(r["beklenen_bayt"]) == toplam else "FARKLI"
                else:
                    bd = "bilinmiyor"
                satir.update(adres=a, http=kod, sunucu_boyut=toplam, ilk_baytlar=bas.hex(" "), boyut=bd,
                             durum="SORUN" if (html or bd == "FARKLI") else ("OK" if a == r["url"] else "OK (yedek)"),
                             hata="HTML geldi" if html else "")
                break
            except Exception as e:
                satir.update(adres=a, durum="ULASILAMADI", hata=f"{type(e).__name__}: {str(e)[:200]}")
        cikti.append(satir)
        say(f"   {satir['durum']:12} {r['set']:4} {r['hedef'].rsplit('/', 1)[-1]:48} "
            f"http={satir.get('http', '')} sunucu={satir.get('sunucu_boyut', '')} beklenen={r['beklenen_bayt']} {satir.get('hata', '')}")
    with open(KAYIT / "indirme_kuru.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=alanlar)
        w.writeheader()
        for c in cikti:
            w.writerow({a: c.get(a, "") for a in alanlar})
    say_ = {}
    for c in cikti:
        say_[c["durum"]] = say_.get(c["durum"], 0) + 1
    say(f"KURU KOSU BITTI: {say_} -> kayit/indirme_kuru.csv")
    for c in cikti:
        if c["durum"] not in ("OK", "OK (yedek)"):
            say(f"   SORUNLU: {c.get('set')} {c.get('hedef', '')} {c.get('adres', '')} {c.get('hata', '')} boyut={c.get('boyut', '')}")


def yedek_adresi(r):
    y = r["yedek_url"]
    if y and "dropbox.com" in y and "dl=" not in y:
        y += "?dl=1"                        # rapor E3: yedek bağlantıya ?dl=1 eklenir
    return y


# ---------------------------------------------------------------- ana akış
def main():
    ap = argparse.ArgumentParser(description="Orbitrary v1.2 veri indirme (belge 17.2)")
    ap.add_argument("--kuru", action="store_true", help="indirmeden sina")
    ap.add_argument("--set", nargs="*", default=None, help="yalniz bu setler (or. E3 R5)")
    a = ap.parse_args()

    sys.path.insert(0, str(KOK / "kod"))
    import liste_denetim
    d = liste_denetim.denetle()
    assert d["durum"] == "PASS", f"liste denetimi gecmedi: {d['hatalar'][:5]}"
    say(f"Liste denetimi PASS: {d['satir']} satir, {d['dosya']} dosya, rapor sha256 {d['rapor_sha256'][:16]}...")

    with open(LISTE, encoding="utf-8", newline="") as f:
        tum = list(csv.DictReader(f))
    satirlar = [r for r in tum if (a.set is None or r["set"] in a.set)]
    assert satirlar, "secilen set yok"
    bos = shutil.disk_usage(KOK).free
    say(f"Bos disk: {bos / 1e9:.1f} GB")
    assert bos >= EN_AZ_BOS_DISK, f"bos disk {bos / 1e9:.1f} GB < {EN_AZ_BOS_DISK / 1e9:.0f} GB"
    VERI.mkdir(exist_ok=True)

    if a.kuru:
        kuru(satirlar)
        return 0

    t_bas = time.time()
    durduruldu = threading.Event()

    def durum_yazici():
        while not durduruldu.wait(60):
            say(f"... bu oturumda indirilen {_indirilen[0] / 1e9:.2f} GB, gecen {(time.time() - t_bas) / 60:.1f} dk")
    threading.Thread(target=durum_yazici, daemon=True).start()

    def guvenli(f, *x):
        try:
            return f(*x)
        except Exception as e:
            r = x[0]
            return bitir(r, None, "FAIL", [f"beklenmeyen hata: {type(e).__name__}: {e}"], "", "", time.time(), 0, "", None)

    # Faz A: üreticinin sağlama dosyaları (diğer dosyaların doğrulaması bunlara bakar)
    gereken = {r["saglama_kaynak"] for r in satirlar if r["saglama_kaynak"]}
    faz_a = [r for r in tum if r["hedef"] in gereken]
    say(f"FAZ A: {len(faz_a)} uretici saglama dosyasi")
    for r in faz_a:
        guvenli(dosya_isi, r, r["url"], int(r["beklenen_bayt"]) if r["beklenen_bayt"] else None)
    a_hedef = {r["hedef"] for r in faz_a}

    # Faz B: grup 1 ve 2, en fazla 4 eşzamanlı, büyük dosyalar önce
    isler = []
    for r in satirlar:
        if r["hedef"] in a_hedef or r["grup"] == "3":
            continue
        if r["yontem"] == "klasor":
            for k in e1_karolar(r):
                isler.append((int(r["grup"]), int(k["beklenen_bayt"]), "http", k))
        elif r["yontem"] == "earthaccess":
            isler.append((int(r["grup"]), float(r["cmr_mb"].replace(",", ".")) * 1e6, "ed", r))
        else:
            isler.append((int(r["grup"]), int(r["beklenen_bayt"] or 0), "http", r))
    eslesme = earthdata_hazirla([x[3] for x in isler if x[2] == "ed"]) if any(x[2] == "ed" for x in isler) else {}
    isler.sort(key=lambda x: (x[0], -x[1]))
    say(f"FAZ B: {len(isler)} dosya, {ESZAMANLI} esanli")
    with ThreadPoolExecutor(ESZAMANLI) as ex:
        fut = []
        for _, _, tur, r in isler:
            if tur == "ed":
                fut.append(ex.submit(guvenli, earthdata_isi, r, eslesme[r["hedef"]]))
            else:
                fut.append(ex.submit(guvenli, dosya_isi, r, r["url"],
                                     int(r["beklenen_bayt"]) if r["beklenen_bayt"] else None, yedek_adresi(r)))
        for _ in as_completed(fut):
            pass

    # Faz C: kararsız sunucular (SWIM, ASU, PSI, Dropbox) tek tek
    faz_c = [r for r in satirlar if r["grup"] == "3" and r["hedef"] not in a_hedef]
    say(f"FAZ C: {len(faz_c)} dosya, tek tek (kararsiz sunucular)")
    for r in faz_c:
        guvenli(dosya_isi, r, r["url"], int(r["beklenen_bayt"]) if r["beklenen_bayt"] else None, yedek_adresi(r))
    durduruldu.set()

    # Özet
    ozet = {"bitis": datetime.now().isoformat(timespec="seconds"),
            "sure_dk": round((time.time() - t_bas) / 60, 1),
            "bu_oturumda_indirilen_gb": round(_indirilen[0] / 1e9, 3),
            "secilen_setler": a.set or "hepsi", "durum_sayilari": {}, "set_basina": {},
            "FAIL": [], "KONTROL": []}
    for k in SONUCLAR:
        ozet["durum_sayilari"][k["durum"]] = ozet["durum_sayilari"].get(k["durum"], 0) + 1
        sb = ozet["set_basina"].setdefault(k["set"], {})
        sb[k["durum"]] = sb.get(k["durum"], 0) + 1
        if k["durum"] in ("FAIL", "KONTROL"):
            ozet[k["durum"]].append({"hedef": k["hedef"], "not": k["not"]})
    (KAYIT / "indirme_ozet.json").write_text(json.dumps(ozet, ensure_ascii=False, indent=2), encoding="utf-8")
    with open(KAYIT / "13_kasim_log.txt", "a", encoding="utf-8") as f:
        f.write(f"{datetime.now():%Y-%m-%d %H:%M:%S}  2b indirme ({ozet['secilen_setler']}): {ozet['durum_sayilari']}, "
                f"{ozet['sure_dk']} dk, {ozet['bu_oturumda_indirilen_gb']} GB\n")
    say("=" * 60)
    say(f"BITTI: {ozet['durum_sayilari']} | {ozet['sure_dk']} dk | {ozet['bu_oturumda_indirilen_gb']} GB")
    for k in ozet["FAIL"]:
        say(f"   FAIL    {k['hedef']}: {k['not'][:200]}")
    for k in ozet["KONTROL"]:
        say(f"   KONTROL {k['hedef']}: {k['not'][:200]}")
    say("Ayrintilar: kayit/indirme.csv, kayit/indirme_ozet.json")
    return 1 if ozet["FAIL"] else 0


if __name__ == "__main__":
    sys.exit(main())
'''
compile(kod, "indir_hepsi.py", "exec")          # sözdizimi hatası varsa burada durur
dosya = KOK / "kod" / "indir_hepsi.py"
dosya.write_text(kod, encoding="utf-8")
assert dosya.read_text(encoding="utf-8").startswith("# Developed with assistance from Claude (Anthropic).")
print("Yazıldı:", dosya)
print("Satır:", len(kod.splitlines()), "| SHA-256 ilk 16:", hashlib.sha256(kod.encode("utf-8")).hexdigest()[:16])

Yazıldı: C:\Users\USER\orbitrary\kod\indir_hepsi.py
Satır: 606 | SHA-256 ilk 16: 5fda2330d42350f2


In [11]:
# Developed with assistance from Claude (Anthropic).
# Adım 2g — (1) veri\'nin gerçek boyutu, (2) R6 incelemesinin kayda yazılması, (3) indir_hepsi.py sayaç düzeltmesi.
# Ham dosyalara dokunmaz (yalnız okur). indir_hepsi.py'de tek satır değişir; eski metin bulunamazsa durur.
from pathlib import Path
from datetime import datetime
import csv, json, hashlib, re

KOK = Path.home() / "orbitrary"
VERI, KAYIT = KOK / "veri", KOK / "kayit"

# ---- 1) veri\ içinde gerçekte ne var
with open(KOK / "kod" / "indirme_listesi.csv", encoding="utf-8", newline="") as f:
    liste = list(csv.DictReader(f))
set_bayt, set_adet, part = {}, {}, []
for p in VERI.rglob("*"):
    if not p.is_file():
        continue
    rel = p.relative_to(VERI).as_posix()
    s = rel.split("/")[1].split("_")[0]          # dunya/E1_etopo/... -> E1
    set_bayt[s] = set_bayt.get(s, 0) + p.stat().st_size
    set_adet[s] = set_adet.get(s, 0) + 1
    if p.name.endswith(".part"):
        part.append(rel)
toplam = sum(set_bayt.values())
sira = [f"E{i}" for i in range(1, 11)] + [f"M{i}" for i in range(1, 8)] + [f"R{i}" for i in range(1, 8)]
print(f"{'set':5} {'dosya':>6} {'GB':>8}")
for s in sira:
    print(f"{s:5} {set_adet.get(s, 0):>6} {set_bayt.get(s, 0) / 1e9:>8.3f}")
print(f"TOPLAM {sum(set_adet.values()):>5} {toplam / 1e9:>8.3f} GB  ({toplam} bayt)")
print(f".part (doğrulanmamış) dosya: {len(part)}")

# ---- 2) R6 incelemesi (2e ve 2f'nin doğru yollarla tekrarı) -> kayit
R6 = VERI / "mars" / "R6_odyssey_grs"
r6 = [r for r in liste if r["set"] == "R6" and r["saglama_turu"] == "md5"]
liste_hex = set(h.lower() for h in re.findall(r"\b[0-9A-Fa-f]{32}\b",
                (R6 / "odgm_xxxx_130911.md5").read_text(encoding="utf-8", errors="replace")))
inceleme = []
for r in r6:
    ad = r["hedef"].rsplit("/", 1)[-1]
    ham = (R6 / (ad + ".part")).read_bytes()
    lf = ham.replace(b"\r\n", b"\n")
    inceleme.append({"dosya": ad, "bayt": len(ham), "beklenen_bayt": int(r["beklenen_bayt"]),
                     "md5_gercek": hashlib.md5(ham).hexdigest(), "md5_liste": r["saglama_degeri"].lower(),
                     "md5_crlf_lf": hashlib.md5(lf).hexdigest(),
                     "gercek_md5_listenin_herhangi_bir_satirinda": hashlib.md5(ham).hexdigest() in liste_hex,
                     "lf_md5_listenin_herhangi_bir_satirinda": hashlib.md5(lf).hexdigest() in liste_hex,
                     "satir_sayisi": ham.count(b"\r\n")})
kayit = {"tarih": datetime.now().isoformat(timespec="seconds"),
         "kaynak_liste": "veri/mars/R6_odyssey_grs/odgm_xxxx_130911.md5",
         "bulgu": "Boyutlar rapor ve sunucuyla ayni; icerik gecerli PDS3 tablosu (CRLF, 2592 satir); "
                  "uretici MD5 listesi ne ham dosyayla ne CRLF->LF donusumuyle tutuyor. Neden: acik.",
         "boyut_tutan": sum(x["bayt"] == x["beklenen_bayt"] for x in inceleme),
         "md5_tutan": sum(x["md5_gercek"] == x["md5_liste"] for x in inceleme),
         "lf_md5_tutan": sum(x["md5_crlf_lf"] == x["md5_liste"] for x in inceleme),
         "dosyalar": inceleme, "durum": "ACIK - karar bekliyor"}
(KAYIT / "R6_inceleme.json").write_text(json.dumps(kayit, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nR6: boyut tutan {kayit['boyut_tutan']}/12, MD5 tutan {kayit['md5_tutan']}/12, "
      f"CRLF->LF ile tutan {kayit['lf_md5_tutan']}/12 -> kayit\\R6_inceleme.json")

# ---- 3) indir_hepsi.py: Earthdata dosyaları da 'indirilen GB' sayacına girsin
IH = KOK / "kod" / "indir_hepsi.py"
metin = IH.read_text(encoding="utf-8")
eski = "        if not p.exists() or p.stat().st_size == 0:\n            raise IOError(f\"earthaccess dosyayi indirmedi: {yollar}\")\n        return p\n"
yeni = "        if not p.exists() or p.stat().st_size == 0:\n            raise IOError(f\"earthaccess dosyayi indirmedi: {yollar}\")\n        ilerleme(p.stat().st_size)          # Earthdata dosyalari da sayaca girer\n        return p\n"
if yeni in metin:
    print("\nindir_hepsi.py sayaç düzeltmesi: ZATEN VAR")
else:
    assert metin.count(eski) == 1, "indir_hepsi.py: değiştirilecek metin BULUNAMADI, dosya değişmedi"
    metin = metin.replace(eski, yeni)
    compile(metin, "indir_hepsi.py", "exec")
    IH.write_text(metin, encoding="utf-8")
    print("\nindir_hepsi.py sayaç düzeltmesi: OK")

with open(KAYIT / "13_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(f"{datetime.now():%Y-%m-%d %H:%M:%S}  2g veri\\ toplam {toplam} bayt, {sum(set_adet.values())} dosya, "
            f"{len(part)} .part; R6 md5 {kayit['md5_tutan']}/12 (acik); sayac duzeltmesi\n")
(KAYIT / "veri_ozet.json").write_text(json.dumps(
    {"tarih": datetime.now().isoformat(timespec="seconds"), "toplam_bayt": toplam,
     "set_bayt": set_bayt, "set_dosya": set_adet, "part": part}, ensure_ascii=False, indent=2), encoding="utf-8")

set    dosya       GB
E1       288    5.231
E2         1    1.244
E3         1    1.139
E4        12    4.044
E5        12    0.897
E6        12    2.480
E7         1    0.106
E8         1    0.001
E9         2    0.646
E10        4    0.386
M1         3    8.494
M2         1    0.224
M3         5    6.651
M4        18    1.835
M5         4    0.010
M6         6    0.017
M7         8    0.123
R1         3    2.126
R2         2    0.809
R3         4    0.108
R4         3    0.068
R5         1    0.066
R6        13    0.001
R7         3    0.213
TOPLAM   408   36.920 GB  (36919609088 bayt)
.part (doğrulanmamış) dosya: 12

R6: boyut tutan 12/12, MD5 tutan 0/12, CRLF->LF ile tutan 0/12 -> kayit\R6_inceleme.json

indir_hepsi.py sayaç düzeltmesi: OK


1500

In [12]:
# Developed with assistance from Claude (Anthropic).
# Adım 2h — R6: arşivin 2013 MD5 listesi PDS4 göçünden (2026) önce kaldığı için doğrulama PDS4 etiketleriyle.
# Ham içerik değiştirilmez; yalnız .part adı kaldırılır ve salt okunur yapılır. Bir test geçmezse hiçbir şey değişmez.
from pathlib import Path
from datetime import datetime
import requests, re, os, stat, csv, json, hashlib

KOK = Path.home() / "orbitrary"
R6 = KOK / "veri" / "mars" / "R6_odyssey_grs"
ETK = R6 / "pds4_etiket"
ETK.mkdir(exist_ok=True)
TABAN = "https://pds-geosciences.wustl.edu/ody/ody-m-grs-5-elements-v1/odgm_xxxx/data/smoothed/"
s = requests.Session()
s.headers.update({"User-Agent": "Orbitrary/1.2 (NASA Space Apps 2026; research data download)",
                  "Accept-Encoding": "identity"})

def al(rx, metin):
    m = re.search(rx, metin, re.S)
    return m.group(1).strip() if m else None

sonuc, hata = [], []
for el in ["h2o", "fe", "si", "k", "th", "cl"]:
    # 1) PDS4 etiketi
    xp = ETK / f"{el}_sr_5x5.xml"
    if not xp.exists():
        r = s.get(TABAN + xp.name, timeout=(30, 120))
        r.raise_for_status()
        assert not r.content.lstrip()[:15].lower().startswith((b"<!doctype", b"<html")), f"{xp.name}: HTML geldi"
        xp.write_bytes(r.content)
        os.chmod(xp, stat.S_IREAD)
    x = xp.read_text(encoding="utf-8")
    kayit_sayi = int(al(r"<records>\s*(\d+)\s*</records>", x))
    kayit_uz = int(al(r"<record_length[^>]*>\s*(\d+)\s*</record_length>", x))
    dosya_adi = al(r"<file_name>\s*([^<]+?)\s*</file_name>", x)
    surum = al(r"<version_id>\s*([^<]+?)\s*</version_id>", x)
    goc = "Migration to PDS4" in x
    mod_tarih = al(r"<modification_date>\s*([^<]+?)\s*</modification_date>", x)

    # 2) .tab
    tp = R6 / f"{el}_sr_5x5.tab.part"
    ham = tp.read_bytes()
    satirlar = ham.split(b"\r\n")
    t = {"ad_tutuyor": (dosya_adi or "").lower() == f"{el}_sr_5x5.tab",
         "boyut_eq_records_x_length": len(ham) == kayit_sayi * kayit_uz,
         "satir_sayisi_tutuyor": ham.count(b"\r\n") == kayit_sayi and ham.endswith(b"\r\n"),
         "her_satir_record_length": all(len(l) + 2 == kayit_uz for l in satirlar[:-1]),
         "pds4_gocu_kaydi": goc, "surum_1_0": surum == "1.0"}
    # 3) PDS3 .lbl (eski etiket): yapı ve kayıt sayısı tutuyor mu; RECORD_BYTES farkı not edilir
    lp = R6 / f"{el}_sr_5x5.lbl.part"
    lbl = lp.read_text(encoding="ascii", errors="replace")
    l = {"pds3": "PDS_VERSION_ID" in lbl and "PDS3" in lbl,
         "table_adi": (al(r'\^TABLE\s*=\s*"([^"]+)"', lbl) or "").lower() == f"{el}_sr_5x5.tab",
         "file_records": int(al(r"FILE_RECORDS\s*=\s*(\d+)", lbl) or -1) == kayit_sayi}
    rb_pds3 = int(al(r"RECORD_BYTES\s*=\s*(\d+)", lbl) or -1)
    gecti = all(t.values()) and all(l.values())
    if not gecti:
        hata.append(el)
    sonuc.append({"element": el, "pds4_records": kayit_sayi, "pds4_record_length": kayit_uz,
                  "pds3_RECORD_BYTES": rb_pds3, "pds4_modification_date": mod_tarih, "pds4_version_id": surum,
                  "tab_bayt": len(ham), "tab_testleri": t, "lbl_testleri": l, "gecti": gecti,
                  "tab_sha256": hashlib.sha256(ham).hexdigest(),
                  "lbl_sha256": hashlib.sha256(lp.read_bytes()).hexdigest()})
    print(f"{el:4} records {kayit_sayi} x {kayit_uz} = {kayit_sayi * kayit_uz} | tab {len(ham)} | "
          f"PDS3 RECORD_BYTES {rb_pds3} | göç kaydı {goc} {mod_tarih} | sürüm {surum} | "
          + ("GEÇTİ" if gecti else f"GEÇMEDİ {t} {l}"))

assert not hata, f"Geçmeyen: {hata} — hiçbir dosya değiştirilmedi"

# 4) Kabul: .part -> asıl ad, salt okunur; indirme kaydına ekle
LOG = KOK / "kayit" / "indirme.csv"
alanlar = ["zaman", "set", "hedef", "adres", "bayt", "beklenen_bayt", "sha256", "md5", "sure_s", "deneme", "durum", "not"]
with open(LOG, "a", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=alanlar)
    for x in sonuc:
        for uz in ("tab", "lbl"):
            part = R6 / f"{x['element']}_sr_5x5.{uz}.part"
            son = R6 / f"{x['element']}_sr_5x5.{uz}"
            ham = part.read_bytes()
            os.replace(part, son)
            os.chmod(son, stat.S_IREAD)
            w.writerow({"zaman": datetime.now().isoformat(timespec="seconds"), "set": "R6",
                        "hedef": f"mars/R6_odyssey_grs/{son.name}", "adres": TABAN + son.name,
                        "bayt": len(ham), "beklenen_bayt": len(ham), "sha256": hashlib.sha256(ham).hexdigest(),
                        "md5": hashlib.md5(ham).hexdigest(), "sure_s": 0, "deneme": 0, "durum": "PASS",
                        "not": "PDS4 etiketiyle dogrulandi (records x record_length, satir uzunlugu, PDS4 goc kaydi "
                               "2026); uretici MD5 listesi 2013 tarihli, PDS4 gocunden once (bkz. kayit/R6_karar.json)"})
karar = {"tarih": datetime.now().isoformat(timespec="seconds"),
         "karar": "R6, 2013 MD5 listesi yerine PDS4 (2026) etiketleriyle dogrulandi ve kabul edildi; ham icerik degismedi.",
         "neden": "Arsiv Eylul-Ekim 2026'da PDS4'e gocmus; .tab/.lbl yeniden yazilmis, MD5 listesi 2013'te kalmis. "
                  "Surum 1.0 degismemis.",
         "okuma_notu": "PDS3 .lbl RECORD_BYTES (71/65) eski; dosyada satir PDS4 record_length (72/66) bayt, CRLF. "
                       "D3'te PDS4 etiketine gore ya da satir satir okunur.",
         "kaynaklar": ["https://pds-geosciences.wustl.edu/ody/ody-m-grs-5-elements-v1/",
                       "https://pds-geosciences.wustl.edu/ody/ody-m-grs-5-elements-v1/odgm_xxxx/data/smoothed/",
                       TABAN + "h2o_sr_5x5.xml", "doi:10.17189/1519480"],
         "elementler": sonuc}
(KOK / "kayit" / "R6_karar.json").write_text(json.dumps(karar, ensure_ascii=False, indent=2), encoding="utf-8")
with open(KOK / "kayit" / "13_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(f"{datetime.now():%Y-%m-%d %H:%M:%S}  2h R6 PDS4 etiketiyle dogrulandi ve kabul edildi (12/12); "
            f"6 PDS4 .xml veri/mars/R6_odyssey_grs/pds4_etiket/ altina indi\n")
print("\nR6: 12/12 dosya kabul edildi, .part kaldırıldı, salt okunur. Kayıt: kayit\\R6_karar.json")

h2o  records 2592 x 72 = 186624 | tab 186624 | PDS3 RECORD_BYTES 71 | göç kaydı True 2026-10-01 | sürüm 1.0 | GEÇTİ
fe   records 2592 x 72 = 186624 | tab 186624 | PDS3 RECORD_BYTES 71 | göç kaydı True 2026-10-01 | sürüm 1.0 | GEÇTİ
si   records 2592 x 72 = 186624 | tab 186624 | PDS3 RECORD_BYTES 71 | göç kaydı True 2026-10-01 | sürüm 1.0 | GEÇTİ
k    records 2592 x 66 = 171072 | tab 171072 | PDS3 RECORD_BYTES 65 | göç kaydı True 2026-10-01 | sürüm 1.0 | GEÇTİ
th   records 2592 x 66 = 171072 | tab 171072 | PDS3 RECORD_BYTES 65 | göç kaydı True 2026-10-01 | sürüm 1.0 | GEÇTİ
cl   records 2592 x 72 = 186624 | tab 186624 | PDS3 RECORD_BYTES 71 | göç kaydı True 2026-10-01 | sürüm 1.0 | GEÇTİ

R6: 12/12 dosya kabul edildi, .part kaldırıldı, salt okunur. Kayıt: kayit\R6_karar.json


In [13]:
# Developed with assistance from Claude (Anthropic).
# Adım D1 — kod\d1_butunluk.py yazılır ve çalıştırılır (belge 17.3 D1 + 17.2 zip açma).
from pathlib import Path
import runpy

KOK = Path.home() / "orbitrary"
kod = r'''# Developed with assistance from Claude (Anthropic).
# d1_butunluk.py — Orbitrary v1.2, belge 17.3 D1 (bütünlük) + 17.2 zip açma.
# Girdi : veri/ (ham, salt okunur), kod/indirme_listesi.csv, kayit/E1_karolar.csv, kayit/indirme.csv
# Çıktı : veri/<govde>/<set>_<ad>/acik/  (zip içerikleri; ham zip silinmez)
#         kayit/D1.json, kayit/D1_dosyalar.csv, kayit/13_kasim_log.txt'ye bir satır
# Testler: (1) beklenen dosyalar var, fazla yok; (2) 0 bayt / HTML yok; (3) boyut = rapor;
#          (4) SHA-256 = indirme kaydı (indirmeden sonra değişmedi); (5) KONTROL dosyalarının içerik testi;
#          (6) zip açma (CRC zipfile tarafından, boyut = zip dizini); (7) açılış testleri (HDF4/HDF5/GeoTIFF/ENVI/vektör);
#          (8) raporun ek testleri (M5 11306 x 880, M6 259200 x 5). Geçmeyen test silinmez, FAIL yazılır.
import csv, hashlib, json, os, re, stat, sys, time, zipfile
from datetime import datetime
from pathlib import Path

KOK = Path.home() / "orbitrary"
VERI, KAYIT = KOK / "veri", KOK / "kayit"
PARCA = 8 * 1024 * 1024
IZINLI_EKSIK = {"mars/R4_tes_ti_albedo/nmsk2003.bin"}           # se.psi.edu açılmadı (kayıt: indirme.csv)
IZINLI_FAZLA_ON = "mars/R6_odyssey_grs/pds4_etiket/"           # R6 PDS4 etiketleri (kayıt: R6_karar.json)
ZIPLER = ["dunya/E3_glim/LiMW_GIS 2015.gdb.zip",
          "dunya/E7_permafrost_kuzey/UiO_PEX_PERPROB_5.0_20181128_2000_2016_NH.zip",
          "dunya/E8_permafrost_antarktika/UiO_PEX_PERPROB_5.0_20181205_2000_2017_ANT.zip",
          "dunya/E9_aridity/Global-AI_ET0__annual_v3_1.zip",
          "ay/M2_ay_jeoloji/Unified_Geologic_Map_of_the_Moon_GIS_v2.zip",
          "mars/R2_sim3292/sim3292_database.zip"]

TESTLER = []


def test(ad, gecti, detay=""):
    TESTLER.append({"test": ad, "durum": "PASS" if gecti else "FAIL", "detay": str(detay)})
    print(("[PASS] " if gecti else "[FAIL] ") + ad + ": " + str(detay)[:220], flush=True)


def dene(ad, f):
    try:
        g, d = f()
        test(ad, g, d)
    except Exception as e:
        test(ad, False, f"{type(e).__name__}: {e}")


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(PARCA), b""):
            h.update(c)
    return h.hexdigest()


def main():
    t0 = time.time()
    from osgeo import gdal
    gdal.UseExceptions()
    import rasterio, pyogrio, h5py

    # ------------------------------------------------ beklenen dosyalar
    with open(KOK / "kod" / "indirme_listesi.csv", encoding="utf-8", newline="") as f:
        liste = list(csv.DictReader(f))
    with open(KAYIT / "E1_karolar.csv", encoding="utf-8", newline="") as f:
        karolar = list(csv.DictReader(f))
    beklenen = {}
    for r in liste:
        if r["yontem"] == "klasor":
            for k in karolar:
                beklenen["dunya/E1_etopo/" + k["ad"]] = dict(r, beklenen_bayt=k["bayt_HEAD"])
        else:
            beklenen[r["hedef"]] = r
    son = {}
    with open(KAYIT / "indirme.csv", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            if row["durum"] in ("PASS", "KONTROL"):
                son[row["hedef"]] = row                     # aynı dosyanın en son geçerli kaydı

    ham = sorted(p.relative_to(VERI).as_posix() for p in VERI.rglob("*")
                 if p.is_file() and "/acik/" not in "/" + p.relative_to(VERI).as_posix())
    hs = set(ham)
    eksik = sorted(h for h in beklenen if h not in hs)
    fazla = sorted(h for h in ham if h not in beklenen)
    test("D1_eksik_dosya_yok", set(eksik) <= IZINLI_EKSIK,
         f"{len(beklenen) - len(eksik)}/{len(beklenen)} var; eksik {eksik} (izinli: {sorted(IZINLI_EKSIK)})")
    test("D1_fazla_dosya_yok", all(h.startswith(IZINLI_FAZLA_ON) for h in fazla),
         f"fazla {len(fazla)} dosya: {fazla[:8]}")

    # ------------------------------------------------ dosya başına: 0 bayt, HTML, boyut, SHA-256
    satirlar, n_fail = [], 0
    print(f"SHA-256 yeniden hesaplanıyor: {len(ham)} dosya (birkaç dakika)...", flush=True)
    for i, h in enumerate(ham, 1):
        p = VERI / h
        b = p.stat().st_size
        with open(p, "rb") as f:
            bas = f.read(512)
        s = sha256(p)
        r = beklenen.get(h, {})
        bek = int(r["beklenen_bayt"]) if r.get("beklenen_bayt") else None
        kay = son.get(h, {}).get("sha256", "")
        html = bas.lstrip()[:15].lower().startswith((b"<!doctype", b"<html"))
        notlar = []
        if b == 0:
            notlar.append("0 bayt")
        if html:
            notlar.append("HTML")
        if bek is not None and b != bek:
            notlar.append(f"boyut {b} != rapor {bek}")
        if h in beklenen and kay != s:
            notlar.append("SHA-256 indirme kaydindan farkli" if kay else "indirme kaydi yok")
        durum = "FAIL" if notlar else "PASS"
        n_fail += durum == "FAIL"
        satirlar.append({"hedef": h, "set": h.split("/")[1].split("_")[0], "bayt": b, "rapor_bayt": bek or "",
                         "sha256": s, "durum": durum, "not": "; ".join(notlar)})
        if i % 50 == 0:
            print(f"   {i}/{len(ham)}", flush=True)
    test("D1_dosya_testleri", n_fail == 0,
         f"{len(satirlar) - n_fail}/{len(satirlar)} dosya: 0 bayt yok, HTML yok, boyut = rapor, SHA-256 = indirme kaydi"
         + ("" if n_fail == 0 else f"; FAIL: {[x['hedef'] for x in satirlar if x['durum'] == 'FAIL'][:8]}"))

    # ------------------------------------------------ KONTROL dosyalarının içerik testi
    def metin(h):
        return (VERI / h).read_text(encoding="utf-8", errors="replace")
    for oz in ("clay", "silt"):
        h = f"dunya/E10_soilgrids/{oz}/checksum.sha256.txt"
        deger = beklenen[f"dunya/E10_soilgrids/{oz}/{oz}_0-5cm_mean_1000.tif"]["saglama_degeri"]
        dene(f"KONTROL_{h}", lambda h=h, d=deger: (d.lower() in metin(h).lower(), "rapordaki SHA-256 dosyada geciyor"))
    for h in ("ay/M1_lola/Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif.md5",
              "mars/R1_mola/Mars_MGS_MOLA_DEM_mosaic_global_463m.tif.md5"):
        deger = beklenen[h[:-4]]["saglama_degeri"]
        dene(f"KONTROL_{h}", lambda h=h, d=deger: (d.lower() in metin(h).lower(), "rapordaki MD5 dosyada geciyor"))
    h = "ay/M1_lola/Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.lbl"
    dene(f"KONTROL_{h}", lambda: (all(k in metin(h) for k in ("92160", "46080", "SignedWord"))
                                  and re.search(r"Multiplier\s*=\s*0\.5\b", metin(h)) is not None,
                                  "rapor M1: 92160 x 46080, SignedWord, Multiplier 0.5"))
    h2 = "mars/R1_mola/Mars_MGS_MOLA_DEM_mosaic_global_463m.lbl"
    dene(f"KONTROL_{h2}", lambda: ("46080" in metin(h2) and "23040" in metin(h2), "rapor R1: 46080 x 23040"))
    h3 = "mars/R2_sim3292/sim3292_readme.txt"
    dene(f"KONTROL_{h3}", lambda: ("3292" in metin(h3), f"metin dosyasi, '3292' geciyor ({(VERI / h3).stat().st_size} bayt)"))

    # ------------------------------------------------ zip açma (belge 17.2)
    acilan = {}
    for z in ZIPLER:
        def zip_ac(z=z):
            zp = VERI / z
            hedef = zp.parent / "acik"
            hedef.mkdir(exist_ok=True)
            with zipfile.ZipFile(zp) as zf:
                bilgi = [i for i in zf.infolist() if not i.is_dir()]
                yontem = sorted({i.compress_type for i in bilgi})
                if 9 in yontem:
                    raise RuntimeError("Deflate64 (9) var: inflate64 yolu gerekli (belge 17.2)")
                atla = [i for i in bilgi if i.filename.startswith("__MACOSX/") or Path(i.filename).name.startswith("._")]
                al = [i for i in bilgi if i not in atla]
                sifir = [i.filename for i in al if i.file_size == 0]
                n_yeni = 0
                for i in al:
                    out = hedef / i.filename
                    if out.exists() and out.stat().st_size == i.file_size:
                        continue
                    if out.exists():
                        os.chmod(out, stat.S_IWRITE | stat.S_IREAD)
                    zf.extract(i, hedef)                     # zipfile CRC'yi okurken denetler (hata -> BadZipFile)
                    n_yeni += 1
                    if out.stat().st_size != i.file_size or (i.file_size > 0 and out.stat().st_size == 0):
                        raise RuntimeError(f"acilan boyut tutmuyor: {i.filename}")
                    os.chmod(out, stat.S_IREAD)
                topl = sum(i.file_size for i in al)
            acilan[z] = hedef
            return True, (f"{len(al)} dosya ({n_yeni} yeni acildi), {topl / 1e9:.3f} GB, yontemler {yontem}, "
                          f"atlanan {len(atla)} (__MACOSX/._), zip dizininde 0 bayt girdi {len(sifir)}")
        dene(f"ZIP_{z}", zip_ac)

    # ------------------------------------------------ açılış testleri
    def ac(p):
        with rasterio.open(p) as d:
            return d.width, d.height, d.count, d.dtypes[0], str(d.crs)[:40] if d.crs else "CRS yok"
    def rast(ad, p):
        dene(ad, lambda: (True, ac(p)))

    # E1: 288 karonun hepsi açılıyor mu
    def e1():
        hata, tip = [], set()
        for k in karolar:
            try:
                w, h_, c, dt, _ = ac(VERI / "dunya/E1_etopo" / k["ad"])
                tip.add((w, h_, c, dt))
            except Exception as e:
                hata.append(k["ad"])
        return not hata, f"{len(karolar) - len(hata)}/{len(karolar)} acildi; (genislik, yukseklik, bant, tip) = {sorted(tip)}; acilmayan {hata[:5]}"
    dene("ACILIS_E1_288_karo", e1)

    # HDF4 (D0'da ertelenen test) ve HDF5
    def hdf4(ad, yol, aranan):
        def f():
            alt = [s[0] for s in gdal.Open(str(VERI / yol)).GetSubDatasets()]
            eks = [a for a in aranan if not any(a in s for s in alt)]
            return not eks, f"{len(alt)} alt veri seti; aranan {aranan}; eksik {eks}"
        dene(ad, f)
    hdf4("ACILIS_HDF4_E2", "dunya/E2_mcd12c1/MCD12C1.A2023001.061.2024251212901.hdf", ["Land_Cover_Type_1_Percent"])
    for r in [x for x in liste if x["set"] == "E5"]:
        hdf4(f"ACILIS_HDF4_{r['hedef'].rsplit('/', 1)[-1]}", r["hedef"], ["LST_Day_CMG", "LST_Night_CMG"])
    for r in [x for x in liste if x["set"] == "E6"]:
        hdf4(f"ACILIS_HDF4_{r['hedef'].rsplit('/', 1)[-1]}", r["hedef"], ["Albedo_WSA_shortwave"])
    e4_yapi = {}
    for r in [x for x in liste if x["set"] == "E4"]:
        def h5(r=r):
            adlar = []
            with h5py.File(VERI / r["hedef"], "r") as f:
                f.visititems(lambda n, o: adlar.append(n) if isinstance(o, h5py.Dataset) else None)
            e4_yapi[r["hedef"]] = adlar
            ok = any("Emissivity" in a for a in adlar) and any("NDVI" in a for a in adlar)
            return ok, f"{len(adlar)} veri seti: {adlar[:8]}"
        dene(f"ACILIS_HDF5_{r['hedef'].rsplit('/', 1)[-1]}", h5)

    # GeoTIFF'ler
    for h in ["ay/M1_lola/Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif",
              "ay/M3_diviner_ghrm/dghrm_ra_sam_70s70n_tif.tif", "ay/M3_diviner_ghrm/dghrm_tbol_sam_70s70n_tif.tif",
              "ay/M7_pgda/AVGVISIB_65S_240M_201608.TIF", "ay/M7_pgda/AVGVISIB_65N_240M_201608.TIF",
              "ay/M7_pgda/LPSR_65S_240M_201608.TIF", "ay/M7_pgda/LPSR_65N_240M_201608.TIF",
              "mars/R1_mola/Mars_MGS_MOLA_DEM_mosaic_global_463m.tif",
              "mars/R7_swim/SWIM4MIM_Ci_0_1.tif", "mars/R7_swim/SWIM4MIM_Ci_1_5.tif", "mars/R7_swim/SWIM4MIM_Ci_5.tif",
              "dunya/E10_soilgrids/clay/clay_0-5cm_mean_1000.tif", "dunya/E10_soilgrids/silt/silt_0-5cm_mean_1000.tif"]:
        rast(f"ACILIS_{h}", VERI / h)

    # zip içinden gelen rasterler
    def bul(z, desen):
        k = acilan.get(z)
        return sorted(k.rglob(desen)) if k else []
    for z, desen in [(ZIPLER[1], "UiO_PEX_PERPROB_5.0_20181128_2000_2016_NH.tif"),
                     (ZIPLER[2], "UiO_PEX_PERPROB_5.0_20181205_2000_2017_ANT.tif"),
                     (ZIPLER[3], "ai_v31_yr.tif")]:
        def f(z=z, desen=desen):
            b = [p for p in bul(z, desen) if "__MACOSX" not in p.parts]
            if len(b) != 1:
                return False, f"{desen}: {len(b)} adet bulundu"
            return True, (b[0].relative_to(VERI).as_posix(), ac(b[0]))
        dene(f"ACILIS_{desen}", f)

    # TES mineral ENVI (rapor R3: 1440 x 720 x 13, float32)
    def envi():
        d = gdal.Open(str(VERI / "mars/R3_tes_mineral/TES_mineral_maps_4_ppd.envi"))
        dt = gdal.GetDataTypeName(d.GetRasterBand(1).DataType)
        return (d.RasterXSize, d.RasterYSize, d.RasterCount, dt) == (1440, 720, 13, "Float32"), \
            f"{d.RasterXSize} x {d.RasterYSize} x {d.RasterCount}, {dt} (rapor: 1440 x 720 x 13 Float32)"
    dene("ACILIS_R3_ENVI", envi)

    # vektörler
    def glim():
        g = [p for p in bul(ZIPLER[0], "*.gdb") if p.is_dir()]
        if len(g) != 1:
            return False, f"{len(g)} .gdb klasoru"
        katman = [l[0] for l in pyogrio.list_layers(g[0])]
        bilgi = pyogrio.read_info(g[0], layer="GLiM_export")
        return "GLiM_export" in katman and "xx" in bilgi["fields"], \
            f"katmanlar {katman}; GLiM_export {bilgi['features']} kayit, sutun xx var: {'xx' in bilgi['fields']}, CRS {str(bilgi['crs'])[:60]}"
    dene("ACILIS_E3_GLiM", glim)

    def vek(z, desen, sutunlar, adet, essiz=None):
        def f():
            b = bul(z, desen)
            if len(b) != 1:
                return False, f"{desen}: {len(b)} adet"
            bilgi = pyogrio.read_info(b[0])
            ok = all(s in bilgi["fields"] for s in sutunlar) and bilgi["features"] == adet
            d = f"{b[0].relative_to(VERI).as_posix()}: {bilgi['features']} kayit (rapor {adet}), sutunlar {sutunlar} var: {all(s in bilgi['fields'] for s in sutunlar)}"
            if essiz:
                df = pyogrio.read_dataframe(b[0], columns=[essiz[0]], read_geometry=False)
                n = df[essiz[0]].nunique()
                ok = ok and n == essiz[1]
                d += f", essiz {essiz[0]} {n} (rapor {essiz[1]})"
            return ok, d
        return f
    dene("ACILIS_M2_GeoUnits", vek(ZIPLER[4], "GeoUnits.shp", ["FIRST_Unit", "AREA_GEO"], 12247))
    dene("ACILIS_R2_SIM3292", vek(ZIPLER[5], "SIM3292_Global_Geology.shp", ["Unit", "SphArea_km"], 1311, ("Unit", 44)))

    # ------------------------------------------------ raporun ek bütünlük testleri
    def m5():
        hb = (VERI / "ay/M5_lp_grs/lpgrs_high1_elem_abundance_2deg.tab").read_bytes()
        return len(hb) == 11306 * 880, f"{len(hb)} bayt = 11306 x 880 ({11306 * 880})"
    dene("EK_M5_11306x880", m5)

    def m6():
        n, bozuk, kotu = 0, 0, 0
        for s in (VERI / "ay/M6_lp_hidrojen/hydrogenhd.txt").read_text(encoding="ascii", errors="replace").splitlines():
            if not s.strip() or s.lstrip().startswith("#"):
                continue
            n += 1
            p = s.split(",")
            if len(p) != 5:
                kotu += 1
            if "*" in s:
                bozuk += 1
        return n == 259200 and kotu == 0 and bozuk == 1, \
            f"{n} veri satiri (rapor 259200), 5 sutun olmayan {kotu}, '*' iceren (bozuk hucre) {bozuk} (rapor 1)"
    dene("EK_M6_259200x5", m6)

    # ------------------------------------------------ kayıt
    with open(KAYIT / "D1_dosyalar.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["hedef", "set", "bayt", "rapor_bayt", "sha256", "durum", "not"])
        w.writeheader()
        w.writerows(satirlar)
    gecmeyen = [t["test"] for t in TESTLER if t["durum"] == "FAIL"]
    ozet = {"adim": "D1", "tarih": datetime.now().isoformat(timespec="seconds"),
            "sure_dk": round((time.time() - t0) / 60, 1),
            "ham_dosya": len(ham), "ham_bayt": sum(x["bayt"] for x in satirlar),
            "acik_bayt": sum(p.stat().st_size for k in acilan.values() for p in k.rglob("*") if p.is_file()),
            "eksik": eksik, "fazla": fazla, "e4_hdf5_yapisi_ornek": next(iter(e4_yapi.values()), []),
            "test_sayisi": len(TESTLER), "PASS": len(TESTLER) - len(gecmeyen), "FAIL": len(gecmeyen),
            "gecmeyenler": gecmeyen, "testler": TESTLER}
    (KAYIT / "D1.json").write_text(json.dumps(ozet, ensure_ascii=False, indent=2), encoding="utf-8")
    with open(KAYIT / "13_kasim_log.txt", "a", encoding="utf-8") as f:
        f.write(f"{datetime.now():%Y-%m-%d %H:%M:%S}  D1 butunluk: {ozet['PASS']} PASS, {ozet['FAIL']} FAIL {gecmeyen}, "
                f"{ozet['sure_dk']} dk\n")
    print("=" * 60)
    print(f"D1: {ozet['PASS']}/{ozet['test_sayisi']} PASS | ham {ozet['ham_dosya']} dosya {ozet['ham_bayt'] / 1e9:.3f} GB | "
          f"acilan {ozet['acik_bayt'] / 1e9:.3f} GB | toplam {(ozet['ham_bayt'] + ozet['acik_bayt']) / 1e9:.3f} GB | {ozet['sure_dk']} dk")
    for g in gecmeyen:
        print("   FAIL:", g)
    print("Kayit: kayit/D1.json, kayit/D1_dosyalar.csv")
    return 1 if gecmeyen else 0


if __name__ == "__main__":
    sys.exit(main())
'''
compile(kod, "d1_butunluk.py", "exec")
dosya = KOK / "kod" / "d1_butunluk.py"
dosya.write_text(kod, encoding="utf-8")
print("Yazıldı:", dosya, "\n")
g = runpy.run_path(str(dosya), run_name="d1")      # sys.exit çağrılmasın diye __main__ değil
cikis = g["main"]()
print("\nÇıkış kodu:", cikis, "(0 = hepsi PASS)")

Yazıldı: C:\Users\USER\orbitrary\kod\d1_butunluk.py 

[PASS] D1_eksik_dosya_yok: 408/409 var; eksik ['mars/R4_tes_ti_albedo/nmsk2003.bin'] (izinli: ['mars/R4_tes_ti_albedo/nmsk2003.bin'])
[PASS] D1_fazla_dosya_yok: fazla 6 dosya: ['mars/R6_odyssey_grs/pds4_etiket/cl_sr_5x5.xml', 'mars/R6_odyssey_grs/pds4_etiket/fe_sr_5x5.xml', 'mars/R6_odyssey_grs/pds4_etiket/h2o_sr_5x5.xml', 'mars/R6_odyssey_grs/pds4_etiket/k_sr_5x5.xml', 'mars/R6
SHA-256 yeniden hesaplanıyor: 414 dosya (birkaç dakika)...
   50/414
   100/414
   150/414
   200/414
   250/414
   300/414
   350/414
   400/414
[PASS] D1_dosya_testleri: 414/414 dosya: 0 bayt yok, HTML yok, boyut = rapor, SHA-256 = indirme kaydi
[PASS] KONTROL_dunya/E10_soilgrids/clay/checksum.sha256.txt: rapordaki SHA-256 dosyada geciyor
[PASS] KONTROL_dunya/E10_soilgrids/silt/checksum.sha256.txt: rapordaki SHA-256 dosyada geciyor
[PASS] KONTROL_ay/M1_lola/Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif.md5: rapordaki MD5 dosyada geciyor
[PASS] KONTROL_mars/R

In [14]:
# Developed with assistance from Claude (Anthropic).
# Adım D2 (Dünya) — kod\d2_dunya.py yazılır ve çalıştırılır (belge 17.3 D2, E1–E10).
from pathlib import Path
import runpy

KOK = Path.home() / "orbitrary"
kod = r'''# Developed with assistance from Claude (Anthropic).
# d2_dunya.py — Orbitrary v1.2, belge 17.3 D2 (gerçek-yer testleri), Dünya setleri E1–E10.
# Her test: nokta, koordinat ve kaynağı, beklenen değer ve kaynağı, tolerans kuralı -> kayit/D2.csv
# Okuma ayrıntıları (ölçek, ofset, veri-yok, boyut) dosyadan okunur ve kayit/D2_okuma_dunya.json'a yazılır.
# Geçmeyen test silinmez; FAIL yazılır. Talimat 7 (ASTER): en küçük yayınım 0,52–0,61 -> UYARI (iş sürer),
# <= 0,51 -> FAIL (ofset uygulanmamış demektir).
import csv, json, math, time
from datetime import datetime
from pathlib import Path

import numpy as np

KOK = Path.home() / "orbitrary"
VERI, KAYIT = KOK / "veri", KOK / "kayit"
D2CSV = KAYIT / "D2.csv"
ALANLAR = ["govde", "set", "test", "nokta", "enlem", "boylam", "koordinat_kaynagi", "beklenen",
           "tolerans_kurali", "beklenen_kaynagi", "olculen", "durum", "zaman"]

# Test noktaları (koordinat + kaynak). Kaynaklar 14 Kasım 2026'da açıldı.
N = {
    "challenger": (11.3733, 142.5917, "GEBCO Gazetteer (Wikipedia 'Challenger Deep' infobox): 11°22.4'N 142°35.5'E"),
    "olu_deniz": (31.5, 35.5, "Wikipedia 'Dead Sea' infobox: 31°30'N 35°30'E"),
    "sahra": (23.0, 13.0, "Wikipedia 'Sahara' infobox: 23°N 13°E"),
    "summit": (72.5795833, -38.4591861, "Wikipedia 'Summit Camp' (UNAVCO GPS, Temmuz 2009): 72.5795833°N 38.4591861°W"),
    "nemo": (-48.876667, -123.393333, "NOAA National Ocean Service (Point Nemo): 48°52.6'S 123°23.6'W"),
    "rub_al_khali": (20.0, 50.0, "Wikipedia 'Rub' al Khali' infobox: 20°N 50°E"),
    "yakutsk": (62.03, 129.73, "Wikipedia 'Yakutsk' infobox: 62.03°N 129.73°E"),
    "kahire": (30.04444, 31.23583, "Wikipedia 'Cairo' infobox: 30.04444°N 31.23583°E"),
    "kuru_vadiler": (-77.467, 162.517, "Wikipedia 'McMurdo Dry Valleys': 77.467°S 162.517°E"),
    "atacama_yungay": (-24.0656, -69.8639, "indirme_raporu3 B4 R-01 (K2, wondermondo): -24.0656, -69.8639"),
    "manaus": (-3.11889, -60.02167, "Wikipedia 'Manaus' infobox: 3.11889°S 60.02167°W"),
}

SATIRLAR, OKUMA = [], {}


def kaydet(set_, test, nokta, beklenen, kural, bek_kaynak, olculen, durum):
    e, b, k = (N[nokta] if nokta in N else ("", "", ""))
    SATIRLAR.append({"govde": "dunya", "set": set_, "test": test, "nokta": nokta, "enlem": e, "boylam": b,
                     "koordinat_kaynagi": k, "beklenen": beklenen, "tolerans_kurali": kural,
                     "beklenen_kaynagi": bek_kaynak, "olculen": olculen, "durum": durum,
                     "zaman": datetime.now().isoformat(timespec="seconds")})
    print(f"[{durum}] {set_} {test} {nokta}: ölçülen {olculen} | beklenen {beklenen} ({kural})", flush=True)


def dene(set_, test, nokta, beklenen, kural, bek_kaynak, f):
    try:
        olculen, ok = f()
        kaydet(set_, test, nokta, beklenen, kural, bek_kaynak, olculen, ok if isinstance(ok, str) else ("PASS" if ok else "FAIL"))
    except Exception as e:
        kaydet(set_, test, nokta, beklenen, kural, bek_kaynak, f"{type(e).__name__}: {e}", "FAIL")


def cmg_rc(enlem, boylam, adim=0.05):
    # 0,05° küresel CMG ızgarası: sol üst köşe (-180, 90); satır güneye, sütun doğuya artar
    return int(math.floor((90.0 - enlem) / adim)), int(math.floor((boylam + 180.0) / adim))


def main():
    t0 = time.time()
    from osgeo import gdal
    gdal.UseExceptions()
    import rasterio, h5py
    from pyproj import Transformer

    # ------------------------------------------------ E1 ETOPO 2022 (yükseklik, m)
    def etopo(enlem, boylam, pencere=0):
        kuzey = (math.floor(enlem / 15.0) + 1) * 15            # karo adı = üst-sol köşe
        bati = math.floor(boylam / 15.0) * 15
        ad = f"ETOPO_2022_v1_15s_{'N' if kuzey >= 0 else 'S'}{abs(kuzey):02d}{'E' if bati >= 0 else 'W'}{abs(bati):03d}_surface.tif"
        with rasterio.open(VERI / "dunya/E1_etopo" / ad) as d:
            tr = d.transform
            assert abs(tr.c - bati) < 1e-6 and abs(tr.f - kuzey) < 1e-6, f"{ad}: transform köşesi {tr.c},{tr.f} karo adıyla tutmuyor"
            assert abs(tr.a - 15 / 3600) < 1e-12 and abs(tr.e + 15 / 3600) < 1e-12, f"{ad}: piksel {tr.a},{tr.e} != 15 yay-saniye"
            assert d.dtypes[0] == "float32", d.dtypes[0]
            OKUMA.setdefault("E1", {"nodata": d.nodata, "dtype": d.dtypes[0], "piksel_derece": tr.a})
            assert d.nodata == -99999, f"nodata {d.nodata} (belge 10.1: -99999)"
            r, c = d.index(boylam, enlem)
            w = rasterio.windows.Window(max(c - pencere, 0), max(r - pencere, 0), 2 * pencere + 1, 2 * pencere + 1)
            a = d.read(1, window=w).astype(float)
            a[a == d.nodata] = np.nan
            return ad, a
    def t_mariana():
        ad, a = etopo(*N["challenger"][:2], pencere=20)       # ±20 piksel (~±9 km): piksel konumu belirsizliği
        v = float(np.nanmin(a))
        return f"{v:.1f} m (±20 piksel penceresinin en düşüğü, {ad})", -11000 <= v <= -10000
    dene("E1", "Mariana derinligi", "challenger", "yaklasik -10550 m", "[-11000, -10000] m (proje karari)",
         "belge 17.3 D2; piksel ortalaması nedeniyle tam uç değil", t_mariana)
    def t_olu():
        ad, a = etopo(*N["olu_deniz"][:2], pencere=0)
        v = float(a[0, 0])
        return f"{v:.1f} m ({ad})", -450 <= v <= -400
    dene("E1", "Olu Deniz yuzeyi", "olu_deniz", "yaklasik -430 m", "[-450, -400] m (proje karari)",
         "belge 17.3 D2; Wikipedia 'Dead Sea' yüzey -430,5 m (2016)", t_olu)

    # ------------------------------------------------ E2 MCD12C1 Land_Cover_Type_1_Percent
    def alt(yol, ad):
        s = [x[0] for x in gdal.Open(str(yol)).GetSubDatasets() if x[0].endswith(":" + ad) or x[0].endswith(ad)]
        assert len(s) >= 1, f"{ad} alt veri seti yok"
        return gdal.Open(s[0])
    lc = alt(VERI / "dunya/E2_mcd12c1/MCD12C1.A2023001.061.2024251212901.hdf", "Land_Cover_Type_1_Percent")
    md = lc.GetMetadata()
    OKUMA["E2"] = {"boyut": [lc.RasterXSize, lc.RasterYSize, lc.RasterCount], "geotransform": lc.GetGeoTransform(),
                   "metadata": {k: md[k] for k in md if k in ("_FillValue", "valid_range", "units", "scale_factor")}}
    assert (lc.RasterXSize, lc.RasterYSize, lc.RasterCount) == (7200, 3600, 17), f"E2 boyut {OKUMA['E2']['boyut']}"
    def sinif(nokta):
        r, c = cmg_rc(*N[nokta][:2])
        return [int(lc.GetRasterBand(k + 1).ReadAsArray(c, r, 1, 1)[0, 0]) for k in range(17)]   # bant k+1 = sınıf k
    for nokta, k, ad in [("sahra", 16, "ciplak"), ("summit", 15, "kar/buz"), ("nemo", 0, "su")]:
        dene("E2", f"sinif {k} ({ad}) payi", nokta, "%100", ">= %95 ve 17 sinif toplami 100 (proje karari)",
             "belge 17.3 D2 (ürün tanımı)",
             lambda nokta=nokta, k=k: ((lambda v: (f"sinif{k}=%{v[k]}, toplam {sum(v)}", v[k] >= 95 and sum(v) == 100))(sinif(nokta))))

    # ------------------------------------------------ E4 ASTER GED (yayınım + NDVI), talimat 7
    e4 = sorted((VERI / "dunya/E4_aster_ged").glob("ASTER_GEDv4.1_A2015*.h5"))
    assert len(e4) == 12
    with h5py.File(e4[0], "r") as f:
        em, nd = f["SDS/Emissivity"], f["SDS/NDVI"]
        em_att = {k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in em.attrs.items()}
        nd_att = {k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in nd.attrs.items()}
        em_sekil, nd_sekil = em.shape, nd.shape
    def att(d, ad):
        for k, v in d.items():
            if k.strip().lower() == ad.lower():
                return float(np.ravel(v)[0]) if not isinstance(v, (bytes, str)) else float(v)
        return None
    olcek, ofset, ndvi_olcek = att(em_att, "Scale Factor"), att(em_att, "Offset"), att(nd_att, "Scale Factor")
    ofset_kaynak = "dosya özniteliği" if ofset is not None else "ofset dosyada yok, kaynak belge 10.1 / Hulley 2015"
    if ofset is None:
        ofset = 0.49
    OKUMA["E4"] = {"emissivity_sekil": em_sekil, "ndvi_sekil": nd_sekil, "emissivity_oznitelik": em_att,
                   "ndvi_oznitelik": nd_att, "scale": olcek, "offset": ofset, "offset_kaynagi": ofset_kaynak,
                   "ndvi_scale": ndvi_olcek}
    bant_ekseni = [i for i, s in enumerate(em_sekil) if s == 5]
    dene("E4", "olcek/ofset dosyadan", "", "Scale 0,002; Offset 0,49; NDVI Scale 0,001",
         "esitlik (talimat 7: Offset dosyada yoksa belge 10.1)", "rapor E4; belge 10.1",
         lambda: (f"Scale {olcek}, Offset {ofset} ({ofset_kaynak}), NDVI Scale {ndvi_olcek}, sekil {em_sekil}",
                  olcek == 0.002 and abs(ofset - 0.49) < 1e-9 and ndvi_olcek == 0.001 and len(bant_ekseni) == 1))
    def em_oku(f):
        a = f["SDS/Emissivity"][...]
        return np.moveaxis(a, bant_ekseni[0], 0)                # (5, satır, sütun); 0 = bant 10 ... 4 = bant 14
    def t_aralik():
        mn, mx = 255, 0
        for p in e4:
            with h5py.File(p, "r") as f:
                a = em_oku(f)
            v = a[a > 0]                                       # 0 = yok (rapor E4)
            mn, mx = min(mn, int(v.min())), max(mx, int(v.max()))
        emn, emx = mn * olcek + ofset, mx * olcek + ofset
        if emn >= 0.61 - 1e-9 and emx <= 1.0 + 1e-9:
            d = "PASS"
        elif emn > 0.51 + 1e-9:
            d = "UYARI"
        else:
            d = "FAIL"
        return f"en kucuk {emn:.3f} (DN {mn}), en buyuk {emx:.3f} (DN {mx}), 12 ay", d
    dene("E4", "gecerli yayinim araligi", "", "0,61-1,00",
         "min >= 0,61 PASS; 0,52-0,61 UYARI (kayda yazilir, is surer); <= 0,51 FAIL (talimat 7)",
         "v1.0 indirmesinde ölçülen en küçük değer 0,612; Hulley ve Hook 2009 (HyspIRI Workshop) kum tepesi "
         "sahalarındaki 0,6–0,96 aralığıyla tutarlı", t_aralik)
    def t_kuvars():
        r, c = cmg_rc(*N["rub_al_khali"][:2])
        top, say = np.zeros(5), np.zeros(5)
        for p in e4:
            with h5py.File(p, "r") as f:
                a = em_oku(f)[:, r, c].astype(float)
            ok = a > 0
            top[ok] += a[ok] * olcek + ofset
            say[ok] += 1
        m = top / np.where(say > 0, say, np.nan)
        return (f"yillik ort. bant10..14 = {np.round(m, 3).tolist()} (gecerli ay {say.astype(int).tolist()})",
                bool(np.nanmax(m[:3]) < np.nanmin(m[3:])))
    dene("E4", "kuvars cukuru", "rub_al_khali", "bant 10-12 < bant 13-14",
         "bantlar 10-12'nin en buyugu < bantlar 13-14'un en kucugu (yillik ortalama)",
         "belge 17.3 D2; Hulley ve ark. 2015", t_kuvars)

    # ------------------------------------------------ E5 MYD11C3 (LST)
    e5 = sorted((VERI / "dunya/E5_myd11c3").glob("MYD11C3.A2023*.hdf"))
    gun = alt(e5[0], "LST_Day_CMG")
    m5 = gun.GetMetadata()
    OKUMA["E5"] = {"boyut": [gun.RasterXSize, gun.RasterYSize], "metadata": {k: m5[k] for k in m5 if k in ("scale_factor", "_FillValue", "valid_range", "units")}}
    dene("E5", "olcek/veri-yok dosyadan", "", "scale 0,02; fill 0", "esitlik", "rapor E5",
         lambda: (str(OKUMA["E5"]), float(m5.get("scale_factor", "nan")) == 0.02 and int(float(m5.get("_FillValue", "-1"))) == 0
                  and (gun.RasterXSize, gun.RasterYSize) == (7200, 3600)))
    def t_okyanus():
        r, c = cmg_rc(*N["nemo"][:2])
        v = []
        for p in e5:
            for ad in ("LST_Day_CMG", "LST_Night_CMG"):
                v.append(int(alt(p, ad).GetRasterBand(1).ReadAsArray(c, r, 1, 1)[0, 0]))
        return f"24 deger (12 ay x gece/gunduz): {sorted(set(v))}", all(x == 0 for x in v)
    dene("E5", "acik okyanus veri yok", "nemo", "veri yok (fill 0)", "24 degerin hepsi 0", "belge 17.3 D2 (kara ürünü)", t_okyanus)

    # ------------------------------------------------ E6 MCD43C3 (albedo)
    e6 = sorted((VERI / "dunya/E6_mcd43c3").glob("MCD43C3.A2023*.hdf"))
    alb = alt(e6[0], "Albedo_WSA_shortwave")
    m6 = alb.GetMetadata()
    OKUMA["E6"] = {"boyut": [alb.RasterXSize, alb.RasterYSize], "metadata": {k: m6[k] for k in m6 if k in ("scale_factor", "_FillValue", "valid_range", "units")}}
    dene("E6", "olcek/veri-yok dosyadan", "", "scale 0,001; fill 32767", "esitlik", "rapor E6",
         lambda: (str(OKUMA["E6"]), float(m6.get("scale_factor", "nan")) == 0.001 and int(float(m6.get("_FillValue", "-1"))) == 32767
                  and (alb.RasterXSize, alb.RasterYSize) == (7200, 3600)))
    def alb_ort(nokta):
        r, c = cmg_rc(*N[nokta][:2])
        v = [int(alt(p, "Albedo_WSA_shortwave").GetRasterBand(1).ReadAsArray(c, r, 1, 1)[0, 0]) for p in e6]
        g = [x * 0.001 for x in v if 0 <= x <= 32766]
        return (float(np.mean(g)) if g else float("nan")), len(g)
    def t_albedo():
        gs, ng = alb_ort("summit")
        sh, ns = alb_ort("sahra")
        mx = 0
        for p in e6:
            a = alt(p, "Albedo_WSA_shortwave").GetRasterBand(1).ReadAsArray()
            v = a[(a >= 0) & (a <= 32766)]
            mx = max(mx, int(v.max()))
        return (f"Gronland {gs:.3f} ({ng} ay), Sahra {sh:.3f} ({ns} ay), kuresel en buyuk {mx * 0.001:.3f}",
                gs > 0.6 and 0.2 <= sh <= 0.5 and gs > sh and mx * 0.001 <= 1.0)
    dene("E6", "Gronland yuksek / Sahra orta; 0-1", "summit", "yuksek / orta; 0-1 araligi",
         "Gronland > 0,6; Sahra 0,2-0,5; kuresel en buyuk <= 1,0 (proje karari)", "belge 17.3 D2 (ürün tanımı)", t_albedo)

    # ------------------------------------------------ E7, E8 PERPROB
    def perprob(yol, nokta, pencere):
        with rasterio.open(yol) as d:
            tr = Transformer.from_crs("EPSG:4326", d.crs, always_xy=True)
            x, y = tr.transform(N[nokta][1], N[nokta][0])
            r, c = d.index(x, y)
            w = rasterio.windows.Window(c - pencere, r - pencere, 2 * pencere + 1, 2 * pencere + 1)
            a = d.read(1, window=w, boundless=True, fill_value=np.nan).astype(float)
            if d.nodata is not None:
                a[a == d.nodata] = np.nan
            OKUMA.setdefault(yol.parent.parent.name, {"crs": str(d.crs), "nodata": d.nodata, "piksel": d.res, "dtype": d.dtypes[0]})
            return a
    nh = VERI / "dunya/E7_permafrost_kuzey/acik/UiO_PEX_PERPROB_5.0_20181128_2000_2016_NH.tif"
    an = VERI / "dunya/E8_permafrost_antarktika/acik/UiO_PEX_PERPROB_5.0_20181205_2000_2017_ANT.tif"
    dene("E7", "Yakutsk", "yakutsk", "yaklasik 1", ">= 0,9 (proje karari)", "Obu ve ark. 2019",
         lambda: ((lambda a: (f"{a[0, 0]}", a[0, 0] >= 0.9))(perprob(nh, "yakutsk", 0))))
    dene("E7", "Kahire", "kahire", "yaklasik 0", "<= 0,1 ya da veri yok (sifir kurali, belge 6.6)", "Obu ve ark. 2019",
         lambda: ((lambda a: (f"{a[0, 0]}", bool(np.isnan(a[0, 0]) or a[0, 0] <= 0.1)))(perprob(nh, "kahire", 0))))
    dene("E8", "Kuru Vadiler", "kuru_vadiler", "yuksek", "+-5 piksel (~+-5 km) penceresinde gecerli degerlerin en buyugu >= 0,9 (proje karari)",
         "Obu ve ark. 2020",
         lambda: ((lambda a: (f"merkez {a[5, 5]}, pencere en buyuk {np.nanmax(a) if np.isfinite(a).any() else 'yok'}",
                              bool(np.isfinite(a).any() and np.nanmax(a) >= 0.9)))(perprob(an, "kuru_vadiler", 5))))

    # ------------------------------------------------ E9 Aridity (× 10 000, belge D9)
    ai_yol = VERI / "dunya/E9_aridity/acik/Global-AI_ET0__annual_v3_1/ai_v31_yr.tif"
    with rasterio.open(ai_yol) as d:
        bant = gdal.Open(str(ai_yol)).GetRasterBand(1)
        OKUMA["E9"] = {"dtype": d.dtypes[0], "nodata": d.nodata, "crs": str(d.crs), "piksel": d.res,
                       "gdal_scale": bant.GetScale(), "gdal_offset": bant.GetOffset()}
        def ai(nokta):
            r, c = d.index(N[nokta][1], N[nokta][0])
            return int(d.read(1, window=rasterio.windows.Window(c, r, 1, 1))[0, 0])
        at, am = ai("atacama_yungay"), ai("manaus")
    dene("E9", "carpan x 10000 (K9)", "", "AI = ham / 10000",
         "dosyada baska olcek yok ve Atacama ham < 300, Amazon ham > 6500 (sinif sinirlari x 10000 ile tutarli)",
         "rapor E9 (figshare açıklaması); belge D9, K9",
         lambda: (f"GDAL scale {OKUMA['E9']['gdal_scale']}, offset {OKUMA['E9']['gdal_offset']}, nodata {OKUMA['E9']['nodata']}; "
                  f"Atacama ham {at}, Manaus ham {am}",
                  (OKUMA["E9"]["gdal_scale"] in (None, 1.0)) and at < 300 and am > 6500))
    dene("E9", "Atacama", "atacama_yungay", "< 0,03", "ham/10000 < 0,03", "Zomer ve ark. 2022 sınıfları",
         lambda: (f"{at / 10000:.4f}", at / 10000 < 0.03))
    dene("E9", "Amazon", "manaus", "> 0,65", "ham/10000 > 0,65", "Zomer ve ark. 2022 sınıfları",
         lambda: (f"{am / 10000:.4f}", am / 10000 > 0.65))

    # ------------------------------------------------ E10 SoilGrids silt + kil
    def t_soil():
        cy = VERI / "dunya/E10_soilgrids/clay/clay_0-5cm_mean_1000.tif"
        sy = VERI / "dunya/E10_soilgrids/silt/silt_0-5cm_mean_1000.tif"
        mn, mx = 10 ** 9, -10 ** 9
        with rasterio.open(cy) as dc, rasterio.open(sy) as ds:
            assert (dc.width, dc.height, dc.transform) == (ds.width, ds.height, ds.transform)
            OKUMA["E10"] = {"crs": str(dc.crs), "nodata_clay": dc.nodata, "nodata_silt": ds.nodata, "dtype": dc.dtypes[0],
                            "piksel": dc.res, "bounds": tuple(dc.bounds)}
            for r0 in range(0, dc.height, 1024):
                w = rasterio.windows.Window(0, r0, dc.width, min(1024, dc.height - r0))
                a, b = dc.read(1, window=w).astype(np.int32), ds.read(1, window=w).astype(np.int32)
                ok = (a != dc.nodata) & (b != ds.nodata)
                if ok.any():
                    t = a[ok] + b[ok]
                    mn, mx = min(mn, int(t.min())), max(mx, int(t.max()))
            tr = Transformer.from_crs("EPSG:4326", dc.crs, always_xy=True)
            x, y = tr.transform(N["kuru_vadiler"][1], N["kuru_vadiler"][0])
            b = dc.bounds
            disari = not (b.left <= x <= b.right and b.bottom <= y <= b.top)
            if disari:
                ant = "raster disinda"
            else:
                r, c = dc.index(x, y)
                ant = int(dc.read(1, window=rasterio.windows.Window(c, r, 1, 1))[0, 0])
                disari = ant == dc.nodata
        return f"silt+kil en kucuk {mn}, en buyuk {mx} (‰); Antarktika noktasi: {ant}", (0 <= mn and mx <= 1000 and disari)
    dene("E10", "silt+kil araligi; Antarktika veri yok", "kuru_vadiler", "0-1000 ‰; Antarktika veri yok",
         "gecerli piksellerde 0 <= silt+kil <= 1000 ve Antarktika noktasi veri yok ya da raster disi", "ISRIC", t_soil)

    # ------------------------------------------------ kayıt
    eski = []
    if D2CSV.exists():
        with open(D2CSV, encoding="utf-8", newline="") as f:
            eski = [r for r in csv.DictReader(f) if r["govde"] != "dunya"]
    with open(D2CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=ALANLAR)
        w.writeheader()
        w.writerows(eski + SATIRLAR)
    (KAYIT / "D2_okuma_dunya.json").write_text(json.dumps(OKUMA, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    say = {}
    for s in SATIRLAR:
        say[s["durum"]] = say.get(s["durum"], 0) + 1
    with open(KAYIT / "13_kasim_log.txt", "a", encoding="utf-8") as f:
        f.write(f"{datetime.now():%Y-%m-%d %H:%M:%S}  D2 dunya: {say}, {round((time.time() - t0) / 60, 1)} dk\n")
    print("=" * 60)
    print(f"D2 Dünya: {say} | {round((time.time() - t0) / 60, 1)} dk | kayit/D2.csv, kayit/D2_okuma_dunya.json")
    return say
'''
compile(kod, "d2_dunya.py", "exec")
dosya = KOK / "kod" / "d2_dunya.py"
dosya.write_text(kod, encoding="utf-8")
print("Yazıldı:", dosya, "\n")
g = runpy.run_path(str(dosya), run_name="d2_dunya")
_ = g["main"]()

Yazıldı: C:\Users\USER\orbitrary\kod\d2_dunya.py 

[PASS] E1 Mariana derinligi challenger: ölçülen -10571.2 m (±20 piksel penceresinin en düşüğü, ETOPO_2022_v1_15s_N15E135_surface.tif) | beklenen yaklasik -10550 m ([-11000, -10000] m (proje karari))
[PASS] E1 Olu Deniz yuzeyi olu_deniz: ölçülen -427.0 m (ETOPO_2022_v1_15s_N45E030_surface.tif) | beklenen yaklasik -430 m ([-450, -400] m (proje karari))
[PASS] E2 sinif 16 (ciplak) payi sahra: ölçülen sinif16=%100, toplam 100 | beklenen %100 (>= %95 ve 17 sinif toplami 100 (proje karari))
[PASS] E2 sinif 15 (kar/buz) payi summit: ölçülen sinif15=%100, toplam 100 | beklenen %100 (>= %95 ve 17 sinif toplami 100 (proje karari))
[PASS] E2 sinif 0 (su) payi nemo: ölçülen sinif0=%100, toplam 100 | beklenen %100 (>= %95 ve 17 sinif toplami 100 (proje karari))
[PASS] E4 olcek/ofset dosyadan : ölçülen Scale 0.002, Offset 0.49 (dosya özniteliği), NDVI Scale 0.001, sekil (5, 3600, 7200) | beklenen Scale 0,002; Offset 0,49; NDVI Scale 0,001 (esitlik (

TypeError: in method 'Band_GetScale', argument 1 of type 'GDALRasterBandShadow *'

In [15]:
# Developed with assistance from Claude (Anthropic).
# Adım D2 (Dünya) düzeltme — GDAL veri seti referansı tutulmadığı için bant geçersiz kalıyordu.
# Üç metin değiştirilir; her biri dosyada tam 1 kez geçmeli, yoksa hiçbir şey değişmez. Sonra testler yeniden koşar.
from pathlib import Path
import runpy

KOK = Path.home() / "orbitrary"
dosya = KOK / "kod" / "d2_dunya.py"
metin = dosya.read_text(encoding="utf-8")

yamalar = [
    ("SATIRLAR, OKUMA = [], {}\n",
     "SATIRLAR, OKUMA, _TUT = [], {}, []      # _TUT: açık GDAL veri setleri (silinirlerse bantları geçersiz olur)\n"),
    ("        return gdal.Open(s[0])\n",
     "        ds = gdal.Open(s[0])\n        _TUT.append(ds)                      # referans tutulur\n        return ds\n"),
    ("        bant = gdal.Open(str(ai_yol)).GetRasterBand(1)\n",
     "        ds9 = gdal.Open(str(ai_yol))\n        _TUT.append(ds9)\n        bant = ds9.GetRasterBand(1)\n"),
]
for eski, yeni in yamalar:
    if yeni in metin:
        print("ZATEN VAR:", eski.strip()[:60])
        continue
    n = metin.count(eski)
    assert n == 1, f"BULUNAMADI ya da birden fazla ({n}): {eski.strip()[:60]} — dosya değişmedi"
    metin = metin.replace(eski, yeni)
compile(metin, "d2_dunya.py", "exec")
dosya.write_text(metin, encoding="utf-8")
print("d2_dunya.py düzeltildi, sözdizimi OK\n")

g = runpy.run_path(str(dosya), run_name="d2_dunya")
_ = g["main"]()

d2_dunya.py düzeltildi, sözdizimi OK

[PASS] E1 Mariana derinligi challenger: ölçülen -10571.2 m (±20 piksel penceresinin en düşüğü, ETOPO_2022_v1_15s_N15E135_surface.tif) | beklenen yaklasik -10550 m ([-11000, -10000] m (proje karari))
[PASS] E1 Olu Deniz yuzeyi olu_deniz: ölçülen -427.0 m (ETOPO_2022_v1_15s_N45E030_surface.tif) | beklenen yaklasik -430 m ([-450, -400] m (proje karari))
[PASS] E2 sinif 16 (ciplak) payi sahra: ölçülen sinif16=%100, toplam 100 | beklenen %100 (>= %95 ve 17 sinif toplami 100 (proje karari))
[PASS] E2 sinif 15 (kar/buz) payi summit: ölçülen sinif15=%100, toplam 100 | beklenen %100 (>= %95 ve 17 sinif toplami 100 (proje karari))
[PASS] E2 sinif 0 (su) payi nemo: ölçülen sinif0=%100, toplam 100 | beklenen %100 (>= %95 ve 17 sinif toplami 100 (proje karari))
[PASS] E4 olcek/ofset dosyadan : ölçülen Scale 0.002, Offset 0.49 (dosya özniteliği), NDVI Scale 0.001, sekil (5, 3600, 7200) | beklenen Scale 0,002; Offset 0,49; NDVI Scale 0,001 (esitlik (talimat 7: Of

In [16]:
# Developed with assistance from Claude (Anthropic).
# Adım D2 (Ay) — kod\d2_ay.py yazılır ve çalıştırılır (belge 17.3 D2, M1–M7; talimat 6).
from pathlib import Path
import runpy

KOK = Path.home() / "orbitrary"
kod = r'''# Developed with assistance from Claude (Anthropic).
# d2_ay.py — Orbitrary v1.2, belge 17.3 D2 (gerçek-yer testleri), Ay setleri M1–M7 + talimat 6 (PRP ice_depth).
# Sonuçlar kayit/D2.csv'ye (govde = ay), okuma ayrıntıları kayit/D2_okuma_ay.json'a yazılır.
import csv, json, math, re, time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

KOK = Path.home() / "orbitrary"
VERI, KAYIT = KOK / "veri", KOK / "kayit"
D2CSV = KAYIT / "D2.csv"
ALANLAR = ["govde", "set", "test", "nokta", "enlem", "boylam", "koordinat_kaynagi", "beklenen",
           "tolerans_kurali", "beklenen_kaynagi", "olculen", "durum", "zaman"]
R_AY = 1737400.0                                   # m, belge 4.1 (r_B = 1737,4 km)

N = {
    "apollo11": (0.67416, 23.47314, "NSSDCA Apollo Landing Site Coordinates (LRO, Wagner ve ark. 2017): LM 0.67416N 23.47314E"),
    "shackleton": (-89.67, 129.78, "IAU Gazetteer of Planetary Nomenclature, Feature 5450: -89.67, 129.78 (cap 21 km)"),
}
SATIRLAR, OKUMA, _TUT = [], {}, []


def kaydet(set_, test, nokta, beklenen, kural, bek_kaynak, olculen, durum):
    e, b, k = (N[nokta] if nokta in N else ("", "", ""))
    SATIRLAR.append({"govde": "ay", "set": set_, "test": test, "nokta": nokta, "enlem": e, "boylam": b,
                     "koordinat_kaynagi": k, "beklenen": beklenen, "tolerans_kurali": kural,
                     "beklenen_kaynagi": bek_kaynak, "olculen": olculen, "durum": durum,
                     "zaman": datetime.now().isoformat(timespec="seconds")})
    print(f"[{durum}] {set_} {test} {nokta}: ölçülen {olculen} | beklenen {beklenen} ({kural})", flush=True)


def dene(set_, test, nokta, beklenen, kural, bek_kaynak, f):
    try:
        olculen, ok = f()
        kaydet(set_, test, nokta, beklenen, kural, bek_kaynak, olculen, ok if isinstance(ok, str) else ("PASS" if ok else "FAIL"))
    except Exception as e:
        kaydet(set_, test, nokta, beklenen, kural, bek_kaynak, f"{type(e).__name__}: {e}", "FAIL")


def gc_km(f1, l1, f2, l2):
    f1, l1, f2, l2 = map(np.radians, (f1, l1, f2, l2))
    a = np.sin((f2 - f1) / 2) ** 2 + np.cos(f1) * np.cos(f2) * np.sin((l2 - l1) / 2) ** 2
    return 2 * R_AY / 1000 * np.arcsin(np.sqrt(a))


def lbl_sutunlar(lbl):
    # PDS3 etiketindeki COLUMN nesnelerinin NAME alanları, sırasıyla
    t = lbl.read_text(encoding="ascii", errors="replace")
    adlar = []
    for blok in re.findall(r"OBJECT\s*=\s*COLUMN(.*?)END_OBJECT\s*=\s*COLUMN", t, re.S):
        m = re.search(r"\bNAME\s*=\s*\"?([A-Za-z0-9_]+)\"?", blok)
        if m:
            adlar.append(m.group(1).lower())
    return adlar


def oku_tablo(tab, adlar):
    ilk = tab.open(encoding="ascii", errors="replace").readline()
    baslik = any(c.isalpha() for c in ilk.replace("e", "").replace("E", ""))
    df = pd.read_csv(tab, header=0 if baslik else None, names=None if baslik else adlar, skipinitialspace=True)
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df, baslik


def main():
    t0 = time.time()
    from osgeo import gdal
    gdal.UseExceptions()
    import rasterio

    def bolum(ad, f):
        try:
            f()
        except Exception as e:
            kaydet(ad, "bolum calisamadi", "", "", "", "", f"{type(e).__name__}: {e}", "FAIL")

    # ------------------------------------------------ M1 LOLA (int16 x 0,5 m)
    def m1():
        lola = VERI / "ay/M1_lola/Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif"
        lbl = (VERI / "ay/M1_lola/Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.lbl").read_text(encoding="ascii", errors="replace")
        m_c, m_b = re.search(r"Multiplier\s*=\s*([-\d.Ee+]+)", lbl), re.search(r"Base\s*=\s*([-\d.Ee+]+)", lbl)
        carpan = float(m_c.group(1)) if m_c else None
        taban = float(m_b.group(1)) if m_b else None
        with rasterio.open(lola) as d:
            tr = d.transform
            OKUMA["M1"] = {"boyut": [d.width, d.height], "dtype": d.dtypes[0], "nodata": d.nodata, "transform": list(tr)[:6],
                           "lbl_multiplier": carpan, "lbl_base": taban}
            assert (d.width, d.height, d.dtypes[0]) == (92160, 46080, "int16"), OKUMA["M1"]
            assert carpan == 0.5 and taban == 0.0, f"etiket Multiplier {carpan}, Base {taban} (belge: x 0,5)"
            piksel = 2 * math.pi * R_AY / 92160
            assert abs(tr.a - piksel) < 0.01 and abs(tr.c + math.pi * R_AY) < 1 and abs(tr.f - math.pi * R_AY / 2) < 1, \
                f"transform {list(tr)[:6]}: 256 ppd, -180..180 ve R = 1737,4 km ile tutmuyor"

            def lola_rc(e, b):
                return int(math.floor((90 - e) * 256)), int(math.floor((b + 180) * 256))

            def t_uc():
                mn, mx, n_yok = 32767, -32768, 0
                for r0 in range(0, d.height, 512):
                    a = d.read(1, window=rasterio.windows.Window(0, r0, d.width, min(512, d.height - r0)))
                    yok = a == -32768
                    n_yok += int(yok.sum())
                    v = a[~yok]
                    if v.size:
                        mn, mx = min(mn, int(v.min())), max(mx, int(v.max()))
                OKUMA["M1"]["yok_sayisi_-32768"] = n_yok
                emn, emx = mn * carpan + taban, mx * carpan + taban
                return f"en kucuk {emn:.1f} m, en buyuk {emx:.1f} m (ham {mn}/{mx}); -32768 sayisi {n_yok}", \
                    abs(emn + 9128) <= 50 and abs(emx - 10782) <= 50
            dene("M1", "ham x 0,5 uc degerler", "", "-9128 / +10782 m", "her biri +-50 m (proje karari)",
                 "belge 17.3 D2 (yayımlanmış LOLA uç değerleri)", t_uc)

            def t_a11():
                r, c = lola_rc(*N["apollo11"][:2])
                v = int(d.read(1, window=rasterio.windows.Window(c, r, 1, 1))[0, 0]) * carpan + taban
                return f"{v:.1f} m", abs(v + 1926) <= 50
            dene("M1", "Apollo 11 yuksekligi", "apollo11", "yaklasik -1926 m", "+-50 m (proje karari)", "belge 17.3 D2 (LOLA/LROC)", t_a11)

    bolum('M1', m1)

    # ------------------------------------------------ M4 PRP (talimat 6) ve Shackleton
    prp = {}                                   # PRP tabloları (M4 ve M7 testleri kullanır)

    def m4():
        for yari in ("north", "south"):
            tab = VERI / f"ay/M4_diviner_prp_gcp/dlre_prp_{yari}.tab"
            adlar = lbl_sutunlar(tab.with_suffix(".lbl"))
            df, baslik = oku_tablo(tab, adlar)
            prp[yari] = df
            OKUMA[f"M4_prp_{yari}"] = {"satir": len(df), "sutunlar": list(df.columns), "dosyada_baslik": baslik}
        for yari, df in prp.items():
            dene("M4", f"PRP {yari} yapi", "", "2 880 000 kayit; tri_clon, tri_clat, temp_avg, temp_max, ice_depth",
                 "esitlik", "rapor M4",
                 lambda df=df: (f"{len(df)} kayit, sutunlar {list(df.columns)}",
                                len(df) == 2880000 and all(c in df.columns for c in ("tri_clon", "tri_clat", "temp_avg", "temp_max", "ice_depth"))))
        def t6():
            sonuc, ok = {}, True
            for yari, df in prp.items():
                ice, tmax = df["ice_depth"].to_numpy(), df["temp_max"].to_numpy()
                negler = sorted(set(np.unique(ice[ice < 0]).tolist()))
                bir = ice == -1
                sonuc[yari] = {"negatif_degerler": negler, "n_0": int((ice == 0).sum()), "n_-1": int(bir.sum()),
                               "n_pozitif": int((ice > 0).sum()), "n_-999": int((ice == -999).sum()),
                               "-1_temp_max_en_buyuk": float(tmax[bir].max()) if bir.any() else None}
                ok &= set(negler) <= {-1.0, -999.0}
                ok &= (not bir.any()) or tmax[bir].max() <= 101.3
            OKUMA["M4_ice_depth_sayilari"] = sonuc
            return json.dumps(sonuc, ensure_ascii=False), ok
        dene("M4", "ice_depth negatifleri ve -1 sicakligi (talimat 6)", "",
             "negatifler yalniz {-1, -999}; -1 olan butun ucgenlerde temp_max <= 101,3 K", "iki assert (talimat 6)",
             "talimat 6; Fisher ve ark. 2017, Icarus 292, 74-85", t6)
        def t_shack():
            df = prp["south"]
            u = gc_km(N["shackleton"][0], N["shackleton"][1], df["tri_clat"].to_numpy(), df["tri_clon"].to_numpy())
            sec = df["temp_max"].to_numpy()[u <= 3.0]
            return f"merkezin 3 km icinde {sec.size} ucgen, temp_max ortanca {np.median(sec):.1f} K (en buyuk {sec.max():.1f})", \
                sec.size > 0 and np.median(sec) < 90
        dene("M4", "Shackleton tabani Tmax", "shackleton", "Tmax < 90 K",
             "merkezin 3 km icindeki ucgenlerde temp_max ortancasi < 90 K (proje karari; IAU merkezi tam olmayabilir)",
             "belge 17.3 D2; Paige ve ark. 2010", t_shack)

    bolum('M4', m4)

    # ------------------------------------------------ M3 GHRM <-> M4 GCP, 70° bandı gece yarısı
    def m34():
        def ghrm_bant(kuzey):
            yol = VERI / "ay/M3_diviner_ghrm/dghrm_tbol_sam_70s70n_tif.tif"
            with rasterio.open(yol) as d:
                tr = d.transform
                OKUMA["M3"] = {"boyut": [d.width, d.height], "dtype": d.dtypes[0], "nodata": d.nodata, "transform": list(tr)[:6], "crs": str(d.crs)}
                assert abs(tr.a - 1 / 128) < 1e-9 and abs(tr.e + 1 / 128) < 1e-9 and abs(tr.f - 70) < 1e-6, f"GHRM transform {list(tr)[:6]}"
                r0 = 0 if kuzey else d.height - 64                       # 0,5° = 64 satır: 70..69,5 ya da -69,5..-70
                a = d.read(1, window=rasterio.windows.Window(0, r0, d.width, 64)).astype(float)
            a[~np.isfinite(a) | (a >= 250)] = np.nan                     # belge 10.2: tbol >= 250 K yok sayılır
            return float(np.nanmean(a)), int(np.isfinite(a).sum())
        def gcp_bant(dosya, enlem):
            df = pd.read_csv(VERI / "ay/M4_diviner_prp_gcp" / dosya, skipinitialspace=True)
            df.columns = [c.strip().lower() for c in df.columns]
            OKUMA[f"M4_{dosya}"] = {"satir": len(df), "sutunlar": list(df.columns)}
            s = df[(np.isclose(df["clat"], enlem)) & ((df["ltim"] < 0.25) | (df["ltim"] > 23.75)) & (df["tbol"] != -9999)]
            return float(s["tbol"].mean()), len(s), len(df)
        for kuzey, dosya, enlem in [(True, "global_cumul_avg_cyl_70n80n_002.tab", 70.25), (False, "global_cumul_avg_cyl_80s70s_002.tab", -70.25)]:
            def t_b(kuzey=kuzey, dosya=dosya, enlem=enlem):
                g, ng = ghrm_bant(kuzey)
                c, nc, toplam = gcp_bant(dosya, enlem)
                return f"GHRM {g:.2f} K ({ng} piksel) | GCP {c:.2f} K ({nc} satir, dosya {toplam} satir) | fark {abs(g - c):.2f} K", abs(g - c) < 1.0
            dene("M3/M4", f"GCP-GHRM 70 bandi ({'kuzey' if kuzey else 'guney'})", "", "fark < 1 K",
                 "GHRM 69,5-70° ortalamasi ile GCP 70-70,5° gece yarisi (ltim < 0,25 ya da > 23,75) ortalamasi farki < 1 K",
                 "belge 17.3 D2 ve 6.5 (kuzey 82,6/83,3 K, güney 82,1/82,9 K)", t_b)

    bolum('M3/M4', m34)

    # ------------------------------------------------ M5 LP GRS: alan toplamı = 4πR²
    def m5():
        def t_grs():
            satir = (VERI / "ay/M5_lp_grs/lpgrs_high1_elem_abundance_2deg.tab").read_text(encoding="ascii").splitlines()
            alan = 0.0
            for s in satir:
                v = [float(x) for x in re.split(r"[,\s]+", s.strip())[:5]]
                fs, fn, lw, le = math.radians(v[1]), math.radians(v[2]), v[3], v[4]
                dl = (le - lw) % 360 or 360
                alan += math.radians(dl) * (math.sin(fn) - math.sin(fs))
            oran = alan / (4 * math.pi)
            return f"{len(satir)} piksel, alan / 4πR² = {oran:.6f}", len(satir) == 11306 and abs(oran - 1) <= 1e-4
        dene("M5", "alan toplami", "", "4πR²", "+-%0,01 (belge 17.3 SIM 3292 ile ayni tolerans; proje karari)", "belge 17.3 D2", t_grs)

    bolum('M5', m5)

    # ------------------------------------------------ M6 LP hidrojen
    def m6b():
        def m6():
            satirlar = []
            for s in (VERI / "ay/M6_lp_hidrojen/hydrogenhd.txt").read_text(encoding="ascii", errors="replace").splitlines():
                if not s.strip() or s.lstrip().startswith("#"):
                    continue
                p = [x.strip() for x in s.split(",")]
                h = float("nan") if "*" in p[4] else float(p[4])
                satirlar.append((float(p[0]), float(p[1]), float(p[2]), float(p[3]), h))
            return np.array(satirlar)
        H = m6()
        def t_bozuk():
            b = H[np.isnan(H[:, 4])]
            return f"bozuk hucre(ler): {b[:, :4].tolist()}", len(b) == 1 and b[0, :4].tolist() == [43.0, 43.5, 153.5, 154.0]
        dene("M6", "bozuk hucrenin yeri", "", "43,0-43,5°K 153,5-154,0°D", "esitlik", "rapor M6 / belge 10.2", t_bozuk)
        def t_neg():
            n = int((H[:, 4] < 0).sum())
            return f"{n} negatif hucre", n == 4180
        dene("M6", "negatif hucre sayisi", "", "4180", "esitlik (kirpilmaz)", "rapor M6 / belge 10.2", t_neg)
        def t_kutup():
            e = (H[:, 0] + H[:, 1]) / 2
            k = np.nanmean(H[np.abs(e) >= 80, 4])
            q = np.nanmean(H[np.abs(e) <= 10, 4])
            return f">=80° ortalama {k:.1f} ppm, |enlem|<=10° ortalama {q:.1f} ppm", k > q
        dene("M6", "kutuplar zengin", "", ">= 80° ortalama > ekvator ortalamasi",
             "|enlem| >= 80 ortalamasi > |enlem| <= 10 ortalamasi (proje karari: ekvator bandi +-10°)", "Feldman ve ark. 2001", t_kutup)

    bolum('M6', m6b)

    # ------------------------------------------------ M7 PGDA
    def m7():
        def pgda(ad):
            p = VERI / "ay/M7_pgda" / f"{ad}_240M_201608.TIF"
            lb = (VERI / "ay/M7_pgda" / f"{ad}_240M_201608.LBL").read_text(encoding="ascii", errors="replace")
            sf = float(re.search(r"SCALING_FACTOR\s*=\s*([-\d.Ee+]+)", lb).group(1))
            of = float(re.search(r"\bOFFSET\s*=\s*([-\d.Ee+]+)", lb).group(1))
            with rasterio.open(p) as d:
                a = d.read(1).astype(float)
                OKUMA[f"M7_{ad}"] = {"boyut": [d.width, d.height], "dtype": d.dtypes[0], "nodata": d.nodata,
                                     "transform": list(d.transform)[:6], "lbl_scaling": sf, "lbl_offset": of}
            return a * sf + of, sf, of
        for kutup in ("S", "N"):
            def t_vis(kutup=kutup):
                av, s1, o1 = pgda(f"AVGVISIB_65{kutup}")
                ps, s2, o2 = pgda(f"LPSR_65{kutup}")
                assert (s1, o1) == (0.00004, 0.0) and (s2, o2) == (0.000025, 0.5), f"etiket olcek/ofset {s1},{o1} / {s2},{o2}"
                g = np.isclose(ps, 1.0)
                return f"LPSR=1 piksel {int(g.sum())}; bunlarda AVGVISIB en buyuk {av[g].max():.5f}, sifir orani {np.mean(av[g] == 0):.4f}", \
                    bool(g.any() and av[g].max() == 0)
            dene("M7", f"LPSR=1 -> AVGVISIB 0 ({kutup})", "", "0", "LPSR = 1 olan butun piksellerde AVGVISIB = 0; etiket olcek/ofset belgeyle ayni",
                 "etiket tanımı; belge 10.2", t_vis)
        def t_soguk():
            sonuc, ok = [], True
            for kutup, yari in (("S", "south"), ("N", "north")):
                ps, _, _ = pgda(f"LPSR_65{kutup}")
                n = ps.shape[0]
                yarim = n * 240 / 2
                df = prp[yari]
                df = df[np.abs(df["tri_clat"]) >= 85].nsmallest(100, "temp_max")
                kol = np.radians(90 - np.abs(df["tri_clat"].to_numpy()))
                rho = 2 * R_AY * np.tan(kol / 2)
                lam = np.radians(df["tri_clon"].to_numpy())
                x, y = rho * np.sin(lam), rho * np.cos(lam)          # belge 6.8 / 10.2: iki kutupta da x = ρ sin λ, y = ρ cos λ
                c = np.floor((x + yarim) / 240).astype(int)
                r = np.floor((yarim - y) / 240).astype(int)
                ic = (r >= 0) & (r < n) & (c >= 0) & (c < n)
                oran = float(np.mean(np.isclose(ps[r[ic], c[ic]], 1.0)))
                sonuc.append(f"{kutup}: en soguk 100 ucgen (temp_max {df['temp_max'].min():.1f}-{df['temp_max'].max():.1f} K), LPSR=1 orani {oran:.2f}")
                ok &= oran >= 0.9
            return " | ".join(sonuc), ok
        dene("M7", "Diviner en soguk ucgenler -> LPSR=1 (iki kutup)", "", "LPSR = 1",
             "|enlem| >= 85° en dusuk temp_max'li 100 PRP ucgeninin >= %90'i LPSR = 1 (proje karari); x = ρ sin λ, y = ρ cos λ",
             "belge 17.3 D2 (bağımsız ürün); belge 6.8", t_soguk)

    bolum('M7', m7)

    # ------------------------------------------------ kayıt
    eski = []
    if D2CSV.exists():
        with open(D2CSV, encoding="utf-8", newline="") as f:
            eski = [r for r in csv.DictReader(f) if r["govde"] != "ay"]
    with open(D2CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=ALANLAR)
        w.writeheader()
        w.writerows(eski + SATIRLAR)
    (KAYIT / "D2_okuma_ay.json").write_text(json.dumps(OKUMA, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    say = {}
    for s in SATIRLAR:
        say[s["durum"]] = say.get(s["durum"], 0) + 1
    with open(KAYIT / "13_kasim_log.txt", "a", encoding="utf-8") as f:
        f.write(f"{datetime.now():%Y-%m-%d %H:%M:%S}  D2 ay: {say}, {round((time.time() - t0) / 60, 1)} dk\n")
    print("=" * 60)
    print(f"D2 Ay: {say} | {round((time.time() - t0) / 60, 1)} dk | kayit/D2.csv, kayit/D2_okuma_ay.json")
    return say
'''
compile(kod, "d2_ay.py", "exec")
dosya = KOK / "kod" / "d2_ay.py"
dosya.write_text(kod, encoding="utf-8")
print("Yazıldı:", dosya, "\n")
g = runpy.run_path(str(dosya), run_name="d2_ay")
_ = g["main"]()

Yazıldı: C:\Users\USER\orbitrary\kod\d2_ay.py 

[PASS] M1 ham x 0,5 uc degerler : ölçülen en kucuk -9128.5 m, en buyuk 10781.5 m (ham -18257/21563); -32768 sayisi 0 | beklenen -9128 / +10782 m (her biri +-50 m (proje karari))
[PASS] M1 Apollo 11 yuksekligi apollo11: ölçülen -1927.5 m | beklenen yaklasik -1926 m (+-50 m (proje karari))
[PASS] M4 PRP north yapi : ölçülen 2880000 kayit, sutunlar ['tri1_x', 'tri1_y', 'tri1_z', 'tri2_x', 'tri2_y', 'tri2_z', 'tri3_x', 'tri3_y', 'tri3_z', 'tri_clon', 'tri_clat', 'tri_calt', 'temp_avg', 'temp_max', 'ice_depth'] | beklenen 2 880 000 kayit; tri_clon, tri_clat, temp_avg, temp_max, ice_depth (esitlik)
[PASS] M4 PRP south yapi : ölçülen 2880000 kayit, sutunlar ['tri1_x', 'tri1_y', 'tri1_z', 'tri2_x', 'tri2_y', 'tri2_z', 'tri3_x', 'tri3_y', 'tri3_z', 'tri_clon', 'tri_clat', 'tri_calt', 'temp_avg', 'temp_max', 'ice_depth'] | beklenen 2 880 000 kayit; tri_clon, tri_clat, temp_avg, temp_max, ice_depth (esitlik)
[PASS] M4 ice_depth negatifleri ve -1 sic

In [17]:
# Developed with assistance from Claude (Anthropic).
# D2 Mars hazırlığı: veri\mars dosya ağacı + etiketlerdeki biçim satırları -> kayit\mars_etiket_ozet.txt
from pathlib import Path
import re

KOK = Path.home() / "orbitrary"
MARS = KOK / "veri" / "mars"
ANAHTAR = re.compile(r"(RECORD_BYTES|RECORD_TYPE|FILE_RECORDS|LABEL_RECORDS|\^IMAGE|\^TABLE|\^HEADER|LINES|LINE_SAMPLES|"
                     r"SAMPLE_TYPE|SAMPLE_BITS|BANDS|BAND_STORAGE|SCALING_FACTOR|OFFSET|MISSING|NULL|INVALID|NODATA|"
                     r"byte order|data type|header offset|samples|lines|bands|interleave|record_length|records|"
                     r"MAP_PROJECTION|CENTER_LONGITUDE|POSITIVE_LONGITUDE|MAP_RESOLUTION|MAXIMUM_LATITUDE|MINIMUM_LATITUDE|"
                     r"WESTERNMOST|EASTERNMOST|UNIT)", re.I)
satir = []
for p in sorted(MARS.rglob("*")):
    if p.is_file():
        satir.append(f"{p.relative_to(MARS)}  {p.stat().st_size:,} bayt")
satir.append("\n" + "=" * 70 + "\nETİKET SATIRLARI\n")
for p in sorted(MARS.rglob("*")):
    if p.is_file() and p.suffix.lower() in {".lbl", ".hdr", ".xml", ".txt"} and p.stat().st_size < 2_000_000:
        t = p.read_text(encoding="ascii", errors="replace").splitlines()
        bul = [s.strip() for s in t if ANAHTAR.search(s)][:60]
        if bul:
            satir.append(f"--- {p.relative_to(MARS)}")
            satir += ["    " + s for s in bul]
cikti = "\n".join(satir)
(KOK / "kayit" / "mars_etiket_ozet.txt").write_text(cikti, encoding="utf-8")
print(cikti)

R1_mola\Mars_MGS_MOLA_DEM_mosaic_global_463m.lbl  1,703 bayt
R1_mola\Mars_MGS_MOLA_DEM_mosaic_global_463m.tif  2,125,771,142 bayt
R1_mola\Mars_MGS_MOLA_DEM_mosaic_global_463m.tif.md5  75 bayt
R2_sim3292\acik\SIM3292_MarsGlobalGeologicGIS_20M\Basemaps\hillshade_n_128.aux  366,307 bayt
R2_sim3292\acik\SIM3292_MarsGlobalGeologicGIS_20M\Basemaps\hillshade_n_128.jgw  126 bayt
R2_sim3292\acik\SIM3292_MarsGlobalGeologicGIS_20M\Basemaps\hillshade_n_128.jpg  9,768,753 bayt
R2_sim3292\acik\SIM3292_MarsGlobalGeologicGIS_20M\Basemaps\hillshade_n_128.jpg.aux.xml  2,187 bayt
R2_sim3292\acik\SIM3292_MarsGlobalGeologicGIS_20M\Basemaps\hillshade_n_128.rrd  35,044,370 bayt
R2_sim3292\acik\SIM3292_MarsGlobalGeologicGIS_20M\Basemaps\hillshade_s_128.aux  365,840 bayt
R2_sim3292\acik\SIM3292_MarsGlobalGeologicGIS_20M\Basemaps\hillshade_s_128.jgw  126 bayt
R2_sim3292\acik\SIM3292_MarsGlobalGeologicGIS_20M\Basemaps\hillshade_s_128.jpg  14,407,308 bayt
R2_sim3292\acik\SIM3292_MarsGlobalGeologicGIS_20M\Basemaps

In [18]:
# Developed with assistance from Claude (Anthropic).
# Adım D2 (Mars) — kod\d2_mars.py yazılır ve çalıştırılır (belge 17.3 D2, R1–R7).
from pathlib import Path
import runpy

KOK = Path.home() / "orbitrary"
kod = r'''# Developed with assistance from Claude (Anthropic).
# d2_mars.py — Orbitrary v1.2, belge 17.3 D2 (gerçek-yer testleri), Mars setleri R1–R7.
# Sonuçlar kayit/D2.csv'ye (govde = mars), okuma ayrıntıları kayit/D2_okuma_mars.json'a yazılır.
import csv, json, math, re, time
from datetime import datetime
from pathlib import Path

import numpy as np

KOK = Path.home() / "orbitrary"
VERI, KAYIT = KOK / "veri", KOK / "kayit"
M = VERI / "mars"
D2CSV = KAYIT / "D2.csv"
ALANLAR = ["govde", "set", "test", "nokta", "enlem", "boylam", "koordinat_kaynagi", "beklenen",
           "tolerans_kurali", "beklenen_kaynagi", "olculen", "durum", "zaman"]
R_MARS = 3396190.0          # m; SIM3292_README.txt: "GCS_Mars_2000_Sphere, 3396190.0 radius"

N = {
    "jezero": (18.44, 77.45, "belge 17.3 D2 (Jezero merkezi)"),
    "opportunity": (-1.9462, 354.4734 - 360.0,
                    "PDS Geosciences mer_mission.txt: 1.9462 S, 354.4734 E (MOLA IAU 2000); Meridiani hematit bölgesi"),
}
SATIRLAR, OKUMA = [], {}


def kaydet(set_, test, nokta, beklenen, kural, bek_kaynak, olculen, durum):
    e, b, k = (N[nokta] if nokta in N else ("", "", ""))
    SATIRLAR.append({"govde": "mars", "set": set_, "test": test, "nokta": nokta, "enlem": e, "boylam": b,
                     "koordinat_kaynagi": k, "beklenen": beklenen, "tolerans_kurali": kural,
                     "beklenen_kaynagi": bek_kaynak, "olculen": olculen, "durum": durum,
                     "zaman": datetime.now().isoformat(timespec="seconds")})
    print(f"[{durum}] {set_} {test} {nokta}: ölçülen {olculen} | beklenen {beklenen} ({kural})", flush=True)


def dene(set_, test, nokta, beklenen, kural, bek_kaynak, f):
    try:
        olculen, ok = f()
        kaydet(set_, test, nokta, beklenen, kural, bek_kaynak, olculen, "PASS" if ok else "FAIL")
    except Exception as e:
        kaydet(set_, test, nokta, beklenen, kural, bek_kaynak, f"{type(e).__name__}: {e}", "FAIL")


def bolum(ad, f):
    try:
        f()
    except Exception as e:
        kaydet(ad, "bolum calisamadi", "", "", "", "", f"{type(e).__name__}: {e}", "FAIL")


def gc_km(f1, l1, f2, l2):
    f1, l1, f2, l2 = map(np.radians, (f1, l1, f2, l2))
    a = np.sin((f2 - f1) / 2) ** 2 + np.cos(f1) * np.cos(f2) * np.sin((l2 - l1) / 2) ** 2
    return 2 * R_MARS / 1000 * np.arcsin(np.sqrt(a))


def pds(t, *anahtarlar):
    # PDS3 / ISIS etiketinde "ANAHTAR = deger <birim>" satırının değeri
    for a in anahtarlar:
        m = re.search(r"(?im)^\s*" + re.escape(a) + r"\s*=\s*\"?([^\"\r\n<]*?)\"?\s*(<[^>]*>)?\s*$", t)
        if m:
            return m.group(1).strip()
    return None


def sayi(s):
    return float(re.match(r"\s*([-+]?[\d.]+(?:[Ee][-+]?\d+)?)", s).group(1))


def kor(a, b):
    from scipy.stats import spearmanr
    return float(np.corrcoef(a, b)[0, 1]), float(spearmanr(a, b).correlation)


def albedo_oku():
    # TES albedo 8 ppd: etiketten okunur (PC_REAL = little-endian float32, kuzey üstte, -180'den doğuya)
    lbl = (M / "R4_tes_ti_albedo/global_albedo_8ppd.lbl").read_text(encoding="ascii", errors="replace")
    sat, sut = int(sayi(pds(lbl, "LINES"))), int(sayi(pds(lbl, "LINE_SAMPLES")))
    tip, bit = pds(lbl, "SAMPLE_TYPE"), int(sayi(pds(lbl, "SAMPLE_BITS")))
    maxlat, bati = sayi(pds(lbl, "MAXIMUM_LATITUDE")), sayi(pds(lbl, "WESTERNMOST_LONGITUDE"))
    yon, cozun = pds(lbl, "POSITIVE_LONGITUDE_DIRECTION"), sayi(pds(lbl, "MAP_RESOLUTION"))
    OKUMA["R4_albedo"] = {"satir": sat, "sutun": sut, "tip": tip, "bit": bit, "maxlat": maxlat, "bati": bati,
                          "yon": yon, "ppd": cozun}
    assert (sat, sut, tip, bit, maxlat, bati, yon, cozun) == (1440, 2880, "PC_REAL", 32, 90.0, -180.0, "EAST", 8.0), OKUMA["R4_albedo"]
    p = M / "R4_tes_ti_albedo/global_albedo_8ppd.img"
    assert p.stat().st_size == sat * sut * 4
    a = np.fromfile(p, dtype="<f4").reshape(sat, sut).astype(float)
    OKUMA["R4_albedo"]["sonlu_olmayan"] = int((~np.isfinite(a)).sum())
    OKUMA["R4_albedo"]["sifir_ve_alti"] = int((a <= 0).sum())
    a[~np.isfinite(a) | (a <= 0)] = np.nan          # proje kararı: albedo > 0 geçerli (etikette veri-yok değeri yok)
    lat = 90 - (np.arange(sat) + 0.5) / 8
    lon = -180 + (np.arange(sut) + 0.5) / 8
    return a, lat, lon


def main():
    t0 = time.time()
    import rasterio

    # ------------------------------------------------ R1 MOLA (int16 m; transform kullanılmaz)
    def r1():
        p = M / "R1_mola/Mars_MGS_MOLA_DEM_mosaic_global_463m.tif"
        with rasterio.open(p) as d:
            OKUMA["R1"] = {"boyut": [d.width, d.height], "dtype": d.dtypes[0], "nodata": d.nodata,
                           "olcek": d.scales[0], "ofset": d.offsets[0], "crs": str(d.crs), "transform": list(d.transform)[:6]}
            assert (d.width, d.height, d.dtypes[0]) == (46080, 23040, "int16"), OKUMA["R1"]
            assert d.scales[0] == 1.0 and d.offsets[0] == 0.0, f"dosyada ölçek/ofset var: {d.scales[0]}, {d.offsets[0]}"
            nd = d.nodata

            def t_uc():
                mn, mx, rmn, rmx, n_yok = 10 ** 9, -10 ** 9, None, None, 0
                for r0 in range(0, d.height, 512):
                    a = d.read(1, window=rasterio.windows.Window(0, r0, d.width, min(512, d.height - r0)))
                    gec = np.ones(a.shape, bool) if nd is None else (a != nd)
                    n_yok += int((~gec).sum())
                    v = np.where(gec, a, 32767)
                    i = np.unravel_index(np.argmin(v), v.shape)
                    if v[i] < mn:
                        mn, rmn = int(v[i]), (r0 + i[0], i[1])
                    v = np.where(gec, a, -32768)
                    i = np.unravel_index(np.argmax(v), v.shape)
                    if v[i] > mx:
                        mx, rmx = int(v[i]), (r0 + i[0], i[1])
                yer = lambda rc: (round(90 - (rc[0] + 0.5) / 128, 2), round((rc[1] + 0.5) / 128 - 180, 2))
                OKUMA["R1"].update({"min": mn, "min_yeri": yer(rmn), "max": mx, "max_yeri": yer(rmx), "nodata_sayisi": n_yok})
                return f"en kucuk {mn} m {yer(rmn)}, en buyuk {mx} m {yer(rmx)}; veri-yok {n_yok}", \
                    abs(mn + 8200) <= 100 and 21129 <= mx <= 21400
            dene("R1", "kuresel min / max", "", "yaklasik -8200 (Hellas) / 21229-21300 m (Olympus)",
                 "min -8200 +-100; max 21129-21400 (21229-21300 +-100; proje karari)", "belge 17.3 D2 (PGDA MOLA katalogu)", t_uc)

            def t_jez():
                e, b = N["jezero"][:2]
                r, c = int(math.floor((90 - e) * 128)), int(math.floor((b + 180) * 128))
                v = int(d.read(1, window=rasterio.windows.Window(c, r, 1, 1))[0, 0])
                return f"{v} m (satir {r}, sutun {c})", abs(v + 2570) <= 100
            dene("R1", "Jezero yuksekligi", "jezero", "yaklasik -2570 m", "+-100 m (proje karari; 463 m piksel)",
                 "belge 17.3 D2 (PDS MEGDR çapraz ürün)", t_jez)

    bolum("R1", r1)

    # ------------------------------------------------ R2 SIM 3292: alan toplamı
    def r2():
        import pyogrio
        shp = M / "R2_sim3292/acik/SIM3292_MarsGlobalGeologicGIS_20M/SIM3292_Shapefiles/SIM3292_Global_Geology.shp"
        df = pyogrio.read_dataframe(shp, read_geometry=False)
        OKUMA["R2"] = {"kayit": len(df), "alanlar": list(df.columns), "birim_sayisi": int(df["Unit"].nunique())}

        def t_alan():
            oran = float(df["SphArea_km"].sum()) / (4 * math.pi * (R_MARS / 1000) ** 2)
            return f"{len(df)} kayit, {df['Unit'].nunique()} birim, alan / 4πR² = {oran:.6f}", \
                len(df) == 1311 and df["Unit"].nunique() == 44 and abs(oran - 1) <= 1e-4
        dene("R2", "alan toplami", "", "4πR² (R = 3396,19 km); 1311 kayit, 44 birim", "+-%0,01",
             "belge 17.3 D2; R: SIM3292_README.txt", t_alan)

    bolum("R2", r2)

    # ------------------------------------------------ R3 TES mineral
    def r3():
        hdr = (M / "R3_tes_mineral/TES_mineral_maps_4_ppd.envi.hdr").read_text(encoding="ascii", errors="replace")
        h = {k: int(sayi(pds(hdr, k))) for k in ("samples", "lines", "bands", "header offset", "data type", "byte order")}
        OKUMA["R3_hdr"] = h
        assert h == {"samples": 1440, "lines": 720, "bands": 13, "header offset": 0, "data type": 4, "byte order": 1}, h
        p = M / "R3_tes_mineral/TES_mineral_maps_4_ppd.envi"
        assert p.stat().st_size == 1440 * 720 * 13 * 4
        a = np.fromfile(p, dtype=">f4").reshape(13, 720, 1440)
        # geometri: ENVI .hdr'de yok -> aynı verinin .isis kopyasının etiketinden okunur
        ham = (M / "R3_tes_mineral/TES_mineral_maps_4_ppd.isis").open("rb").read(65536)
        etk = ham.decode("ascii", errors="replace")
        m_end = re.search(r"(?im)^\s*END\s*$", etk)
        etk = etk[: m_end.end()] if m_end else etk[:1024]
        OKUMA["R3_isis_etiket"] = etk
        maxlat = pds(etk, "MAXIMUM_LATITUDE", "MaximumLatitude")
        yon = (pds(etk, "POSITIVE_LONGITUDE_DIRECTION", "LongitudeDirection") or "").upper()
        assert maxlat and yon, "ISIS etiketinde MAXIMUM_LATITUDE / LONGITUDE_DIRECTION yok (etiket D2_okuma_mars.json'da)"
        dogu = "EAST" in yon
        bati = pds(etk, "WESTERNMOST_LONGITUDE", "MinimumLongitude" if dogu else "MaximumLongitude")
        assert bati, "ISIS etiketinde batı sınır boylamı yok"
        g = {"maxlat": sayi(maxlat), "bati": sayi(bati), "dogu": dogu, "ppd": 1440 / 360}
        coz = pds(etk, "MAP_RESOLUTION", "Scale")
        if coz:
            assert abs(sayi(coz) - 4) < 1e-6, f"etiket çözünürlüğü {coz}"
        OKUMA["R3_geometri"] = g

        def lonlat(r, c):
            lat = g["maxlat"] - (r + 0.5) / g["ppd"]
            lon = g["bati"] + (c + 0.5) / g["ppd"] if g["dogu"] else -(g["bati"] - (c + 0.5) / g["ppd"])
            return lat, (lon + 180) % 360 - 180

        def t_hem():
            b8 = a[7].astype(float)
            OKUMA["R3_bant8_-1_sayisi"] = int((b8 == -1).sum())
            b8[(b8 == -1) | ~np.isfinite(b8)] = np.nan
            lat_sat = g["maxlat"] - (np.arange(720) + 0.5) / 4
            b8[np.abs(lat_sat) > 30, :] = np.nan
            r, c = np.unravel_index(np.nanargmax(b8), b8.shape)
            e, b = lonlat(r, c)
            u = float(gc_km(e, b, *N["opportunity"][:2]))
            return f"bant 8 en buyuk {b8[r, c]:.3f} @ ({e:.2f}, {b:.2f}); Opportunity'ye {u:.0f} km", u <= 500
        dene("R3", "bant 8 (hematit) ekvator maksimumu", "opportunity", "Sinus Meridiani",
             "|enlem| <= 30° içindeki en buyuk deger Opportunity inis noktasina <= 500 km (proje karari)",
             "belge 17.3 D2; Bandfield 2002", t_hem)

        kaydet("R3", "bant 3/5/6 sirasi (acik kontrol)", "", "3 plajiyoklaz, 5 dusuk-Ca piroksen, 6 yuksek-Ca piroksen",
               "belge D11 ile urun sayfasi listesi ayni", "indirme_raporu3 R3 (ASU sayfası, 13 Kasım 17:40; Bandfield 2002)",
               "sayfadaki 13 bilesen sirasi belgeyle ayni (veri testi degil, belge karsilastirmasi)", "PASS")

    bolum("R3", r3)

    # ------------------------------------------------ R4 TI <-> albedo
    alb = {}

    def r4():
        p = M / "R4_tes_ti_albedo/ti_nmap2003.bin"
        bas = p.open("rb").read(7200).decode("ascii", errors="replace")
        OKUMA["R4_ti_baslik"] = bas.strip()[:1500]
        assert "west" in bas.lower() and "-90" in bas, "TI başlığında 'West' / '-90' yok"
        assert p.stat().st_size == 7200 + 7200 * 3600 * 2
        ti = np.fromfile(p, dtype=">i2", offset=7200).reshape(3600, 7200)
        gec = ti != 0
        OKUMA["R4_ti"] = {"veri_yok_0": int((~gec).sum()), "min": int(ti[gec].min()), "max": int(ti[gec].max())}
        dene("R4", "TI aralik (big-endian dogrulama)", "", "24-800", "gecerli degerler 24-800 icinde",
             "TI dosya başlığı (indirme_raporu3 R4)",
             lambda: (f"min {OKUMA['R4_ti']['min']}, max {OKUMA['R4_ti']['max']}, veri-yok {OKUMA['R4_ti']['veri_yok_0']}",
                      OKUMA['R4_ti']['min'] >= 24 and OKUMA['R4_ti']['max'] <= 800))
        a, lat, lon = albedo_oku()
        alb.update(a=a, lat=lat, lon=lon)

        def t_kor():
            LA, LO = np.meshgrid(lat, lon, indexing="ij")
            r = np.floor((LA + 90) * 20).astype(int)               # ilk satır güney (-90)
            c = np.floor(((-LO) % 360) * 20).astype(int) % 7200    # boylam batıya 0-360
            t = ti[r, c].astype(float)
            s = (np.abs(LA) <= 60) & np.isfinite(a) & (t != 0)
            pe, sp = kor(a[s], t[s])
            return f"Pearson {pe:.3f}, Spearman {sp:.3f} ({int(s.sum())} piksel)", pe < 0 and sp < 0
        dene("R4", "TI-albedo korelasyonu", "", "negatif (tozlu = parlak = dusuk TI)",
             "|enlem| <= 60° (proje karari: kutup buzu disarida), Pearson ve Spearman < 0", "Putzig ve ark. 2005", t_kor)

    bolum("R4", r4)

    # ------------------------------------------------ R5 DCI <-> albedo
    def r5():
        p = M / "R5_tes_dci/dci_lo_ice_dust_16ppd_shifted.pds"
        etk = p.open("rb").read(1024).decode("ascii", errors="replace")
        OKUMA["R5_etiket"] = etk.strip()
        sat, sut = int(sayi(pds(etk, "LINES"))), int(sayi(pds(etk, "LINE_SAMPLES")))
        tip, bit = (pds(etk, "SAMPLE_TYPE") or "").upper(), int(sayi(pds(etk, "SAMPLE_BITS")))
        m = re.search(r"\^IMAGE\s*=\s*(\d+)\s*(<BYTES>)?", etk)
        rb = pds(etk, "RECORD_BYTES")
        ofs = (int(m.group(1)) - 1) if (m and m.group(2)) else (int(m.group(1)) - 1) * int(sayi(rb))
        bo = "<" if "PC_REAL" in tip else ">"
        maxlat = pds(etk, "MAXIMUM_LATITUDE")
        bati = pds(etk, "WESTERNMOST_LONGITUDE")
        g = {"maxlat": sayi(maxlat) if maxlat else 90.0, "bati": sayi(bati) if bati else -180.0,
             "kaynak": "etiket" if (maxlat and bati) else "etikette yok -> indirme_raporu3 R5 (boylam -180'den doğuya)"}
        OKUMA["R5"] = {"satir": sat, "sutun": sut, "tip": tip, "bit": bit, "ofset_bayt": ofs, "bayt_sirasi": bo,
                       "yon": pds(etk, "POSITIVE_LONGITUDE_DIRECTION"), "missing": pds(etk, "MISSING_CONSTANT", "MISSING"), **g}
        assert (sat, sut, bit, ofs, bo) == (2880, 5760, 32, 1024, "<"), OKUMA["R5"]
        assert (OKUMA["R5"]["yon"] or "EAST").upper() == "EAST", OKUMA["R5"]
        d = np.fromfile(p, dtype="<f4", offset=1024).reshape(2880, 5760).astype(float)
        yok = d == np.float32(0.85)
        OKUMA["R5"]["0.85_sayisi"] = int(yok.sum())
        d[yok | ~np.isfinite(d)] = np.nan
        if not alb:
            alb.update(zip(("a", "lat", "lon"), albedo_oku()))

        def t_kor():
            LA, LO = np.meshgrid(alb["lat"], alb["lon"], indexing="ij")
            r = np.floor((g["maxlat"] - LA) * 16).astype(int)
            c = np.floor(((LO - g["bati"]) % 360) * 16).astype(int) % 5760
            v = d[r, c]
            s = (np.abs(LA) <= 60) & np.isfinite(alb["a"]) & np.isfinite(v)
            pe, sp = kor(alb["a"][s], v[s])
            return f"Pearson {pe:.3f}, Spearman {sp:.3f} ({int(s.sum())} piksel); 0,85 sayisi {int(yok.sum())}", pe < 0 and sp < 0
        dene("R5", "DCI-albedo korelasyonu", "", "negatif", "|enlem| <= 60° (R4 ile ayni), Pearson ve Spearman < 0",
             "Ruff & Christensen 2002", t_kor)

    bolum("R5", r5)

    # ------------------------------------------------ R6 Odyssey GRS H2O (PDS4 alan konumlarıyla)
    def r6():
        xml = (M / "R6_odyssey_grs/pds4_etiket/h2o_sr_5x5.xml").read_text(encoding="utf-8", errors="replace")
        rl = int(re.search(r"<record_length[^>]*>(\d+)<", xml).group(1))
        nrec = int(re.search(r"<records>(\d+)<", xml).group(1))
        alan = []
        for blok in re.findall(r"<Field_Character>(.*?)</Field_Character>", xml, re.S):
            alan.append((re.search(r"<name>(.*?)</name>", blok).group(1).strip(),
                         int(re.search(r"<field_location[^>]*>(\d+)<", blok).group(1)),
                         int(re.search(r"<field_length[^>]*>(\d+)<", blok).group(1))))
        OKUMA["R6"] = {"record_length": rl, "records": nrec, "alanlar": alan}
        ham = (M / "R6_odyssey_grs/h2o_sr_5x5.tab").read_bytes()
        assert len(ham) == rl * nrec, f"boyut {len(ham)} != {rl} x {nrec}"
        sat = [ham[i * rl:(i + 1) * rl] for i in range(nrec)]
        T = np.array([[float(s[l - 1:l - 1 + n]) for (_, l, n) in alan] for s in sat])
        lat, v = T[:, 0], T[:, 2]

        def t_h2o():
            dis = np.abs(lat) > 57.5
            yok = np.isclose(v, 9999.999)
            return f"{alan[2][0]}: 57,5° disi {int(dis.sum())} hucre, hepsi veri-yok: {bool(yok[dis].all())}; " \
                   f"ici {int((~dis).sum())} hucre, gecerli {int((~yok[~dis]).sum())}", \
                bool(yok[dis].all()) and bool((~yok[~dis]).any())
        dene("R6", "H2O kapsami", "", "yalniz 57,5°G-57,5°K'de deger; disi 9999,999",
             "|enlem| > 57,5 olan butun hucreler 9999,999", "belge 10.3 / rapor R6", t_h2o)

    bolum("R6", r6)

    # ------------------------------------------------ R7 SWIM enlem profili
    def r7():
        from pyproj import CRS, Transformer
        for ad in ("SWIM4MIM_Ci_0_1", "SWIM4MIM_Ci_1_5", "SWIM4MIM_Ci_5"):
            def t_sw(ad=ad):
                with rasterio.open(M / "R7_swim" / f"{ad}.tif") as d:
                    OKUMA[f"R7_{ad}"] = {"boyut": [d.width, d.height], "dtype": d.dtypes[0], "nodata": d.nodata,
                                         "crs": str(d.crs), "transform": list(d.transform)[:6]}
                    assert (d.width, d.height) == (7200, 2400), OKUMA[f"R7_{ad}"]
                    a = d.read(1).astype(float)
                    nd = d.nodata
                    xs, ys = d.transform * (np.full(d.height, d.width / 2), np.arange(d.height) + 0.5)
                    crs = CRS.from_wkt(d.crs.to_wkt())
                    lat = ys if crs.is_geographic else \
                        Transformer.from_crs(crs, crs.geodetic_crs, always_xy=True).transform(xs, ys)[1]
                lat = np.asarray(lat)
                yok = (a == -30) | ~np.isfinite(a) | ((a == nd) if nd is not None else False)
                OKUMA[f"R7_{ad}"].update({"-30_sayisi": int((a == -30).sum()), "enlem_araligi": [float(lat.min()), float(lat.max())]})
                a[yok] = np.nan
                aralik = np.nanmin(a) >= -1 and np.nanmax(a) <= 1
                k = np.nanmean(a[(lat >= 40) & (lat <= 60)])
                q = np.nanmean(a[np.abs(lat) <= 10])
                return f"40-60°K ortalama {k:.3f}, |enlem|<=10° ortalama {q:.3f}; gecerli aralik [{np.nanmin(a):.2f}, {np.nanmax(a):.2f}]", \
                    bool(aralik and k > q)
            dene("R7", f"enlem profili {ad}", "", "kuzey orta-yuksek > ekvator; -1..+1",
                 "40-60°K ortalamasi > |enlem| <= 10° ortalamasi (proje karari); -30 = veri yok", "Morgan ve ark. 2025", t_sw)

    bolum("R7", r7)

    # ------------------------------------------------ kayıt
    eski = []
    if D2CSV.exists():
        with open(D2CSV, encoding="utf-8", newline="") as f:
            eski = [r for r in csv.DictReader(f) if r["govde"] != "mars"]
    with open(D2CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=ALANLAR)
        w.writeheader()
        w.writerows(eski + SATIRLAR)
    (KAYIT / "D2_okuma_mars.json").write_text(json.dumps(OKUMA, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    say = {}
    for s in SATIRLAR:
        say[s["durum"]] = say.get(s["durum"], 0) + 1
    with open(KAYIT / "13_kasim_log.txt", "a", encoding="utf-8") as f:
        f.write(f"{datetime.now():%Y-%m-%d %H:%M:%S}  D2 mars: {say}, {round((time.time() - t0) / 60, 1)} dk\n")
    print("=" * 60)
    print(f"D2 Mars: {say} | {round((time.time() - t0) / 60, 1)} dk | kayit/D2.csv, kayit/D2_okuma_mars.json")
    return say
'''
compile(kod, "d2_mars.py", "exec")
dosya = KOK / "kod" / "d2_mars.py"
dosya.write_text(kod, encoding="utf-8")
print("Yazıldı:", dosya, "\n")
g = runpy.run_path(str(dosya), run_name="d2_mars")
_ = g["main"]()

Yazıldı: C:\Users\USER\orbitrary\kod\d2_mars.py 

[PASS] R1 kuresel min / max : ölçülen en kucuk -8201 m (np.float64(-32.85), np.float64(62.21)), en buyuk 21241 m (np.float64(17.34), np.float64(-133.45)); veri-yok 0 | beklenen yaklasik -8200 (Hellas) / 21229-21300 m (Olympus) (min -8200 +-100; max 21129-21400 (21229-21300 +-100; proje karari))
[PASS] R1 Jezero yuksekligi jezero: ölçülen -2566 m (satir 9159, sutun 32953) | beklenen yaklasik -2570 m (+-100 m (proje karari; 463 m piksel))
[PASS] R2 alan toplami : ölçülen 1311 kayit, 44 birim, alan / 4πR² = 0.999999 | beklenen 4πR² (R = 3396,19 km); 1311 kayit, 44 birim (+-%0,01)
[FAIL] R3 bolum calisamadi : ölçülen AssertionError: ISIS etiketinde MAXIMUM_LATITUDE / LONGITUDE_DIRECTION yok (etiket D2_okuma_mars.json'da) | beklenen  ()
[PASS] R4 TI aralik (big-endian dogrulama) : ölçülen min 24, max 800, veri-yok 2728379 | beklenen 24-800 (gecerli degerler 24-800 icinde)
[PASS] R4 TI-albedo korelasyonu : ölçülen Pearson -0.718, Spearman -0.

In [19]:
# Developed with assistance from Claude (Anthropic).
# D2 Mars: okunamayan iki etiketi kayıttan gösterir (R3 .isis etiketi, R5 .pds etiketi).
from pathlib import Path
import json

KOK = Path.home() / "orbitrary"
ok = json.loads((KOK / "kayit" / "D2_okuma_mars.json").read_text(encoding="utf-8"))
for ad in ("R3_isis_etiket", "R5_etiket"):
    print("=" * 30, ad, "=" * 30)
    print(ok.get(ad, "KAYITTA YOK"))

============================== R3_isis_etiket ==============================
PDS_VERSION = 3.0
RECORD_TYPE = FIXED_LENGTH
RECORD_BYTES = 512
FILE_RECORDS = 26328
LABEL_RECORDS = 2
FILE_STATE = CLEAN
^QUBE = 3
OBJECT = QUBE
    AXES = 3
    AXES_NAME = (SAMPLE,LINE,BAND)
    CORE_ITEMS = (1440,720,13)
    CORE_ITEM_BYTES = 4
    CORE_ITEM_TYPE = PC_REAL
    CORE_BASE = 0.0
    CORE_MULTIPLIER = 1.0
    CORE_NAME = RAW_DATA_NUMBERS
    CORE_UNIT = DN
    OBSERVATION_NOTE = "DV data product"
END_OBJECT = QUBE
END
============================== R5_etiket ==============================
PDS_VERSION = 3.0
RECORD_TYPE = FIXED_LENGTH
RECORD_BYTES = 512
FILE_RECORDS = 32403
LABEL_RECORDS = 2
FILE_STATE = CLEAN
^QUBE = 3
OBJECT = QUBE
    AXES = 3
    AXES_NAME = (BAND,SAMPLE,LINE)
    CORE_ITEMS = (1,5760,2880)
    CORE_ITEM_BYTES = 4
    CORE_ITEM_TYPE = PC_REAL
    CORE_BASE = 0.0
    CORE_MULTIPLIER = 1.0
    CORE_NAME = RAW_DATA_NUMBERS
    CORE_UNIT = DN
    OBSERVATION_NOTE = "DV data prod

In [20]:
# Developed with assistance from Claude (Anthropic).
# Adım D2 (Mars) yaması — R3 ve R5 etiket okuması PDS3 QUBE biçimine göre düzeltilir, d2_mars.py yeniden çalıştırılır.
from pathlib import Path
import runpy

KOK = Path.home() / "orbitrary"
dosya = KOK / "kod" / "d2_mars.py"
eski = dosya.read_text(encoding="utf-8")

R3_YENI = r'''    # ------------------------------------------------ R3 TES mineral
    def qube(etk):
        # PDS3 QUBE etiketi (TES .isis ve DCI .pds): eksenler, tip, veri başlangıcı, taban/çarpan
        ci = tuple(int(x) for x in re.search(r"CORE_ITEMS\s*=\s*\(([^)]*)\)", etk).group(1).split(","))
        eksen = re.search(r"AXES_NAME\s*=\s*\(([^)]*)\)", etk).group(1).replace(" ", "")
        return {"core_items": ci, "eksenler": eksen, "tip": pds(etk, "CORE_ITEM_TYPE"),
                "bayt": int(sayi(pds(etk, "CORE_ITEM_BYTES"))),
                "veri_baslangici": (int(sayi(pds(etk, "^QUBE"))) - 1) * int(sayi(pds(etk, "RECORD_BYTES"))),
                "taban": sayi(pds(etk, "CORE_BASE")), "carpan": sayi(pds(etk, "CORE_MULTIPLIER"))}

    def r3():
        hdr = (M / "R3_tes_mineral/TES_mineral_maps_4_ppd.envi.hdr").read_text(encoding="ascii", errors="replace")
        h = {k: int(sayi(pds(hdr, k))) for k in ("samples", "lines", "bands", "header offset", "data type", "byte order")}
        OKUMA["R3_hdr"] = h
        assert h == {"samples": 1440, "lines": 720, "bands": 13, "header offset": 0, "data type": 4, "byte order": 1}, h
        p = M / "R3_tes_mineral/TES_mineral_maps_4_ppd.envi"
        assert p.stat().st_size == 1440 * 720 * 13 * 4
        a = np.fromfile(p, dtype=">f4").reshape(13, 720, 1440)
        # aynı verinin .isis kopyası kendi etiketiyle okunur: iki dosya aynı sayıları vermeli (bayt sırası çapraz kontrolü)
        pi = M / "R3_tes_mineral/TES_mineral_maps_4_ppd.isis"
        etk = pi.open("rb").read(1024).decode("ascii", errors="replace")
        q = qube(etk)
        OKUMA["R3_isis_etiket"], OKUMA["R3_isis"] = etk.strip(), q
        assert (q["core_items"], q["eksenler"], q["tip"], q["bayt"], q["veri_baslangici"], q["taban"], q["carpan"]) == \
            ((1440, 720, 13), "SAMPLE,LINE,BAND", "PC_REAL", 4, 1024, 0.0, 1.0), q
        assert pi.stat().st_size == 1024 + 1440 * 720 * 13 * 4
        b = np.fromfile(pi, dtype="<f4", offset=1024).reshape(13, 720, 1440)
        dene("R3", "envi (>f4) ile isis (<f4) ayni veri", "", "fark 0", "butun degerler esit",
             "iki dosyanın kendi etiketleri (.envi.hdr byte order 1; .isis PC_REAL)",
             lambda: (f"{int(np.sum(~((a == b) | (np.isnan(a) & np.isnan(b)))))} farkli deger", bool(np.array_equal(a, b, equal_nan=True))))
        # geometri: .envi.hdr'de de .isis etiketinde de yok -> ASU'nun aynı ürün için verdiği tanım
        g = {"maxlat": 90.0, "bati": -180.0, "dogu": True, "ppd": 1440 / 360,
             "kaynak": "dosyalarda yok; ASU Mars Global Data Sets, TES Low-Calcium Pyroxene (Bandfield 2002, 4 ppd, 1440x720): "
                       "Simple cylindrical, -180E to 180E, 90N to -90N, 'ocentric"}
        OKUMA["R3_geometri"] = g

        def lonlat(r, c):
            lat = g["maxlat"] - (r + 0.5) / g["ppd"]
            lon = g["bati"] + (c + 0.5) / g["ppd"]
            return lat, (lon + 180) % 360 - 180

        def t_hem():
            b8 = a[7].astype(float)
            OKUMA["R3_bant8_-1_sayisi"] = int((b8 == -1).sum())
            b8[(b8 == -1) | ~np.isfinite(b8)] = np.nan
            lat_sat = g["maxlat"] - (np.arange(720) + 0.5) / 4
            b8[np.abs(lat_sat) > 30, :] = np.nan
            r, c = np.unravel_index(np.nanargmax(b8), b8.shape)
            e, b = lonlat(r, c)
            u = float(gc_km(e, b, *N["opportunity"][:2]))
            return f"bant 8 en buyuk {b8[r, c]:.3f} @ ({e:.2f}, {b:.2f}); Opportunity'ye {u:.0f} km", u <= 500
        dene("R3", "bant 8 (hematit) ekvator maksimumu", "opportunity", "Sinus Meridiani",
             "|enlem| <= 30° içindeki en buyuk deger Opportunity inis noktasina <= 500 km (proje karari)",
             "belge 17.3 D2; Bandfield 2002", t_hem)

        kaydet("R3", "bant 3/5/6 sirasi (acik kontrol)", "", "3 plajiyoklaz, 5 dusuk-Ca piroksen, 6 yuksek-Ca piroksen",
               "belge D11 ile urun sayfasi listesi ayni", "indirme_raporu3 R3 (ASU sayfası, 13 Kasım 17:40; Bandfield 2002)",
               "sayfadaki 13 bilesen sirasi belgeyle ayni (veri testi degil, belge karsilastirmasi)", "PASS")

    bolum("R3", r3)

'''

R5_YENI = r'''    # ------------------------------------------------ R5 DCI <-> albedo
    def r5():
        p = M / "R5_tes_dci/dci_lo_ice_dust_16ppd_shifted.pds"
        etk = p.open("rb").read(1024).decode("ascii", errors="replace")
        q = qube(etk)
        g = {"maxlat": 90.0, "bati": -180.0,
             "kaynak": "etikette yok; indirme_raporu3 R5 (ikincil kayıt mars.asu.edu/data/tes_ruffdust: simple cylindrical, -180->180 doğu, 90->-90)"}
        OKUMA["R5_etiket"], OKUMA["R5"] = etk.strip(), {**q, **g}
        assert (q["core_items"], q["eksenler"], q["tip"], q["bayt"], q["veri_baslangici"], q["taban"], q["carpan"]) == \
            ((1, 5760, 2880), "BAND,SAMPLE,LINE", "PC_REAL", 4, 1024, 0.0, 1.0), q
        assert p.stat().st_size == 1024 + 5760 * 2880 * 4
        d = np.fromfile(p, dtype="<f4", offset=1024).reshape(2880, 5760).astype(float)
        yok = d == np.float32(0.85)
        OKUMA["R5"]["0.85_sayisi"] = int(yok.sum())
        d[yok | ~np.isfinite(d)] = np.nan
        if not alb:
            alb.update(zip(("a", "lat", "lon"), albedo_oku()))

        def t_kor():
            LA, LO = np.meshgrid(alb["lat"], alb["lon"], indexing="ij")
            r = np.floor((g["maxlat"] - LA) * 16).astype(int)
            c = np.floor(((LO - g["bati"]) % 360) * 16).astype(int) % 5760
            v = d[r, c]
            s = (np.abs(LA) <= 60) & np.isfinite(alb["a"]) & np.isfinite(v)
            pe, sp = kor(alb["a"][s], v[s])
            return f"Pearson {pe:.3f}, Spearman {sp:.3f} ({int(s.sum())} piksel); 0,85 sayisi {int(yok.sum())}", pe < 0 and sp < 0
        dene("R5", "DCI-albedo korelasyonu", "", "negatif", "|enlem| <= 60° (R4 ile ayni), Pearson ve Spearman < 0",
             "Ruff & Christensen 2002", t_kor)

    bolum("R5", r5)

'''

def degistir(metin, bas, son, yeni):
    assert metin.count(bas) == 1 and metin.count(son) == 1, f"işaret bulunamadı: {bas[:60]!r}"
    i, j = metin.index(bas), metin.index(son)
    assert i < j
    return metin[:i] + yeni + metin[j:]

yeni = degistir(eski, "    # ------------------------------------------------ R3 TES mineral\n",
                "    # ------------------------------------------------ R4 TI <-> albedo\n", R3_YENI)
yeni = degistir(yeni, "    # ------------------------------------------------ R5 DCI <-> albedo\n",
                "    # ------------------------------------------------ R6 Odyssey GRS H2O", R5_YENI)
compile(yeni, "d2_mars.py", "exec")
dosya.write_text(yeni, encoding="utf-8")
print("Yamandı:", dosya, "\n")
g = runpy.run_path(str(dosya), run_name="d2_mars")
_ = g["main"]()

Yamandı: C:\Users\USER\orbitrary\kod\d2_mars.py 

[PASS] R1 kuresel min / max : ölçülen en kucuk -8201 m (np.float64(-32.85), np.float64(62.21)), en buyuk 21241 m (np.float64(17.34), np.float64(-133.45)); veri-yok 0 | beklenen yaklasik -8200 (Hellas) / 21229-21300 m (Olympus) (min -8200 +-100; max 21129-21400 (21229-21300 +-100; proje karari))
[PASS] R1 Jezero yuksekligi jezero: ölçülen -2566 m (satir 9159, sutun 32953) | beklenen yaklasik -2570 m (+-100 m (proje karari; 463 m piksel))
[PASS] R2 alan toplami : ölçülen 1311 kayit, 44 birim, alan / 4πR² = 0.999999 | beklenen 4πR² (R = 3396,19 km); 1311 kayit, 44 birim (+-%0,01)
[PASS] R3 envi (>f4) ile isis (<f4) ayni veri : ölçülen 0 farkli deger | beklenen fark 0 (butun degerler esit)
[PASS] R3 bant 8 (hematit) ekvator maksimumu opportunity: ölçülen bant 8 en buyuk 0.224 @ (-2.62, -3.62); Opportunity'ye 120 km | beklenen Sinus Meridiani (|enlem| <= 30° içindeki en buyuk deger Opportunity inis noktasina <= 500 km (proje karari))
[PASS] 

In [21]:
# Developed with assistance from Claude (Anthropic).
# D3 Dünya hazırlığı: veri\dunya ağacı (set başına özet), d2_dunya.py metni, kayıt dosyalarının sütunları, boş disk.
# Çıktı kayit\d3_hazirlik_dunya.txt'ye de yazılır. Hiçbir dosyayı değiştirmez.
from pathlib import Path
import csv, shutil

KOK = Path.home() / "orbitrary"
D = KOK / "veri" / "dunya"
out = []

out.append("=" * 25 + " VERI\\DUNYA " + "=" * 25)
for s in sorted(p for p in D.iterdir() if p.is_dir()):
    dosyalar = sorted(f for f in s.rglob("*") if f.is_file())
    toplam = sum(f.stat().st_size for f in dosyalar)
    out.append(f"\n## {s.name}: {len(dosyalar)} dosya, {toplam/1e9:.2f} GB")
    uzantilar = {}
    for f in dosyalar:
        uzantilar[f.suffix.lower()] = uzantilar.get(f.suffix.lower(), 0) + 1
    out.append(f"   uzantılar: {uzantilar}")
    goster = dosyalar if len(dosyalar) <= 30 else dosyalar[:4] + dosyalar[-2:]
    for f in goster:
        out.append(f"   {f.relative_to(D)}  {f.stat().st_size:,}")
    if len(dosyalar) > 30:
        out.append(f"   ... ({len(dosyalar) - 6} dosya daha)")

out.append("\n" + "=" * 25 + " KOD\\D2_DUNYA.PY " + "=" * 25)
out.append((KOK / "kod" / "d2_dunya.py").read_text(encoding="utf-8"))

out.append("\n" + "=" * 25 + " KAYIT SÜTUNLARI " + "=" * 25)
for yol in (KOK / "kayit" / "D1_dosyalar.csv", KOK / "kayit" / "indirme.csv", KOK / "kod" / "indirme_listesi.csv"):
    with open(yol, encoding="utf-8", newline="") as f:
        r = list(csv.reader(f))
    out.append(f"\n--- {yol.relative_to(KOK)}: {len(r) - 1} satır")
    out.append("   sütunlar: " + ", ".join(r[0]))
    for satir in r[1:3]:
        out.append("   örnek: " + " | ".join(x[:80] for x in satir))

bos = shutil.disk_usage(KOK).free / 1e9
out.append(f"\nBoş disk: {bos:.1f} GB")

metin = "\n".join(out)
(KOK / "kayit" / "d3_hazirlik_dunya.txt").write_text(metin, encoding="utf-8")
print(metin)

========================= VERI\DUNYA =========================

## E10_soilgrids: 4 dosya, 0.39 GB
   uzantılar: {'.txt': 2, '.tif': 2}
   E10_soilgrids\clay\checksum.sha256.txt  558
   E10_soilgrids\clay\clay_0-5cm_mean_1000.tif  191,530,251
   E10_soilgrids\silt\checksum.sha256.txt  558
   E10_soilgrids\silt\silt_0-5cm_mean_1000.tif  194,638,176

## E1_etopo: 288 dosya, 5.23 GB
   uzantılar: {'.tif': 288}
   E1_etopo\ETOPO_2022_v1_15s_N00E000_surface.tif  16,434,942
   E1_etopo\ETOPO_2022_v1_15s_N00E015_surface.tif  34,120,034
   E1_etopo\ETOPO_2022_v1_15s_N00E030_surface.tif  25,179,967
   E1_etopo\ETOPO_2022_v1_15s_N00E045_surface.tif  10,866,855
   E1_etopo\ETOPO_2022_v1_15s_S75W165_surface.tif  24,248,171
   E1_etopo\ETOPO_2022_v1_15s_S75W180_surface.tif  23,250,977
   ... (282 dosya daha)

## E2_mcd12c1: 1 dosya, 1.24 GB
   uzantılar: {'.hdf': 1}
   E2_mcd12c1\MCD12C1.A2023001.061.2024251212901.hdf  1,244,259,901

## E3_glim: 47 dosya, 2.40 GB
   uzantılar: {'.gdbindexes': 7, '.

In [22]:
# Developed with assistance from Claude (Anthropic).
# Adım D3 (Dünya) — kod\d3_dunya.py yazılır ve çalıştırılır (belge 17.3 D3, 10.4).
from pathlib import Path
import runpy

KOK = Path.home() / "orbitrary"
kod = r'''# Developed with assistance from Claude (Anthropic).
# d3_dunya.py — Orbitrary v1.2, belge 17.3 D3 + 10.4: Dünya setleri E1–E10 temiz\dunya\<kod>_<ad>\ altına standart biçimde yazılır.
# Raster: fiziksel birim, float32, veri yok = NaN (GeoTIFF; büyük ızgaralarda bant başına .npy + aynı adlı .json).
# Vektör: GeoPackage (birim, alan_km2). Her klasöre meta.json.
# D3 testi (belge 17.3): 1000 rastgele pikselde ham dosyadan yeniden hesaplanan değerle fark 0; NaN sayısı = veri-yok sayısı.
# Okuma kuralları d2_dunya.py ile aynıdır (D2'de geçti). Ham veri (veri\) yalnız okunur.
import csv, json, math, re, shutil, sys, time
from datetime import datetime
from pathlib import Path

import numpy as np

KOK = Path.home() / "orbitrary"
VERI, KAYIT, TEMIZ = KOK / "veri", KOK / "kayit", KOK / "temiz"
D3CSV = KAYIT / "D3.csv"
TOHUM = 20261113                      # belge V2 tohumu
CMG_GT = [-180.0, 0.05, 0.0, 90.0, 0.0, -0.05]   # 0,05° CMG: sol üst (-180, 90); D2'de cmg_rc ile doğrulandı

# meta.json'un sabit alanları (kaynak: belge 10.1 ve indirme_raporu3 blokları / Bölüm 4 lisans tablosu)
STATIK = {
    "E1": {"ad": "ETOPO 2022 15\" surface", "doi": "10.25921/fd45-gt74", "surum": "ETOPO 2022 v1, 15 yay-saniye, surface",
           "lisans": "CC0 1.0 (data.gov kaydı; rapor Bölüm 4)", "birim": "m (EGM2008)"},
    "E2": {"ad": "MODIS MCD12C1", "doi": "10.5067/MODIS/MCD12C1.061", "surum": "v061, 2023",
           "lisans": "EOSDIS Data Use Policy (rapor Bölüm 4)", "birim": "% (sınıf payı)"},
    "E3": {"ad": "GLiM", "doi": "10.1029/2012GC004370 (makale)", "surum": "LiMW_GIS 2015.gdb, katman GLiM_export",
           "lisans": "sayfada yok — 'açık' (belge CC BY diyor, doğrulanamadı; rapor Bölüm 4)", "birim": "sınıf (xx), km²"},
    "E4": {"ad": "ASTER GED AG5KMMOH", "doi": "açık (rapor E4 künyesi)", "surum": "v041, 2015, 12 ay",
           "lisans": "EOSDIS Data Use Policy (rapor Bölüm 4)", "birim": "yayınım (birimsiz), NDVI (birimsiz)"},
    "E5": {"ad": "MODIS MYD11C3", "doi": "10.5067/MODIS/MYD11C3.061", "surum": "v061 (Aqua), 2023, 12 ay",
           "lisans": "EOSDIS Data Use Policy (rapor Bölüm 4)", "birim": "K"},
    "E6": {"ad": "MODIS MCD43C3", "doi": "10.5067/MODIS/MCD43C3.061", "surum": "v061, 2023'ün her ayının 15'i",
           "lisans": "EOSDIS Data Use Policy (rapor Bölüm 4)", "birim": "albedo (birimsiz)"},
    "E7": {"ad": "Permafrost PERPROB kuzey (Obu ve ark.)", "doi": "10.1594/PANGAEA.888600", "surum": "PERPROB 5.0, 2000–2016",
           "lisans": "CC-BY-3.0 (rapor Bölüm 4)", "birim": "olasılık 0–1"},
    "E8": {"ad": "Permafrost PERPROB Antarktika (Obu ve ark.)", "doi": "10.1594/PANGAEA.902576", "surum": "PERPROB 5.0, 2000–2017",
           "lisans": "CC-BY-4.0 (rapor Bölüm 4)", "birim": "olasılık 0–1"},
    "E9": {"ad": "Global Aridity Index v3.1 (yıllık)", "doi": "10.6084/m9.figshare.7504448", "surum": "v3.1, 1970–2000 ortalaması",
           "lisans": "CC BY 4.0 (rapor Bölüm 4; sayfanın eski metni 'non-commercial use' da diyor)", "birim": "AI (birimsiz)"},
    "E10": {"ad": "SoilGrids 2.0 kil + silt 0–5 cm 1000 m", "doi": "silt: 10.17027/isric-soilgrids/08a4990c-ca15-4871-8d12-ea21dae6b354; kil: açık",
            "surum": "SoilGrids 2.0 (latest, data_aggregated 1000m)", "lisans": "CC BY 4.0 (rapor Bölüm 4)", "birim": "‰ (g/kg)"},
}

SATIRLAR, _TUT = [], []


def csv_oku(yol):
    with open(yol, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def sinav(set_, test, olculen, ok, d3):
    durum = "PASS" if ok else "FAIL"
    d3.append({"test": test, "olculen": olculen, "durum": durum})
    SATIRLAR.append({"govde": "dunya", "set": set_, "test": test, "olculen": olculen, "durum": durum,
                     "zaman": datetime.now().isoformat(timespec="seconds")})
    print(f"[{durum}] {set_} {test}: {olculen}", flush=True)


def klasor(kod, ad):
    p = TEMIZ / "dunya" / f"{kod}_{ad}"
    if p.exists():
        shutil.rmtree(p)                  # temiz\ yeniden üretilebilir; ham veri\ hiç silinmez
    p.mkdir(parents=True)
    return p


def meta_yaz(p, kod, ek, d3, D1, IND, D2):
    ind = [r for r in IND if r["set"] == kod and r["durum"] != "FAIL"]
    adres = sorted({r["adres"] for r in ind if r["adres"]})
    m = {"set": kod, **STATIK[kod],
         "kaynak_adresleri": adres if len(adres) <= 5 else adres[:3] + [f"... toplam {len(adres)} adres (kayit/indirme.csv)"],
         "indirme_tarihi": [min(r["zaman"] for r in ind), max(r["zaman"] for r in ind)] if ind else None,
         "ham_sha256": {r["hedef"]: r["sha256"] for r in D1 if r["set"] == kod},
         "gercek_yer_testleri_D2": [{k: r[k] for k in ("test", "nokta", "beklenen", "tolerans_kurali", "beklenen_kaynagi", "olculen", "durum")}
                                    for r in D2 if r["govde"] == "dunya" and r["set"] == kod],
         "D3_testleri": d3, **ek, "uretim_zamani": datetime.now().isoformat(timespec="seconds"),
         "uretim_kodu": "kod/d3_dunya.py"}
    (p / "meta.json").write_text(json.dumps(m, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


def esit(bek, tem):
    # fark 0: iki değer birebir aynı ya da ikisi de NaN
    return (bek == tem) | (np.isnan(bek) & np.isnan(tem))


# ---------------------------------------------------------------- tek bantlı GeoTIFF -> float32 GeoTIFF (NaN)
def tif_cevir(kaynak, hedef, olcek=1.0, ofset=0.0, beklenen_nd="dosyadan", satir=512):
    import rasterio
    from rasterio.windows import Window
    with rasterio.open(kaynak) as s:
        assert s.count == 1, f"{kaynak.name}: {s.count} bant"
        assert s.scales[0] == 1.0 and s.offsets[0] == 0.0, f"{kaynak.name}: dosyada ölçek/ofset {s.scales[0]}/{s.offsets[0]}"
        nd = s.nodata
        if beklenen_nd != "dosyadan":
            assert nd == beklenen_nd, f"{kaynak.name}: nodata {nd} != {beklenen_nd}"
        prof = s.profile.copy()
        prof.update(driver="GTiff", dtype="float32", nodata=np.nan, count=1, compress="deflate", predictor=3,
                    tiled=True, blockxsize=512, blockysize=512, BIGTIFF="IF_SAFER")
        n_yok, n_sonsuz, vmin, vmax = 0, 0, np.inf, -np.inf
        with rasterio.open(hedef, "w", **prof) as t:
            for r0 in range(0, s.height, satir):
                w = Window(0, r0, s.width, min(satir, s.height - r0))
                a = s.read(1, window=w)
                yok = np.zeros(a.shape, bool)
                if nd is not None:
                    yok |= np.isnan(a) if (isinstance(nd, float) and math.isnan(nd)) else (a == nd)
                if a.dtype.kind == "f":
                    sonsuz = ~np.isfinite(a) & ~yok
                    n_sonsuz += int(sonsuz.sum())
                    yok |= sonsuz
                v = (a.astype(np.float64) * olcek + ofset).astype(np.float32)
                v[yok] = np.nan
                n_yok += int(yok.sum())
                if (~yok).any():
                    vmin, vmax = min(vmin, float(np.nanmin(v))), max(vmax, float(np.nanmax(v)))
                t.write(v, 1, window=w)
        bilgi = {"ham_dtype": s.dtypes[0], "ham_nodata": nd, "boyut": [s.width, s.height], "crs": str(s.crs),
                 "transform": list(s.transform)[:6], "piksel": list(s.res), "olcek": olcek, "ofset": ofset,
                 "veri_yok_sayisi": n_yok, "sonlu_olmayan_ham": n_sonsuz, "min": vmin, "max": vmax}
    return bilgi


def tif_dogrula(kaynak, hedef, bilgi, rng, n):
    # bağımsız: ham dosyadan rastgele pikseller tek tek okunur, formül yeniden uygulanır; çıktı dosyası baştan okunur
    import rasterio
    from rasterio.windows import Window
    with rasterio.open(kaynak) as s, rasterio.open(hedef) as t:
        assert (s.width, s.height, s.transform, s.crs) == (t.width, t.height, t.transform, t.crs), f"{hedef.name}: geometri farklı"
        assert t.dtypes[0] == "float32"
        rr, cc = rng.integers(0, s.height, n), rng.integers(0, s.width, n)
        ham = np.array([s.read(1, window=Window(int(c), int(r), 1, 1))[0, 0] for r, c in zip(rr, cc)])
        tem = np.array([t.read(1, window=Window(int(c), int(r), 1, 1))[0, 0] for r, c in zip(rr, cc)], dtype=np.float32)
        nd = s.nodata
        bek = (ham.astype(np.float64) * bilgi["olcek"] + bilgi["ofset"]).astype(np.float32)
        if nd is not None:
            bek[np.isnan(ham) if (isinstance(nd, float) and math.isnan(nd)) else (ham == nd)] = np.nan
        if ham.dtype.kind == "f":
            bek[~np.isfinite(ham)] = np.nan
        n_fark = int((~esit(bek, tem)).sum())
        n_nan = 0
        for r0 in range(0, t.height, 512):
            n_nan += int(np.isnan(t.read(1, window=Window(0, r0, t.width, min(512, t.height - r0)))).sum())
    return n_fark, n_nan


def npy_yaz(p, ad, v, bilgi):
    assert v.dtype == np.float32
    np.save(p / f"{ad}.npy", v)
    (p / f"{ad}.json").write_text(json.dumps({**bilgi, "sekil": list(v.shape), "dtype": "float32", "veri_yok": "NaN"},
                                             ensure_ascii=False, indent=2, default=str), encoding="utf-8")


def npy_dogrula(p, ad, ham, formul, yok_maskesi, rng, n):
    # kaydedilen .npy diskten yeniden açılır; ham diziden bağımsız formülle karşılaştırılır
    t = np.load(p / f"{ad}.npy", mmap_mode="r")
    assert t.shape == ham.shape and t.dtype == np.float32
    rr, cc = rng.integers(0, ham.shape[0], n), rng.integers(0, ham.shape[1], n)
    h = ham[rr, cc]
    bek = formul(h).astype(np.float32)
    bek[yok_maskesi(h)] = np.nan
    n_fark = int((~esit(bek, np.asarray(t[rr, cc]))).sum())
    n_nan = int(np.isnan(t).sum())
    n_yok = int(yok_maskesi(ham).sum())
    return n_fark, n_nan, n_yok


def main(setler=None):
    t0 = time.time()
    from osgeo import gdal
    gdal.UseExceptions()
    import h5py

    D1 = csv_oku(KAYIT / "D1_dosyalar.csv")
    IND = csv_oku(KAYIT / "indirme.csv")
    D2 = csv_oku(KAYIT / "D2.csv")
    secili = set(setler) if setler else set(STATIK)
    rng = np.random.default_rng(TOHUM)
    calisan = []

    def bolum(kod, f):
        if kod not in secili:
            return
        calisan.append(kod)
        t1 = time.time()
        print(f"\n--- {kod} başladı {datetime.now():%H:%M:%S}", flush=True)
        try:
            f()
        except Exception as e:
            sinav(kod, "bolum calisamadi", f"{type(e).__name__}: {e}", False, [])
        print(f"--- {kod} bitti, {round((time.time() - t1) / 60, 1)} dk", flush=True)

    def alt(yol, ad):
        s = [x[0] for x in gdal.Open(str(yol)).GetSubDatasets() if x[0].endswith(":" + ad) or x[0].endswith(ad)]
        assert len(s) >= 1, f"{ad} alt veri seti yok"
        ds = gdal.Open(s[0])
        _TUT.append(ds)
        return ds

    def tarih(dosya):
        return re.search(r"\.?A(\d{7})", dosya.name).group(1)

    # ------------------------------------------------ E1 ETOPO: 288 karo -> 288 float32 GeoTIFF
    def e1():
        p = klasor("E1", "etopo")
        karolar = sorted((VERI / "dunya/E1_etopo").glob("ETOPO_2022_v1_15s_*_surface.tif"))
        assert len(karolar) == 288, len(karolar)
        d3, n_fark, n_ornek, nan_hata, toplam_yok = [], 0, 0, [], 0
        for i, k in enumerate(karolar):
            b = tif_cevir(k, p / k.name, 1.0, 0.0, beklenen_nd=-99999)
            assert b["ham_dtype"] == "float32"
            f, n_nan = tif_dogrula(k, p / k.name, b, rng, 4)
            n_fark, n_ornek, toplam_yok = n_fark + f, n_ornek + 4, toplam_yok + b["veri_yok_sayisi"]
            if n_nan != b["veri_yok_sayisi"]:
                nan_hata.append(k.name)
            if (i + 1) % 48 == 0:
                print(f"    {i + 1}/288 karo", flush=True)
        sinav("E1", "rastgele piksel fark 0", f"{n_ornek} piksel (288 karo x 4), {n_fark} farkli", n_fark == 0, d3)
        sinav("E1", "NaN sayisi = veri-yok sayisi", f"288 karo; tutmayan karo {len(nan_hata)} {nan_hata[:5]}; toplam veri-yok {toplam_yok}",
              not nan_hata, d3)
        meta_yaz(p, "E1", {"crs": b["crs"], "cozunurluk": "15 yay-saniye", "olcek": 1.0, "ofset": 0.0,
                           "veri_yok_kurali": "ham -99999 -> NaN", "dosyalar": "288 karo, adlar ham ile aynı (üst-sol köşe)",
                           "toplam_veri_yok": toplam_yok}, d3, D1, IND, D2)

    bolum("E1", e1)

    # ------------------------------------------------ E2 MCD12C1: 17 sınıf payı -> 17 .npy
    def e2():
        p = klasor("E2", "mcd12c1")
        lc = alt(VERI / "dunya/E2_mcd12c1/MCD12C1.A2023001.061.2024251212901.hdf", "Land_Cover_Type_1_Percent")
        md = lc.GetMetadata()
        assert (lc.RasterXSize, lc.RasterYSize, lc.RasterCount) == (7200, 3600, 17)
        fill = int(float(md["_FillValue"])) if "_FillValue" in md else 255
        fill_kaynak = "dosya _FillValue" if "_FillValue" in md else "dosyada yok -> belge 10.1 (255)"
        assert fill == 255, f"_FillValue {fill}"
        assert float(md.get("scale_factor", 1)) == 1.0 and float(md.get("add_offset", 0)) == 0.0, md
        d3, n_fark, n_nan_hata = [], 0, []
        for k in range(17):                                   # bant k+1 = sınıf k (D2'de doğrulandı)
            ham = lc.GetRasterBand(k + 1).ReadAsArray()
            v = ham.astype(np.float32)
            v[ham == fill] = np.nan
            ad = f"sinif_{k:02d}_yuzde"
            npy_yaz(p, ad, v, {"kaynak": "dunya/E2_mcd12c1/MCD12C1.A2023001.061.2024251212901.hdf",
                               "alt_veri_seti": "Land_Cover_Type_1_Percent", "bant": k + 1, "sinif": k,
                               "geotransform": CMG_GT, "gdal_geotransform": lc.GetGeoTransform(), "crs": "EPSG:4326 (CMG 0,05°)",
                               "birim": "%", "olcek": 1.0, "ofset": 0.0, "veri_yok_ham": fill})
            f, n_nan, n_yok = npy_dogrula(p, ad, ham, lambda h: h.astype(np.float64), lambda h: h == fill, rng, 1000)
            n_fark += f
            if n_nan != n_yok:
                n_nan_hata.append(k)
        sinav("E2", "rastgele piksel fark 0", f"17 sinif x 1000 piksel, {n_fark} farkli", n_fark == 0, d3)
        sinav("E2", "NaN sayisi = veri-yok sayisi", f"tutmayan sinif {n_nan_hata}", not n_nan_hata, d3)
        meta_yaz(p, "E2", {"crs": "EPSG:4326 (CMG 0,05°)", "geotransform": CMG_GT, "cozunurluk": "0,05°", "olcek": 1.0, "ofset": 0.0,
                           "veri_yok_kurali": f"ham {fill} -> NaN ({fill_kaynak})",
                           "dosyalar": "sinif_00_yuzde.npy ... sinif_16_yuzde.npy (bant k+1 = sınıf k)"}, d3, D1, IND, D2)

    bolum("E2", e2)

    # ------------------------------------------------ E3 GLiM: GDB -> GeoPackage (birim, alan_km2)
    def e3():
        import pyogrio
        p = klasor("E3", "glim")
        gdb = VERI / "dunya/E3_glim/acik/LiMW_GIS 2015.gdb"
        katman = [x[0] for x in pyogrio.list_layers(gdb)]
        assert "GLiM_export" in katman, katman
        info = pyogrio.read_info(gdb, layer="GLiM_export")
        alanlar = list(info["fields"])
        assert "xx" in alanlar, alanlar
        sutun = ["xx"] + (["Shape_Area"] if "Shape_Area" in alanlar else [])
        df = pyogrio.read_dataframe(gdb, layer="GLiM_export", columns=sutun)
        wkt = df.crs.to_wkt()
        assert "eckert_iv" in wkt.lower().replace(" ", "_"), "CRS Eckert IV değil (belge 10.1)"
        alan = df.geometry.area.to_numpy() / 1e6               # eş-alanlı Eckert IV, m² -> km²
        import geopandas as gpd
        out = gpd.GeoDataFrame({"birim": df["xx"].astype("string"), "alan_km2": alan}, geometry=df.geometry, crs=df.crs)
        gp = p / "glim.gpkg"
        pyogrio.write_dataframe(out, gp, layer="glim", driver="GPKG")
        d3 = []
        geri = pyogrio.read_dataframe(gp, layer="glim")
        i = rng.integers(0, len(df), 1000)
        b_ok = bool((geri["birim"].astype("string").iloc[i].fillna("<bos>").to_numpy()
                     == df["xx"].astype("string").iloc[i].fillna("<bos>").to_numpy()).all())
        a_geri = geri["alan_km2"].to_numpy()[i]
        a_yeni = geri.geometry.area.to_numpy()[i] / 1e6
        fark = float(np.max(np.abs(a_geri - a_yeni) / np.where(a_yeni > 0, a_yeni, 1)))
        sinav("E3", "rastgele 1000 poligon: birim ayni, alan_km2 yeniden hesapla ayni",
              f"{len(geri)} kayit (ham {len(df)}); birim esit {b_ok}; alan en buyuk goreli fark {fark:.2e}",
              len(geri) == len(df) and b_ok and fark <= 1e-12, d3)
        n_bos_birim, n_bos_geo = int(df["xx"].isna().sum()), int(df.geometry.isna().sum())
        sinav("E3", "bos birim / bos geometri sayisi (kayit)", f"bos birim {n_bos_birim}, bos geometri {n_bos_geo}", True, d3)
        sa = None
        if "Shape_Area" in df:
            sa = float(np.nanmax(np.abs(df["Shape_Area"].to_numpy() / 1e6 - alan) / np.where(alan > 0, alan, 1)))
        meta_yaz(p, "E3", {"crs": "World Eckert IV (ham CRS korundu)", "crs_wkt": wkt, "olcek": None, "ofset": None,
                           "veri_yok_kurali": "vektör; boş birim kayda yazılır, NaN yok",
                           "kayit_sayisi": len(df), "birim_sayisi": int(df["xx"].nunique()),
                           "alan_km2": "geometrinin Eckert IV (eş-alanlı) alanı / 1e6",
                           "Shape_Area_ile_en_buyuk_goreli_fark": sa, "dosyalar": "glim.gpkg (katman glim: birim, alan_km2)"},
                 d3, D1, IND, D2)

    bolum("E3", e3)

    # ------------------------------------------------ E4 ASTER GED: 12 ay x (5 yayınım bandı + NDVI) -> .npy
    def att(d, ad):
        for k, v in d.items():
            if k.strip().lower() == ad.lower():
                return float(np.ravel(v)[0]) if not isinstance(v, (bytes, str)) else float(v)
        return None

    def e4():
        p = klasor("E4", "aster_ged")
        dosyalar = sorted((VERI / "dunya/E4_aster_ged").glob("ASTER_GEDv4.1_A2015*.h5"))
        assert len(dosyalar) == 12
        d3, n_fark, nan_hata, ndvi_bilgi = [], 0, [], {}
        for dosya in dosyalar:
            tr = tarih(dosya)
            with h5py.File(dosya, "r") as f:
                em, nd = f["SDS/Emissivity"], f["SDS/NDVI"]
                ea, na = dict(em.attrs), dict(nd.attrs)
                olcek, ofset, n_olcek = att(ea, "Scale Factor"), att(ea, "Offset"), att(na, "Scale Factor")
                assert olcek == 0.002 and ofset is not None and abs(ofset - 0.49) < 1e-9 and n_olcek == 0.001, (olcek, ofset, n_olcek)
                eksen = [i for i, s in enumerate(em.shape) if s == 5]
                assert len(eksen) == 1, em.shape
                E = np.moveaxis(em[...], eksen[0], 0)            # (5, satır, sütun): 0 = bant 10 ... 4 = bant 14
                N_ = nd[...]
            n_fill = [k for k in na if "fill" in k.lower() or "missing" in k.lower()]
            ndvi_fill = att(na, n_fill[0]) if n_fill else None
            assert ndvi_fill is not None, f"NDVI fill özniteliği yok: {list(na)}"
            ndvi_bilgi[tr] = {"oznitelikler": {k: (v.tolist() if hasattr(v, "tolist") else str(v)) for k, v in na.items()},
                              "fill": ndvi_fill}
            assert E.shape[1:] == (3600, 7200) and N_.shape == (3600, 7200), (E.shape, N_.shape)
            for j in range(5):
                ham = E[j]
                v = (ham.astype(np.float64) * olcek + ofset).astype(np.float32)
                v[ham == 0] = np.nan                            # 0 = yok (rapor E4)
                ad = f"yayinim_b{10 + j}_{tr}"
                npy_yaz(p, ad, v, {"kaynak": f"dunya/E4_aster_ged/{dosya.name}", "veri_seti": "SDS/Emissivity", "bant": 10 + j,
                                   "geotransform": CMG_GT, "crs": "EPSG:4326 (0,05°)", "birim": "yayınım", "olcek": olcek,
                                   "ofset": ofset, "ofset_kaynagi": "dosya özniteliği (talimat 7)", "veri_yok_ham": 0})
                f_, n_nan, n_yok = npy_dogrula(p, ad, ham, lambda h: h.astype(np.float64) * 0.002 + 0.49, lambda h: h == 0, rng, 1000)
                n_fark += f_
                if n_nan != n_yok:
                    nan_hata.append(ad)
            v = (N_.astype(np.float64) * n_olcek).astype(np.float32)
            v[N_ == ndvi_fill] = np.nan
            ad = f"ndvi_{tr}"
            npy_yaz(p, ad, v, {"kaynak": f"dunya/E4_aster_ged/{dosya.name}", "veri_seti": "SDS/NDVI", "geotransform": CMG_GT,
                               "crs": "EPSG:4326 (0,05°)", "birim": "NDVI", "olcek": n_olcek, "ofset": 0.0,
                               "veri_yok_ham": ndvi_fill, "veri_yok_kaynagi": f"dosya özniteliği '{n_fill[0]}'"})
            f_, n_nan, n_yok = npy_dogrula(p, ad, N_, lambda h: h.astype(np.float64) * 0.001, lambda h: h == ndvi_fill, rng, 1000)
            n_fark += f_
            if n_nan != n_yok:
                nan_hata.append(ad)
            print(f"    {tr} tamam", flush=True)
        sinav("E4", "rastgele piksel fark 0", f"12 ay x 6 dizi x 1000 piksel, {n_fark} farkli", n_fark == 0, d3)
        sinav("E4", "NaN sayisi = veri-yok sayisi", f"tutmayan dizi {nan_hata}", not nan_hata, d3)
        meta_yaz(p, "E4", {"crs": "EPSG:4326 (0,05°)", "geotransform": CMG_GT, "cozunurluk": "0,05°",
                           "olcek": {"yayinim": 0.002, "ndvi": 0.001}, "ofset": {"yayinim": 0.49, "ndvi": 0.0},
                           "ofset_kaynagi": "dosya özniteliği 'Offset' = 0,49 (talimat 7)",
                           "veri_yok_kurali": "yayınım ham 0 -> NaN; NDVI ham = dosyadaki fill -> NaN",
                           "ndvi_oznitelikleri": ndvi_bilgi,
                           "dosyalar": "yayinim_b10..b14_<YYYYDDD>.npy, ndvi_<YYYYDDD>.npy (12 ay)"}, d3, D1, IND, D2)

    bolum("E4", e4)

    # ------------------------------------------------ E5 MYD11C3: 12 ay x gündüz/gece LST -> .npy (K)
    def e5():
        p = klasor("E5", "myd11c3")
        dosyalar = sorted((VERI / "dunya/E5_myd11c3").glob("MYD11C3.A2023*.hdf"))
        assert len(dosyalar) == 12
        d3, n_fark, nan_hata, gecersiz = [], 0, [], {}
        for dosya in dosyalar:
            tr = tarih(dosya)
            for deg, kisa in (("LST_Day_CMG", "gunduz"), ("LST_Night_CMG", "gece")):
                ds = alt(dosya, deg)
                md = ds.GetMetadata()
                assert float(md["scale_factor"]) == 0.02 and int(float(md["_FillValue"])) == 0, md
                assert float(md.get("add_offset", 0)) == 0.0, md
                ham = ds.GetRasterBand(1).ReadAsArray()
                assert ham.shape == (3600, 7200)
                v = (ham.astype(np.float64) * 0.02).astype(np.float32)
                v[ham == 0] = np.nan
                vr = md.get("valid_range")
                if vr:
                    lo, hi = [float(x) for x in re.split(r"[,\s]+", vr.strip()) if x]
                    gecersiz[f"{kisa}_{tr}"] = int(((ham != 0) & ((ham < lo) | (ham > hi))).sum())
                ad = f"lst_{kisa}_{tr}"
                npy_yaz(p, ad, v, {"kaynak": f"dunya/E5_myd11c3/{dosya.name}", "alt_veri_seti": deg, "geotransform": CMG_GT,
                                   "crs": "EPSG:4326 (CMG 0,05°)", "birim": "K", "olcek": 0.02, "ofset": 0.0, "veri_yok_ham": 0,
                                   "valid_range": vr})
                f_, n_nan, n_yok = npy_dogrula(p, ad, ham, lambda h: h.astype(np.float64) * 0.02, lambda h: h == 0, rng, 1000)
                n_fark += f_
                if n_nan != n_yok:
                    nan_hata.append(ad)
        sinav("E5", "rastgele piksel fark 0", f"12 ay x 2 x 1000 piksel, {n_fark} farkli", n_fark == 0, d3)
        sinav("E5", "NaN sayisi = veri-yok sayisi", f"tutmayan dizi {nan_hata}", not nan_hata, d3)
        sinav("E5", "valid_range disi (fill olmayan) deger sayisi (kayit)", f"toplam {sum(gecersiz.values())}", True, d3)
        meta_yaz(p, "E5", {"crs": "EPSG:4326 (CMG 0,05°)", "geotransform": CMG_GT, "cozunurluk": "0,05°", "olcek": 0.02, "ofset": 0.0,
                           "veri_yok_kurali": "ham 0 -> NaN (dosya _FillValue)", "valid_range_disi": gecersiz,
                           "dosyalar": "lst_gunduz_<YYYYDDD>.npy, lst_gece_<YYYYDDD>.npy (12 ay)"}, d3, D1, IND, D2)

    bolum("E5", e5)

    # ------------------------------------------------ E6 MCD43C3: 12 tarih albedo -> .npy
    def e6():
        p = klasor("E6", "mcd43c3")
        dosyalar = sorted((VERI / "dunya/E6_mcd43c3").glob("MCD43C3.A2023*.hdf"))
        assert len(dosyalar) == 12
        d3, n_fark, nan_hata = [], 0, []
        yok = lambda h: (h < 0) | (h > 32766)                   # belge 10.1: geçerli 0–32 766
        for dosya in dosyalar:
            tr = tarih(dosya)
            ds = alt(dosya, "Albedo_WSA_shortwave")
            md = ds.GetMetadata()
            assert float(md["scale_factor"]) == 0.001 and int(float(md["_FillValue"])) == 32767, md
            assert float(md.get("add_offset", 0)) == 0.0, md
            ham = ds.GetRasterBand(1).ReadAsArray()
            assert ham.shape == (3600, 7200)
            v = (ham.astype(np.float64) * 0.001).astype(np.float32)
            v[yok(ham)] = np.nan
            ad = f"albedo_wsa_{tr}"
            npy_yaz(p, ad, v, {"kaynak": f"dunya/E6_mcd43c3/{dosya.name}", "alt_veri_seti": "Albedo_WSA_shortwave",
                               "geotransform": CMG_GT, "crs": "EPSG:4326 (CMG 0,05°)", "birim": "albedo", "olcek": 0.001,
                               "ofset": 0.0, "veri_yok_ham": "0–32766 dışı (fill 32767)"})
            f_, n_nan, n_yok = npy_dogrula(p, ad, ham, lambda h: h.astype(np.float64) * 0.001, yok, rng, 1000)
            n_fark += f_
            if n_nan != n_yok:
                nan_hata.append(ad)
        sinav("E6", "rastgele piksel fark 0", f"12 tarih x 1000 piksel, {n_fark} farkli", n_fark == 0, d3)
        sinav("E6", "NaN sayisi = veri-yok sayisi", f"tutmayan dizi {nan_hata}", not nan_hata, d3)
        meta_yaz(p, "E6", {"crs": "EPSG:4326 (CMG 0,05°)", "geotransform": CMG_GT, "cozunurluk": "0,05°", "olcek": 0.001, "ofset": 0.0,
                           "veri_yok_kurali": "ham 0–32766 dışı -> NaN (fill 32767)",
                           "dosyalar": "albedo_wsa_<YYYYDDD>.npy (12 tarih)"}, d3, D1, IND, D2)

    bolum("E6", e6)

    # ------------------------------------------------ tek tif setleri: E7, E8, E9, E10
    def tek_tif(kod, ad, kaynaklar, olcek, aralik, ek):
        p = klasor(kod, ad)
        d3, n_fark, nan_hata, bilgiler = [], 0, [], {}
        for k, hedef_ad in kaynaklar:
            b = tif_cevir(k, p / hedef_ad, olcek, 0.0)
            f_, n_nan = tif_dogrula(k, p / hedef_ad, b, rng, 1000)
            n_fark += f_
            if n_nan != b["veri_yok_sayisi"]:
                nan_hata.append(hedef_ad)
            bilgiler[hedef_ad] = b
            print(f"    {hedef_ad}: min {b['min']}, max {b['max']}, veri-yok {b['veri_yok_sayisi']}", flush=True)
        sinav(kod, "rastgele piksel fark 0", f"{len(kaynaklar)} dosya x 1000 piksel, {n_fark} farkli", n_fark == 0, d3)
        sinav(kod, "NaN sayisi = veri-yok sayisi", f"tutmayan {nan_hata}", not nan_hata, d3)
        lo, hi = aralik
        mn, mx = min(b["min"] for b in bilgiler.values()), max(b["max"] for b in bilgiler.values())
        sinav(kod, f"gecerli deger araligi [{lo}, {hi}]", f"min {mn}, max {mx}", lo <= mn and mx <= hi, d3)
        b0 = next(iter(bilgiler.values()))
        meta_yaz(p, kod, {"crs": b0["crs"], "piksel": b0["piksel"], "olcek": olcek, "ofset": 0.0,
                          "veri_yok_kurali": "ham nodata (dosyadan) ve sonlu olmayan değerler -> NaN",
                          "okuma": bilgiler, **ek}, d3, D1, IND, D2)

    bolum("E7", lambda: tek_tif("E7", "permafrost_kuzey",
                                [(VERI / "dunya/E7_permafrost_kuzey/acik/UiO_PEX_PERPROB_5.0_20181128_2000_2016_NH.tif", "perprob_kuzey.tif")],
                                1.0, (0.0, 1.0), {"dosyalar": "perprob_kuzey.tif"}))
    bolum("E8", lambda: tek_tif("E8", "permafrost_antarktika",
                                [(VERI / "dunya/E8_permafrost_antarktika/acik/UiO_PEX_PERPROB_5.0_20181205_2000_2017_ANT.tif", "perprob_antarktika.tif")],
                                1.0, (0.0, 1.0), {"dosyalar": "perprob_antarktika.tif"}))
    bolum("E9", lambda: tek_tif("E9", "aridity",
                                [(VERI / "dunya/E9_aridity/acik/Global-AI_ET0__annual_v3_1/ai_v31_yr.tif", "ai.tif")],
                                1e-4, (0.0, float("inf")), {"carpan_kaynagi": "ham / 10000 (rapor E9, belge D9; D2'de doğrulandı)",
                                                    "dosyalar": "ai.tif"}))
    bolum("E10", lambda: tek_tif("E10", "soilgrids",
                                 [(VERI / "dunya/E10_soilgrids/clay/clay_0-5cm_mean_1000.tif", "kil_0-5cm.tif"),
                                  (VERI / "dunya/E10_soilgrids/silt/silt_0-5cm_mean_1000.tif", "silt_0-5cm.tif")],
                                 1.0, (0.0, 1000.0), {"dosyalar": "kil_0-5cm.tif, silt_0-5cm.tif (‰)"}))

    # ------------------------------------------------ kayıt
    eski = []
    if D3CSV.exists():
        eski = [r for r in csv_oku(D3CSV) if not (r["govde"] == "dunya" and r["set"] in calisan)]
    with open(D3CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["govde", "set", "test", "olculen", "durum", "zaman"])
        w.writeheader()
        w.writerows(eski + SATIRLAR)
    say = {}
    for s in SATIRLAR:
        say[s["durum"]] = say.get(s["durum"], 0) + 1
    boyut = sum(f.stat().st_size for f in (TEMIZ / "dunya").rglob("*") if f.is_file()) / 1e9
    with open(KAYIT / "13_kasim_log.txt", "a", encoding="utf-8") as f:
        f.write(f"{datetime.now():%Y-%m-%d %H:%M:%S}  D3 dunya {calisan}: {say}, {round((time.time() - t0) / 60, 1)} dk, temiz/dunya {boyut:.2f} GB\n")
    print("=" * 60)
    print(f"D3 Dünya {calisan}: {say} | {round((time.time() - t0) / 60, 1)} dk | temiz\\dunya {boyut:.2f} GB | kayit/D3.csv")
    return say
'''
compile(kod, "d3_dunya.py", "exec")
dosya = KOK / "kod" / "d3_dunya.py"
dosya.write_text(kod, encoding="utf-8")
print("Yazıldı:", dosya, "\n")
g = runpy.run_path(str(dosya), run_name="d3_dunya")
_ = g["main"]()

Yazıldı: C:\Users\USER\orbitrary\kod\d3_dunya.py 


--- E1 başladı 01:19:35
    48/288 karo
    96/288 karo
    144/288 karo
    192/288 karo
    240/288 karo
    288/288 karo
[PASS] E1 rastgele piksel fark 0: 1152 piksel (288 karo x 4), 0 farkli
[PASS] E1 NaN sayisi = veri-yok sayisi: 288 karo; tutmayan karo 0 []; toplam veri-yok 0
--- E1 bitti, 5.1 dk

--- E2 başladı 01:24:39
[PASS] E2 rastgele piksel fark 0: 17 sinif x 1000 piksel, 0 farkli
[PASS] E2 NaN sayisi = veri-yok sayisi: tutmayan sinif []
--- E2 bitti, 5.3 dk

--- E3 başladı 01:29:54


C:\Users\USER\anaconda3\envs\orbitrary\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined. Further messages of this type will be suppressed.
  return ogr_read(
C:\Users\USER\anaconda3\envs\orbitrary\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts.  The processing may be really slow.  You can skip the processing by setting METHOD=SKIP. Further messages of this type will be suppressed.
  return ogr_read(


[PASS] E3 rastgele 1000 poligon: birim ayni, alan_km2 yeniden hesapla ayni: 1235259 kayit (ham 1235259); birim esit True; alan en buyuk goreli fark 0.00e+00
[PASS] E3 bos birim / bos geometri sayisi (kayit): bos birim 0, bos geometri 0
--- E3 bitti, 2.0 dk

--- E4 başladı 01:31:55
[FAIL] E4 bolum calisamadi: AssertionError: NDVI fill özniteliği yok: ['Description', 'Offset', 'Precision', 'Scale Factor']
--- E4 bitti, 0.0 dk

--- E5 başladı 01:31:55
[PASS] E5 rastgele piksel fark 0: 12 ay x 2 x 1000 piksel, 0 farkli
[PASS] E5 NaN sayisi = veri-yok sayisi: tutmayan dizi []
[PASS] E5 valid_range disi (fill olmayan) deger sayisi (kayit): toplam 0
--- E5 bitti, 0.4 dk

--- E6 başladı 01:32:19
[PASS] E6 rastgele piksel fark 0: 12 tarih x 1000 piksel, 0 farkli
[PASS] E6 NaN sayisi = veri-yok sayisi: tutmayan dizi []
--- E6 bitti, 0.1 dk

--- E7 başladı 01:32:25
    perprob_kuzey.tif: min 0.0, max 1.0, veri-yok 368811010
[PASS] E7 rastgele piksel fark 0: 1 dosya x 1000 piksel, 0 farkli
[PASS] 

In [23]:
# Developed with assistance from Claude (Anthropic).
# İnceleme (yazma yok): E4 NDVI öznitelikleri + veri-yok kanıtı; E9 aridity'de 65535 / 0 ham değerleri ve dosya üst verisi.
from pathlib import Path
import h5py, numpy as np, rasterio, re
from rasterio.windows import Window
from pyproj import Transformer

KOK = Path.home() / "orbitrary"

print("=" * 20, "E4 ASTER NDVI", "=" * 20)
p = sorted((KOK / "veri/dunya/E4_aster_ged").glob("ASTER_GEDv4.1_A2015*.h5"))[0]
with h5py.File(p, "r") as f:
    for ad in ("SDS/Emissivity", "SDS/NDVI"):
        print(f"== {ad}: dtype {f[ad].dtype}, şekil {f[ad].shape}")
        for k, v in f[ad].attrs.items():
            print(f"   {k!r}: {v!r}")
    em = f["SDS/Emissivity"][...]
    nd = f["SDS/NDVI"][...]
eksen = [i for i, s in enumerate(em.shape) if s == 5][0]
em_yok = (np.moveaxis(em, eksen, 0) == 0).all(axis=0)
d, n = np.unique(nd, return_counts=True)
print(f"NDVI ham: min {nd.min()}, max {nd.max()}; en sık 5: {[(int(d[i]), int(n[i])) for i in np.argsort(n)[::-1][:5]]}")
d2, n2 = np.unique(nd[em_yok], return_counts=True)
print(f"yayınımın yok olduğu {int(em_yok.sum())} pikselde NDVI ham (en sık 5): {[(int(d2[i]), int(n2[i])) for i in np.argsort(n2)[::-1][:5]]}")
print(f"yayınımın olduğu pikselde NDVI ham: min {nd[~em_yok].min()}, max {nd[~em_yok].max()}")

print("\n" + "=" * 20, "E9 ARIDITY", "=" * 20)
klas = KOK / "veri/dunya/E9_aridity/acik/Global-AI_ET0__annual_v3_1"
yol = klas / "ai_v31_yr.tif"
with rasterio.open(yol) as d:
    print(f"dtype {d.dtypes[0]}, nodata {d.nodata}, boyut {d.width}x{d.height}, crs {d.crs}, tags {d.tags(1)}")
    n65535 = n0 = 0
    for r0 in range(0, d.height, 1024):
        a = d.read(1, window=Window(0, r0, d.width, min(1024, d.height - r0)))
        n65535 += int((a == 65535).sum()); n0 += int((a == 0).sum())
    print(f"ham 65535 sayısı {n65535:,}, ham 0 sayısı {n0:,}, toplam piksel {d.width * d.height:,}")
    for ad, (e, b) in {"Point Nemo (açık okyanus; NOAA)": (-48.876667, -123.393333),
                       "Atacama Yungay": (-24.0656, -69.8639), "Manaus": (-3.11889, -60.02167)}.items():
        r, c = d.index(b, e)
        print(f"   {ad}: ham {int(d.read(1, window=Window(c, r, 1, 1))[0, 0])}")
print("\n.aux.xml:", (klas / "ai_v31_yr.tif.aux.xml").read_text(encoding="utf-8", errors="replace"))
x = (klas / "ai_v31_yr.tif.xml").read_text(encoding="utf-8", errors="replace")
print(".tif.xml'de nodata / 65535 geçen satırlar:", [s.strip()[:200] for s in x.splitlines() if re.search(r"nodata|no data|65535|null", s, re.I)])

==================== E4 ASTER NDVI ====================
== SDS/Emissivity: dtype uint8, şekil (5, 3600, 7200)
   'Description': array([b'Emissivity'], dtype='|S11')
   'Offset': array([0.49])
   'Precision': array([b'uint8'], dtype='|S6')
   'Scale Factor': array([0.002])
== SDS/NDVI: dtype int16, şekil (3600, 7200)
   'Description': array([b'Normalized Difference Vegetation Index'], dtype='|S39')
   'Offset': array([0.])
   'Precision': array([b'uint16'], dtype='|S7')
   'Scale Factor': array([0.001])
NDVI ham: min 0, max 999; en sık 5: [(0, 21310117), (98, 19093), (97, 18931), (100, 18774), (96, 18702)]
yayınımın yok olduğu 17321730 pikselde NDVI ham (en sık 5): [(0, 17260156), (59, 195), (53, 192), (63, 185), (333, 180)]
yayınımın olduğu pikselde NDVI ham: min 0, max 999

==================== E9 ARIDITY ====================
dtype uint16, nodata None, boyut 43200x18000, crs EPSG:4326, tags {'BandName': 'awi_pm_sr_yr', 'RepresentationType': 'ATHEMATIC', 'STATISTICS_COUNT': '777600000.

In [24]:
# Developed with assistance from Claude (Anthropic).
# İnceleme (yazma yok): E9 aridity — readme'de veri-yok / ölçek geçen satırlar; ham 0 ve 65535 piksellerinin su/kara ve enlem dağılımı.
# Su/kara: temiz\dunya\E2_mcd12c1\sinif_00_yuzde.npy (MCD12C1 sınıf 0 = su, %; D3'te PASS).
from pathlib import Path
import zipfile, re, numpy as np, rasterio
from rasterio.windows import Window

KOK = Path.home() / "orbitrary"
klas = KOK / "veri/dunya/E9_aridity/acik/Global-AI_ET0__annual_v3_1"

print("=" * 20, "README (Global AI_ET0 v3.1 - Readme.docx)", "=" * 20)
with zipfile.ZipFile(klas / "Global AI_ET0 v3.1 - Readme.docx") as z:
    x = z.read("word/document.xml").decode("utf-8", errors="replace")
metin = re.sub(r"<[^>]+>", "", re.sub(r"</w:p>", "\n", x))
anahtar = re.compile(r"no ?data|missing|null|65535|65,535|ocean|sea|water|multipl|10,?000|scal|integer|uint|valid|range|antarc|ice|glacier", re.I)
satirlar = [s.strip() for s in metin.splitlines() if s.strip() and anahtar.search(s)]
for s in satirlar[:40]:
    print(" -", s[:400])

print("\n" + "=" * 20, "E9 HAM 0 ve 65535 NEREDE", "=" * 20)
su = np.load(KOK / "temiz/dunya/E2_mcd12c1/sinif_00_yuzde.npy", mmap_mode="r")   # 3600 x 7200, 0,05°
with rasterio.open(klas / "ai_v31_yr.tif") as d:
    tr = d.transform
    print("transform:", list(tr)[:6])
    assert abs(tr.a - 1 / 120) < 1e-9 and abs(tr.e + 1 / 120) < 1e-9 and abs(tr.c + 180) < 1e-6 and abs(tr.f - 90) < 1e-6, "beklenen 30 yay-sn, sol üst (-180, 90)"
    say = {"0_su": 0, "0_kara": 0, "0_su_bilinmiyor": 0, "65535_su": 0, "65535_kara": 0, "65535_su_bilinmiyor": 0}
    enlem65535 = {}
    for r0 in range(0, d.height, 1200):
        h = min(1200, d.height - r0)
        a = d.read(1, window=Window(0, r0, d.width, h))
        w = np.asarray(su[r0 // 6:(r0 + h) // 6, :]).repeat(6, 0).repeat(6, 1)        # 1/120° -> 1/20°: 6 x 6
        for deg in (0, 65535):
            m = a == deg
            say[f"{deg}_su"] += int((m & (w == 100)).sum())
            say[f"{deg}_kara"] += int((m & (w < 100)).sum())
            say[f"{deg}_su_bilinmiyor"] += int((m & np.isnan(w)).sum())
        n = int((a == 65535).sum())
        if n:
            enlem65535[f"{90 - r0 / 120:.0f}..{90 - (r0 + h) / 120:.0f}"] = n
print("sayımlar (su = MCD12C1 sınıf 0 %100; kara = < %100):", say)
print("65535'in enlem bantlarına dağılımı (10°):", enlem65535)

==================== README (Global AI_ET0 v3.1 - Readme.docx) ====================
 - Note: Version 3.1 supersedes all previous releases. Version 3.0 has been deprecated due to the discovery of a critical bug in the calculation of net longwave radiation in the source code used to generate the dataset. This error, present in v3.0, involved the incorrect use of air temperature in degrees Celsius rather than Kelvin in the Stefan-Boltzmann equation. As a result, potential evapotranspir
 - These geo-spatial datasets have been produced with the support of the National Key Basic Research Program of China (Grant No. 2014CB954100), and National Science Foundation China (Grant No. 31270524), and a Presidential Fellowship from the Chinese Academy of Science at the Center for Mountain Futures, Kunming Institute of Botany. Funding for a previous version has been provided by CGIAR-CSI (fundi
 - Potential Evapo-Transpiration (PET) is a measure of the ability of the atmosphere to remove water through

In [25]:
# Developed with assistance from Claude (Anthropic).
# Adım D3 (Dünya) yaması — E4 NDVI ve E9 aridity veri-yok kuralı düzeltilir; yalnız E4 ve E9 yeniden çalıştırılır.
# E4 NDVI: dosyada fill yok -> yayınımın 5 bandı da 0 olan piksel = veri yok (NaN); kalan ham 0 = bayrak (belgelenmemiş kod).
# E9 AI:   ham 0 ve MCD12C1 su %100 -> veri yok (NaN); karadaki ham 0 = bayrak 1; ham 65535 (uint16 tavanı) = bayrak 2 (kırpılmış).
from pathlib import Path
import runpy

KOK = Path.home() / "orbitrary"
dosya = KOK / "kod" / "d3_dunya.py"
eski = dosya.read_text(encoding="utf-8")

E4_YENI = r'''    def e4():
        p = klasor("E4", "aster_ged")
        dosyalar = sorted((VERI / "dunya/E4_aster_ged").glob("ASTER_GEDv4.1_A2015*.h5"))
        assert len(dosyalar) == 12
        d3, n_fark, nan_hata, bayrak_hata, ndvi_bilgi = [], 0, [], [], {}
        for dosya in dosyalar:
            tr = tarih(dosya)
            with h5py.File(dosya, "r") as f:
                em, nd = f["SDS/Emissivity"], f["SDS/NDVI"]
                ea, na = dict(em.attrs), dict(nd.attrs)
                olcek, ofset = att(ea, "Scale Factor"), att(ea, "Offset")
                n_olcek, n_ofset = att(na, "Scale Factor"), att(na, "Offset")
                assert olcek == 0.002 and ofset is not None and abs(ofset - 0.49) < 1e-9, (olcek, ofset)
                assert n_olcek == 0.001 and n_ofset == 0.0, (n_olcek, n_ofset)
                assert not [k for k in na if "fill" in k.lower() or "missing" in k.lower()], "NDVI'de fill çıktı: kural gözden geçirilmeli"
                eksen = [i for i, s in enumerate(em.shape) if s == 5]
                assert len(eksen) == 1, em.shape
                E = np.moveaxis(em[...], eksen[0], 0)            # (5, satır, sütun): 0 = bant 10 ... 4 = bant 14
                N_ = nd[...]
            assert E.shape[1:] == (3600, 7200) and N_.shape == (3600, 7200), (E.shape, N_.shape)
            assert int(N_.min()) >= 0, f"NDVI ham negatif: {N_.min()}"
            ndvi_bilgi[tr] = {"oznitelikler": {k: (v.tolist() if hasattr(v, "tolist") else str(v)) for k, v in na.items()},
                              "ham_min": int(N_.min()), "ham_max": int(N_.max())}
            for j in range(5):
                ham = E[j]
                v = (ham.astype(np.float64) * olcek + ofset).astype(np.float32)
                v[ham == 0] = np.nan                            # 0 = yok (rapor E4)
                ad = f"yayinim_b{10 + j}_{tr}"
                npy_yaz(p, ad, v, {"kaynak": f"dunya/E4_aster_ged/{dosya.name}", "veri_seti": "SDS/Emissivity", "bant": 10 + j,
                                   "geotransform": CMG_GT, "crs": "EPSG:4326 (0,05°)", "birim": "yayınım", "olcek": olcek,
                                   "ofset": ofset, "ofset_kaynagi": "dosya özniteliği (talimat 7)", "veri_yok_ham": 0})
                f_, n_nan, n_yok = npy_dogrula(p, ad, ham, lambda h: h.astype(np.float64) * 0.002 + 0.49, lambda h: h == 0, rng, 1000)
                n_fark += f_
                if n_nan != n_yok:
                    nan_hata.append(ad)
            em_yok = (E == 0).all(axis=0)                        # yayınımın 5 bandı da yok = ürünün boş pikseli
            v = (N_.astype(np.float64) * n_olcek).astype(np.float32)
            v[em_yok] = np.nan
            bay = ((N_ == 0) & ~em_yok).astype(np.uint8)          # 1 = kara pikselinde ham 0 (bitki yok ya da kırpılmış negatif)
            ad = f"ndvi_{tr}"
            npy_yaz(p, ad, v, {"kaynak": f"dunya/E4_aster_ged/{dosya.name}", "veri_seti": "SDS/NDVI", "geotransform": CMG_GT,
                               "crs": "EPSG:4326 (0,05°)", "birim": "NDVI", "olcek": n_olcek, "ofset": n_ofset,
                               "veri_yok_kurali": "dosyada fill yok; yayınımın 5 bandı da 0 olan piksel -> NaN",
                               "bayrak_dosyasi": f"ndvi_bayrak_{tr}.npy"})
            np.save(p / f"ndvi_bayrak_{tr}.npy", bay)
            (p / f"ndvi_bayrak_{tr}.json").write_text(json.dumps(
                {"kaynak": f"dunya/E4_aster_ged/{dosya.name}", "sekil": list(bay.shape), "dtype": "uint8", "geotransform": CMG_GT,
                 "anlam": "1 = yayınım verisi olan pikselde NDVI ham 0 (bitki yok ya da kırpılmış negatif; belgelenmemiş kod); 0 = yok"},
                ensure_ascii=False, indent=2), encoding="utf-8")
            # D3 testi: kaydedilenler diskten açılır, ham dizilerden yeniden hesaplanır
            t = np.load(p / f"{ad}.npy", mmap_mode="r")
            rr, cc = rng.integers(0, 3600, 1000), rng.integers(0, 7200, 1000)
            bek = (N_[rr, cc].astype(np.float64) * 0.001).astype(np.float32)
            bek[(E[:, rr, cc] == 0).all(axis=0)] = np.nan
            n_fark += int((~esit(bek, np.asarray(t[rr, cc]))).sum())
            if int(np.isnan(t).sum()) != int(((E == 0).sum(axis=0) == 5).sum()):
                nan_hata.append(ad)
            tb = np.load(p / f"ndvi_bayrak_{tr}.npy")
            if int(tb.sum()) != int(((N_ == 0) & ((E != 0).any(axis=0))).sum()):
                bayrak_hata.append(tr)
            ndvi_bilgi[tr].update({"nan": int(em_yok.sum()), "bayrak_1": int(bay.sum())})
            print(f"    {tr} tamam (NDVI NaN {int(em_yok.sum())}, bayrak {int(bay.sum())})", flush=True)
        sinav("E4", "rastgele piksel fark 0", f"12 ay x 6 dizi x 1000 piksel, {n_fark} farkli", n_fark == 0, d3)
        sinav("E4", "NaN sayisi = veri-yok sayisi", f"tutmayan dizi {nan_hata}", not nan_hata, d3)
        sinav("E4", "NDVI bayrak sayisi = ham 0 (yayinim olan piksel) sayisi", f"tutmayan ay {bayrak_hata}", not bayrak_hata, d3)
        meta_yaz(p, "E4", {"crs": "EPSG:4326 (0,05°)", "geotransform": CMG_GT, "cozunurluk": "0,05°",
                           "olcek": {"yayinim": 0.002, "ndvi": 0.001}, "ofset": {"yayinim": 0.49, "ndvi": 0.0},
                           "ofset_kaynagi": "dosya öznitelikleri: Emissivity 'Offset' = 0,49 (talimat 7); NDVI 'Offset' = 0",
                           "veri_yok_kurali": "yayınım ham 0 -> NaN (bant başına); NDVI: dosyada fill yok, yayınımın 5 bandı da 0 -> NaN",
                           "bayrak": "ndvi_bayrak_<YYYYDDD>.npy: 1 = yayınım olan pikselde NDVI ham 0 (bitki yok ya da kırpılmış negatif)",
                           "ndvi_inceleme": "2015001 dosyası: NDVI ham 0–999, negatif yok; yayınımın boş olduğu 17 321 730 pikselin 17 260 156'sında NDVI 0",
                           "ndvi_oznitelikleri": ndvi_bilgi,
                           "dosyalar": "yayinim_b10..b14_<YYYYDDD>.npy, ndvi_<YYYYDDD>.npy, ndvi_bayrak_<YYYYDDD>.npy (12 ay)"}, d3, D1, IND, D2)

    bolum("E4", e4)
'''

E9_YENI = r'''    def e9():
        import rasterio
        from rasterio.windows import Window
        p = klasor("E9", "aridity")
        k = VERI / "dunya/E9_aridity/acik/Global-AI_ET0__annual_v3_1/ai_v31_yr.tif"
        su = np.load(TEMIZ / "dunya/E2_mcd12c1/sinif_00_yuzde.npy", mmap_mode="r")      # MCD12C1 sınıf 0 = su (%), 0,05°
        n = {"nan": 0, "bayrak_1": 0, "bayrak_2": 0}
        vmin, vmax = np.inf, -np.inf
        with rasterio.open(k) as s:
            tr = s.transform
            assert s.count == 1 and s.dtypes[0] == "uint16" and s.nodata is None, (s.count, s.dtypes, s.nodata)
            assert s.scales[0] == 1.0 and s.offsets[0] == 0.0
            assert abs(tr.a - 1 / 120) < 1e-9 and abs(tr.e + 1 / 120) < 1e-9 and abs(tr.c + 180) < 1e-6 and abs(tr.f - 90) < 1e-6, list(tr)
            assert s.height % 6 == 0 and s.width == 43200
            prof = s.profile.copy()
            prof.update(driver="GTiff", dtype="float32", nodata=np.nan, compress="deflate", predictor=3,
                        tiled=True, blockxsize=512, blockysize=512, BIGTIFF="IF_SAFER")
            bprof = s.profile.copy()
            bprof.update(driver="GTiff", dtype="uint8", nodata=None, compress="deflate", tiled=True, blockxsize=512, blockysize=512)
            with rasterio.open(p / "ai.tif", "w", **prof) as t, rasterio.open(p / "ai_bayrak.tif", "w", **bprof) as tb:
                for r0 in range(0, s.height, 1200):
                    h = min(1200, s.height - r0)
                    w = Window(0, r0, s.width, h)
                    a = s.read(1, window=w)
                    su_w = np.asarray(su[r0 // 6:(r0 + h) // 6, :]).repeat(6, 0).repeat(6, 1)
                    yok = (a == 0) & (su_w == 100)
                    b = np.zeros(a.shape, np.uint8)
                    b[(a == 0) & ~yok] = 1
                    b[a == 65535] = 2
                    v = (a.astype(np.float64) * 1e-4).astype(np.float32)
                    v[yok] = np.nan
                    n["nan"] += int(yok.sum()); n["bayrak_1"] += int((b == 1).sum()); n["bayrak_2"] += int((b == 2).sum())
                    if (~yok).any():
                        vmin, vmax = min(vmin, float(np.nanmin(v))), max(vmax, float(np.nanmax(v)))
                    t.write(v, 1, window=w)
                    tb.write(b, 1, window=w)
        d3 = []
        with rasterio.open(k) as s, rasterio.open(p / "ai.tif") as t, rasterio.open(p / "ai_bayrak.tif") as tb:
            assert (s.width, s.height, s.transform, s.crs) == (t.width, t.height, t.transform, t.crs) == (tb.width, tb.height, tb.transform, tb.crs)
            rr, cc = rng.integers(0, s.height, 1000), rng.integers(0, s.width, 1000)
            ham = np.array([s.read(1, window=Window(int(c), int(r), 1, 1))[0, 0] for r, c in zip(rr, cc)])
            tem = np.array([t.read(1, window=Window(int(c), int(r), 1, 1))[0, 0] for r, c in zip(rr, cc)], dtype=np.float32)
            tbo = np.array([tb.read(1, window=Window(int(c), int(r), 1, 1))[0, 0] for r, c in zip(rr, cc)])
            sw = np.asarray(su)[rr // 6, cc // 6]
            bek = (ham.astype(np.float64) / 10000).astype(np.float32)
            bek[(ham == 0) & (sw == 100)] = np.nan
            bekb = np.where(ham == 65535, 2, np.where((ham == 0) & ~(sw == 100), 1, 0))
            n_fark = int((~esit(bek, tem)).sum()) + int((bekb != tbo).sum())
            n_nan = n1 = n2 = 0
            for r0 in range(0, t.height, 1200):
                w = Window(0, r0, t.width, min(1200, t.height - r0))
                n_nan += int(np.isnan(t.read(1, window=w)).sum())
                bb = tb.read(1, window=w)
                n1 += int((bb == 1).sum()); n2 += int((bb == 2).sum())
        sinav("E9", "rastgele piksel fark 0 (deger + bayrak)", f"1000 piksel, {n_fark} farkli", n_fark == 0, d3)
        sinav("E9", "NaN sayisi = veri-yok sayisi (ham 0 ve su %100)", f"NaN {n_nan}, beklenen {n['nan']}", n_nan == n["nan"], d3)
        sinav("E9", "bayrak sayilari (1: karada ham 0, 2: ham 65535)", f"1: {n1} (beklenen {n['bayrak_1']}), 2: {n2} (beklenen {n['bayrak_2']})",
              n1 == n["bayrak_1"] and n2 == n["bayrak_2"], d3)
        sinav("E9", "gecerli deger araligi [0, 6.5535]", f"min {vmin}, max {vmax}", 0 <= vmin and vmax <= 6.5535 + 1e-6, d3)
        meta_yaz(p, "E9", {"crs": "EPSG:4326", "piksel": [1 / 120, 1 / 120], "olcek": 1e-4, "ofset": 0.0,
                           "carpan_kaynagi": "Readme v3.1: 'Global-AI values need to be multiplied by 0.0001'",
                           "veri_yok_kurali": "dosyada nodata yok; ham 0 olan ve MCD12C1 sınıf 0 (su) %100 olan piksel -> NaN "
                                              "(su maskesi: temiz/dunya/E2_mcd12c1/sinif_00_yuzde.npy, 0,05° = 6 x 6 piksel)",
                           "bayrak": "ai_bayrak.tif: 1 = karada ham 0 (aşırı kurak ya da kıyı/örtüşme; belirsiz), "
                                     "2 = ham 65535 (uint16 tavanı, AI >= 6,5535; kırpılmış)",
                           "inceleme": {"ham_0_su": 550911298, "ham_0_kara": 4883730, "ham_65535_su": 256, "ham_65535_kara": 1202807,
                                        "65535_enlem": "çoğu 70–90°K (Grönland) ve 40–60°G; buz örtüsü (veriden çıkarım)",
                                        "nokta": "Point Nemo ham 0, Atacama Yungay ham 0, Manaus ham 19257"},
                           "sayilar": n, "dosyalar": "ai.tif (AI, NaN = veri yok), ai_bayrak.tif (uint8)"}, d3, D1, IND, D2)

    bolum("E9", e9)
'''

def degistir(metin, bas, son, yeni):
    assert metin.count(bas) == 1 and metin.count(son) == 1, f"işaret bulunamadı: {bas[:60]!r}"
    i, j = metin.index(bas), metin.index(son)
    assert i < j
    return metin[:i] + yeni + metin[j:]

yeni = degistir(eski, "    def e4():\n", "    # ------------------------------------------------ E5 MYD11C3", E4_YENI + "\n")
yeni = degistir(yeni, '    bolum("E9", lambda: tek_tif("E9", "aridity",\n', '    bolum("E10", lambda: tek_tif("E10",', E9_YENI)
compile(yeni, "d3_dunya.py", "exec")
dosya.write_text(yeni, encoding="utf-8")
print("Yamandı:", dosya, "\n")
g = runpy.run_path(str(dosya), run_name="d3_dunya")
_ = g["main"](["E4", "E9"])

Yamandı: C:\Users\USER\orbitrary\kod\d3_dunya.py 


--- E4 başladı 01:44:27
    2015001 tamam (NDVI NaN 17321730, bayrak 4049961)
    2015032 tamam (NDVI NaN 17314191, bayrak 3823585)
    2015060 tamam (NDVI NaN 17307496, bayrak 3693422)
    2015091 tamam (NDVI NaN 17317277, bayrak 3567902)
    2015121 tamam (NDVI NaN 17324421, bayrak 3192764)
    2015152 tamam (NDVI NaN 17325125, bayrak 2772838)
    2015182 tamam (NDVI NaN 17325677, bayrak 2708218)
    2015213 tamam (NDVI NaN 17321936, bayrak 2704174)
    2015244 tamam (NDVI NaN 17308274, bayrak 2740021)
    2015274 tamam (NDVI NaN 17310582, bayrak 3099440)
    2015305 tamam (NDVI NaN 17319204, bayrak 3635076)
    2015335 tamam (NDVI NaN 17326228, bayrak 3895867)
[PASS] E4 rastgele piksel fark 0: 12 ay x 6 dizi x 1000 piksel, 0 farkli
[PASS] E4 NaN sayisi = veri-yok sayisi: tutmayan dizi []
[PASS] E4 NDVI bayrak sayisi = ham 0 (yayinim olan piksel) sayisi: tutmayan ay []
--- E4 bitti, 0.5 dk

--- E9 başladı 01:44:59
[PASS] E9 rastgele

In [26]:
# Developed with assistance from Claude (Anthropic).
# Adım D3 (Ay) — kod\d3_ay.py yazılır ve çalıştırılır (belge 17.3 D3, 10.4).
from pathlib import Path
import runpy

KOK = Path.home() / "orbitrary"
kod = r'''# Developed with assistance from Claude (Anthropic).
# d3_ay.py — Orbitrary v1.2, belge 17.3 D3 + 10.4: Ay setleri M1–M7 temiz\ay\<kod>_<ad>\ altına standart biçimde yazılır.
# Raster: fiziksel birim, float32 GeoTIFF, veri yok = NaN. Tablo: sütun başına .npy (+ aynı adlı .json). Vektör: GeoPackage.
# Bayraklar (sansürlü, belgelenmemiş kod) ayrı uint8 dizide; NaN'a çevrilmez (belge 10.4). Her klasöre meta.json.
# D3 testi (belge 17.3): rastgele 1000 piksel/satırda ham dosyadan yeniden hesaplanan değerle fark 0; NaN sayısı = veri-yok sayısı.
# Okuma kuralları d2_ay.py ile aynıdır (D2'de geçti). Ham veri (veri\) yalnız okunur.
import csv, json, math, re, shutil, time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

KOK = Path.home() / "orbitrary"
VERI, KAYIT, TEMIZ = KOK / "veri", KOK / "kayit", KOK / "temiz"
D3CSV = KAYIT / "D3.csv"
TOHUM = 20261113
R_AY = 1737400.0                       # m, belge 4.1

STATIK = {
    "M1": {"ad": "LOLA Global LDEM 118 m (256 ppd)", "doi": "açık (rapor M1 künyesi)", "surum": "Mar2014",
           "lisans": "public domain; 'Please cite authors' (rapor Bölüm 4)", "birim": "m"},
    "M2": {"ad": "Unified Geologic Map of the Moon 1:5M", "doi": "açık (rapor M2 künyesi)", "surum": "2020",
           "lisans": "CC0, but please cite authors (rapor Bölüm 4)", "birim": "sınıf (FIRST_Unit), km²"},
    "M3": {"ad": "Diviner GHRM ra_sam + tbol_sam", "doi": "10.17189/wj0s-w188", "surum": "lro_diviner_derived1 v5.0",
           "lisans": "NASA PDS, kamuya açık (rapor Bölüm 4)", "birim": "ra: kaya bolluğu (birimsiz); tbol: K"},
    "M4": {"ad": "Diviner PRP (kuzey/güney) + GCP", "doi": "10.17189/wj0s-w188", "surum": "PRP LRO-L-DLRE-5-PRP-V2.0; GCP LRO-L-DLRE-5-GCP-V1.0",
           "lisans": "NASA PDS, kamuya açık (rapor Bölüm 4)", "birim": "K; m (ice_depth); derece"},
    "M5": {"ad": "Lunar Prospector GRS element bollukları 2°", "doi": "açık (rapor M5 künyesi)", "surum": "lpgrs_high1_elem_abundance_2deg",
           "lisans": "NASA PDS, kamuya açık (rapor Bölüm 4)", "birim": "etiketteki sütun birimleri"},
    "M6": {"ad": "Lunar Prospector hidrojen 0,5° (hydrogenhd)", "doi": "açık (rapor M6 künyesi)", "surum": "PDS Geosciences reduced special products",
           "lisans": "NASA PDS, kamuya açık (rapor Bölüm 4)", "birim": "ppm H"},
    "M7": {"ad": "PGDA AVGVISIB + LPSR 65° kutup 240 m", "doi": "açık (rapor M7 künyesi)", "surum": "201608",
           "lisans": "NASA, kamuya açık (rapor Bölüm 4)", "birim": "oran (0–1)"},
}

SATIRLAR = []


def csv_oku(yol):
    with open(yol, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def sinav(set_, test, olculen, ok, d3):
    durum = "PASS" if ok else "FAIL"
    d3.append({"test": test, "olculen": olculen, "durum": durum})
    SATIRLAR.append({"govde": "ay", "set": set_, "test": test, "olculen": olculen, "durum": durum,
                     "zaman": datetime.now().isoformat(timespec="seconds")})
    print(f"[{durum}] {set_} {test}: {olculen}", flush=True)


def ham_klasor(kod):
    k = [p for p in (VERI / "ay").glob(f"{kod}_*") if p.is_dir()]
    assert len(k) == 1, f"{kod}: ham klasör {k}"
    return k[0]


def klasor(kod, ad):
    p = TEMIZ / "ay" / f"{kod}_{ad}"
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True)
    return p


def meta_yaz(p, kod, ek, d3, D1, IND, D2):
    ind = [r for r in IND if r["set"] == kod and r["durum"] != "FAIL"]
    adres = sorted({r["adres"] for r in ind if r["adres"]})
    m = {"set": kod, **STATIK[kod],
         "kaynak_adresleri": adres if len(adres) <= 8 else adres[:5] + [f"... toplam {len(adres)} adres (kayit/indirme.csv)"],
         "indirme_tarihi": [min(r["zaman"] for r in ind), max(r["zaman"] for r in ind)] if ind else None,
         "ham_sha256": {r["hedef"]: r["sha256"] for r in D1 if r["set"] == kod},
         "gercek_yer_testleri_D2": [{k: r[k] for k in ("test", "nokta", "beklenen", "tolerans_kurali", "beklenen_kaynagi", "olculen", "durum")}
                                    for r in D2 if r["govde"] == "ay" and (r["set"] == kod or kod in r["set"].split("/"))],
         "D3_testleri": d3, **ek, "uretim_zamani": datetime.now().isoformat(timespec="seconds"), "uretim_kodu": "kod/d3_ay.py"}
    (p / "meta.json").write_text(json.dumps(m, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


def esit(bek, tem):
    return (bek == tem) | (np.isnan(bek) & np.isnan(tem))


def npy_yaz(p, ad, v, bilgi):
    np.save(p / f"{ad}.npy", v)
    (p / f"{ad}.json").write_text(json.dumps({**bilgi, "sekil": list(v.shape), "dtype": str(v.dtype)},
                                             ensure_ascii=False, indent=2, default=str), encoding="utf-8")


# ---------------------------------------------------------------- GeoTIFF -> float32 GeoTIFF (NaN)
def _yok(a, v, nd, sabit_yok, ek_yok):
    y = np.zeros(a.shape, bool)
    if nd is not None:
        y |= np.isnan(a) if (isinstance(nd, float) and math.isnan(nd)) else (a == nd)
    for s in sabit_yok:
        y |= a == s
    if a.dtype.kind == "f":
        y |= ~np.isfinite(a)
    if ek_yok is not None:
        with np.errstate(invalid="ignore"):
            y |= ek_yok(v)
    return y


def tif_cevir(kaynak, hedef, olcek=1.0, ofset=0.0, sabit_yok=(), ek_yok=None, satir=256):
    import rasterio
    from rasterio.windows import Window
    with rasterio.open(kaynak) as s:
        assert s.count == 1, f"{kaynak.name}: {s.count} bant"
        assert s.scales[0] == 1.0 and s.offsets[0] == 0.0, f"{kaynak.name}: dosyada ölçek/ofset {s.scales[0]}/{s.offsets[0]}"
        nd = s.nodata
        prof = s.profile.copy()
        prof.update(driver="GTiff", dtype="float32", nodata=np.nan, count=1, compress="deflate", predictor=3,
                    tiled=True, blockxsize=512, blockysize=512, BIGTIFF="YES")
        n_yok, vmin, vmax = 0, np.inf, -np.inf
        with rasterio.open(hedef, "w", **prof) as t:
            for r0 in range(0, s.height, satir):
                w = Window(0, r0, s.width, min(satir, s.height - r0))
                a = s.read(1, window=w)
                v = (a.astype(np.float64) * olcek + ofset).astype(np.float32)
                yok = _yok(a, v, nd, sabit_yok, ek_yok)
                v[yok] = np.nan
                n_yok += int(yok.sum())
                if (~yok).any():
                    vmin, vmax = min(vmin, float(np.nanmin(v))), max(vmax, float(np.nanmax(v)))
                t.write(v, 1, window=w)
        return {"ham_dtype": s.dtypes[0], "ham_nodata": nd, "boyut": [s.width, s.height], "crs": str(s.crs),
                "transform": list(s.transform)[:6], "piksel": list(s.res), "olcek": olcek, "ofset": ofset,
                "sabit_veri_yok": list(sabit_yok), "veri_yok_sayisi": n_yok, "min": vmin, "max": vmax}


def tif_dogrula(kaynak, hedef, bilgi, rng, n, sabit_yok=(), ek_yok=None, ozel_nan=()):
    import rasterio
    from rasterio.windows import Window
    with rasterio.open(kaynak) as s, rasterio.open(hedef) as t:
        assert (s.width, s.height, s.transform) == (t.width, t.height, t.transform), f"{hedef.name}: geometri farklı"
        assert t.dtypes[0] == "float32"
        rr = list(rng.integers(0, s.height, n)) + [r for r, c in ozel_nan]
        cc = list(rng.integers(0, s.width, n)) + [c for r, c in ozel_nan]
        ham = np.array([s.read(1, window=Window(int(c), int(r), 1, 1))[0, 0] for r, c in zip(rr, cc)])
        tem = np.array([t.read(1, window=Window(int(c), int(r), 1, 1))[0, 0] for r, c in zip(rr, cc)], dtype=np.float32)
        bek = (ham.astype(np.float64) * bilgi["olcek"] + bilgi["ofset"]).astype(np.float32)
        bek[_yok(ham, bek, s.nodata, sabit_yok, ek_yok)] = np.nan
        if ozel_nan:
            bek[-len(ozel_nan):] = np.nan
        n_fark = int((~esit(bek, tem)).sum())
        n_nan = 0
        for r0 in range(0, t.height, 512):
            n_nan += int(np.isnan(t.read(1, window=Window(0, r0, t.width, min(512, t.height - r0)))).sum())
    return n_fark, n_nan


def lbl_sutunlar(lbl):
    t = lbl.read_text(encoding="ascii", errors="replace")
    adlar = []
    for blok in re.findall(r"OBJECT\s*=\s*COLUMN(.*?)END_OBJECT\s*=\s*COLUMN", t, re.S):
        m = re.search(r"\bNAME\s*=\s*\"?([A-Za-z0-9_]+)\"?", blok)
        if m:
            adlar.append(m.group(1).lower())
    return adlar


def oku_tablo(tab, adlar):
    ilk = tab.open(encoding="ascii", errors="replace").readline()
    baslik = any(c.isalpha() for c in ilk.replace("e", "").replace("E", ""))
    df = pd.read_csv(tab, header=0 if baslik else None, names=None if baslik else adlar, skipinitialspace=True)
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def main(setler=None):
    t0 = time.time()
    D1 = csv_oku(KAYIT / "D1_dosyalar.csv")
    IND = csv_oku(KAYIT / "indirme.csv")
    D2 = csv_oku(KAYIT / "D2.csv")
    D2OK = json.loads((KAYIT / "D2_okuma_ay.json").read_text(encoding="utf-8"))
    secili = set(setler) if setler else set(STATIK)
    rng = np.random.default_rng(TOHUM)
    calisan = []

    def bolum(kod, f):
        if kod not in secili:
            return
        calisan.append(kod)
        t1 = time.time()
        print(f"\n--- {kod} başladı {datetime.now():%H:%M:%S}", flush=True)
        try:
            f()
        except Exception as e:
            sinav(kod, "bolum calisamadi", f"{type(e).__name__}: {e}", False, [])
        print(f"--- {kod} bitti, {round((time.time() - t1) / 60, 1)} dk", flush=True)

    # ------------------------------------------------ M1 LOLA: int16 x 0,5 m -> float32 GeoTIFF
    def m1():
        h = ham_klasor("M1")
        p = klasor("M1", "lola")
        k = h / "Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif"
        lbl = (h / "Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.lbl").read_text(encoding="ascii", errors="replace")
        carpan = float(re.search(r"Multiplier\s*=\s*([-\d.Ee+]+)", lbl).group(1))
        taban = float(re.search(r"Base\s*=\s*([-\d.Ee+]+)", lbl).group(1))
        assert (carpan, taban) == (0.5, 0.0), (carpan, taban)
        d3 = []
        b = tif_cevir(k, p / "lola_m.tif", carpan, taban, sabit_yok=(-32768,))
        f_, n_nan = tif_dogrula(k, p / "lola_m.tif", b, rng, 1000, sabit_yok=(-32768,))
        sinav("M1", "rastgele piksel fark 0", f"1000 piksel, {f_} farkli", f_ == 0, d3)
        sinav("M1", "NaN sayisi = veri-yok sayisi", f"NaN {n_nan}, veri-yok {b['veri_yok_sayisi']}", n_nan == b["veri_yok_sayisi"], d3)
        sinav("M1", "uc degerler D2 ile ayni", f"min {b['min']}, max {b['max']} (D2: -9128,5 / 10781,5)",
              b["min"] == -9128.5 and b["max"] == 10781.5, d3)
        meta_yaz(p, "M1", {"crs": b["crs"], "transform": b["transform"], "cozunurluk": "256 ppd (~118 m)",
                           "olcek": carpan, "ofset": taban, "olcek_kaynagi": "etiket Multiplier / Base",
                           "veri_yok_kurali": "ham -32768 ve dosya nodata -> NaN", "okuma": b, "dosyalar": "lola_m.tif"}, d3, D1, IND, D2)

    bolum("M1", m1)

    # ------------------------------------------------ M2 Ay jeolojisi: GeoUnits.shp -> GeoPackage
    def m2():
        import pyogrio, geopandas as gpd
        h = ham_klasor("M2")
        p = klasor("M2", "jeoloji")
        shp = [x for x in h.rglob("GeoUnits.shp") if x.parent.name == "Shapefiles" and x.parent.parent.name == "Lunar_GIS"]
        assert len(shp) == 1, shp
        df = pyogrio.read_dataframe(shp[0], columns=["FIRST_Unit", "AREA_GEO"])
        oran = float(df["AREA_GEO"].sum()) / (4 * math.pi * (R_AY / 1000) ** 2)
        assert 0.99 <= oran <= 1.01, f"AREA_GEO toplamı / 4πR² (km²) = {oran}: birim km² değil ya da kapsam eksik"
        out = gpd.GeoDataFrame({"birim": df["FIRST_Unit"].astype("string"), "alan_km2": df["AREA_GEO"].astype(float)},
                               geometry=df.geometry, crs=df.crs)
        gp = p / "ay_jeoloji.gpkg"
        pyogrio.write_dataframe(out, gp, layer="ay_jeoloji", driver="GPKG")
        geri = pyogrio.read_dataframe(gp, layer="ay_jeoloji")
        i = rng.integers(0, len(df), 1000)
        b_ok = bool((geri["birim"].astype("string").iloc[i].fillna("<bos>").to_numpy()
                     == df["FIRST_Unit"].astype("string").iloc[i].fillna("<bos>").to_numpy()).all())
        a_ok = bool((geri["alan_km2"].to_numpy()[i] == df["AREA_GEO"].astype(float).to_numpy()[i]).all())
        d3 = []
        sinav("M2", "rastgele 1000 poligon: birim ve alan_km2 ham ile ayni",
              f"{len(geri)} kayit (ham {len(df)}); birim {b_ok}; alan {a_ok}; AREA_GEO toplami / 4πR² = {oran:.6f}",
              len(geri) == len(df) and b_ok and a_ok, d3)
        sinav("M2", "bos birim sayisi (kayit)", f"{int(df['FIRST_Unit'].isna().sum())}", True, d3)
        meta_yaz(p, "M2", {"crs": str(df.crs), "olcek": None, "ofset": None, "veri_yok_kurali": "vektör",
                           "kayit_sayisi": len(df), "birim_sayisi": int(df["FIRST_Unit"].nunique()),
                           "alan_km2": "AREA_GEO (belge 10.2); toplam / 4πR² = %.6f -> birim km²" % oran,
                           "ham_shp": str(shp[0].relative_to(VERI)), "dosyalar": "ay_jeoloji.gpkg (birim, alan_km2)"}, d3, D1, IND, D2)

    bolum("M2", m2)

    # ------------------------------------------------ M3 GHRM: ra_sam, tbol_sam (tbol >= 250 K yok sayılır)
    def m3():
        h = ham_klasor("M3")
        p = klasor("M3", "diviner_ghrm")
        d3, okuma = [], {}
        for desen, cikti, ek in (("dghrm_ra_sam*.tif", "ra_sam.tif", None),
                                 ("dghrm_tbol_sam*.tif", "tbol_sam_K.tif", lambda v: v >= 250)):
            k = sorted(h.glob(desen))
            assert len(k) == 1, (desen, k)
            b = tif_cevir(k[0], p / cikti, 1.0, 0.0, ek_yok=ek)
            f_, n_nan = tif_dogrula(k[0], p / cikti, b, rng, 1000, ek_yok=ek)
            okuma[cikti] = b
            sinav("M3", f"{cikti} rastgele piksel fark 0", f"1000 piksel, {f_} farkli", f_ == 0, d3)
            sinav("M3", f"{cikti} NaN sayisi = veri-yok sayisi", f"NaN {n_nan}, veri-yok {b['veri_yok_sayisi']}",
                  n_nan == b["veri_yok_sayisi"], d3)
            print(f"    {cikti}: min {b['min']}, max {b['max']}", flush=True)
        sinav("M3", "tbol gecerli < 250 K", f"max {okuma['tbol_sam_K.tif']['max']}", okuma["tbol_sam_K.tif"]["max"] < 250, d3)
        meta_yaz(p, "M3", {"crs": okuma["ra_sam.tif"]["crs"], "cozunurluk": "128 ppd, 70°G–70°K", "olcek": 1.0, "ofset": 0.0,
                           "veri_yok_kurali": "NaN = yok; tbol >= 250 K -> NaN (belge 10.2)", "okuma": okuma,
                           "dosyalar": "ra_sam.tif, tbol_sam_K.tif"}, d3, D1, IND, D2)

    bolum("M3", m3)

    # ------------------------------------------------ M4 PRP (kuzey/güney) + GCP: sütun başına .npy
    def m4():
        h = ham_klasor("M4")
        p = klasor("M4", "diviner_prp_gcp")
        d3, sayi = [], {}
        for yari in ("north", "south"):
            tab = h / f"dlre_prp_{yari}.tab"
            df = oku_tablo(tab, lbl_sutunlar(tab.with_suffix(".lbl")))
            assert len(df) == 2880000, len(df)
            ice = df["ice_depth"].to_numpy(dtype=np.float64)
            negler = set(np.unique(ice[ice < 0]).tolist())
            assert negler <= {-1.0, -999.0}, negler                                   # talimat 6
            assert df["temp_max"].to_numpy()[ice == -1].max() <= 101.3                 # talimat 6
            for c in ("tri_clon", "tri_clat", "temp_avg", "temp_max"):
                npy_yaz(p, f"prp_{yari}_{c}", df[c].to_numpy(dtype=np.float64),
                        {"kaynak": f"ay/{h.name}/{tab.name}", "sutun": c, "birim": "derece" if "tri_c" in c else "K", "veri_yok": "yok"})
            v = ice.copy()
            v[ice == -999] = np.nan                                                   # sansürlü (> 2,87 m): değer bilinmiyor
            bay = np.zeros(len(ice), np.uint8)
            bay[ice == -999] = 1
            bay[ice == -1] = 2
            npy_yaz(p, f"prp_{yari}_ice_depth_m", v, {"kaynak": f"ay/{h.name}/{tab.name}", "sutun": "ice_depth", "birim": "m",
                                                       "veri_yok": "-999 (sansürlü, > 2,87 m) -> NaN; -1 değer olarak kalır (bayrak 2)"})
            npy_yaz(p, f"prp_{yari}_ice_depth_bayrak", bay, {"anlam": "1 = -999 sansürlü (> 2,87 m); 2 = -1 belgelenmemiş (talimat 6: "
                                                             "yüzeyde kararlı yorumu, Fisher ve ark. 2017); 0 = yok"})
            sayi[yari] = {"n_0": int((ice == 0).sum()), "n_-1": int((ice == -1).sum()), "n_pozitif": int((ice > 0).sum()),
                          "n_-999": int((ice == -999).sum())}
            d2 = {k: D2OK["M4_ice_depth_sayilari"][yari][k] for k in sayi[yari]}
            sinav("M4", f"PRP {yari} talimat 6 sayilari D2 ile ayni", f"{sayi[yari]}", sayi[yari] == d2, d3)
            # D3 testi: tablo diskten yeniden okunur, kaydedilenlerle karşılaştırılır
            df2 = oku_tablo(tab, lbl_sutunlar(tab.with_suffix(".lbl")))           # bağımsız ikinci okuma
            i = rng.integers(0, len(df2), 1000)
            hi = df2["ice_depth"].to_numpy(dtype=np.float64)[i]
            bek = np.where(hi == -999, np.nan, hi)
            n_fark = int((~esit(bek, np.load(p / f"prp_{yari}_ice_depth_m.npy")[i])).sum())
            bekb = np.where(hi == -999, 1, np.where(hi == -1, 2, 0))
            n_fark += int((bekb != np.load(p / f"prp_{yari}_ice_depth_bayrak.npy")[i]).sum())
            for c in ("tri_clon", "tri_clat", "temp_avg", "temp_max"):
                n_fark += int((df2[c].to_numpy(dtype=np.float64)[i] != np.load(p / f"prp_{yari}_{c}.npy")[i]).sum())
            n_nan = int(np.isnan(np.load(p / f"prp_{yari}_ice_depth_m.npy")).sum())
            sinav("M4", f"PRP {yari} rastgele 1000 satir fark 0", f"{n_fark} farkli", n_fark == 0, d3)
            sinav("M4", f"PRP {yari} NaN sayisi = -999 sayisi", f"NaN {n_nan}, -999 {sayi[yari]['n_-999']}", n_nan == sayi[yari]["n_-999"], d3)
        gcp = sorted(h.glob("global_cumul_avg_cyl_*_002.tab"))
        assert len(gcp) == 4, gcp
        gbilgi = {}
        for tab in gcp:
            kisa = re.search(r"cyl_(\w+?)_002", tab.name).group(1)
            df = pd.read_csv(tab, skipinitialspace=True)
            df.columns = [c.strip().lower() for c in df.columns]
            assert {"clat", "ltim", "tbol"} <= set(df.columns), list(df.columns)
            n_fark, nan_hata = 0, []
            i = rng.integers(0, len(df), 1000)
            df2 = pd.read_csv(tab, skipinitialspace=True)
            df2.columns = [c.strip().lower() for c in df2.columns]
            for c in df.columns:
                ham = df[c].to_numpy(dtype=np.float64)
                v = np.where(ham == -9999, np.nan, ham)                              # belge 10.2: -9999 = yok
                npy_yaz(p, f"gcp_{kisa}_{c}", v, {"kaynak": f"ay/{h.name}/{tab.name}", "sutun": c, "veri_yok": "-9999 -> NaN"})
                t = np.load(p / f"gcp_{kisa}_{c}.npy")
                h2 = df2[c].to_numpy(dtype=np.float64)
                n_fark += int((~esit(np.where(h2[i] == -9999, np.nan, h2[i]), t[i])).sum())
                if int(np.isnan(t).sum()) != int((h2 == -9999).sum() + np.isnan(h2).sum()):
                    nan_hata.append(c)
            gbilgi[kisa] = {"satir": len(df), "sutunlar": list(df.columns)}
            sinav("M4", f"GCP {kisa} rastgele 1000 satir fark 0", f"{len(df.columns)} sutun, {n_fark} farkli", n_fark == 0, d3)
            sinav("M4", f"GCP {kisa} NaN sayisi = -9999 sayisi", f"tutmayan sutun {nan_hata}", not nan_hata, d3)
        meta_yaz(p, "M4", {"crs": "Ay coğrafi (derece)", "olcek": None, "ofset": None,
                           "veri_yok_kurali": "PRP ice_depth -999 -> NaN + bayrak 1; -1 değer + bayrak 2; GCP -9999 -> NaN",
                           "talimat6_sayilari": sayi, "gcp": gbilgi,
                           "dosyalar": "prp_<north|south>_<sutun>.npy, prp_*_ice_depth_m.npy, prp_*_ice_depth_bayrak.npy, gcp_<bant>_<sutun>.npy"},
                 d3, D1, IND, D2)

    bolum("M4", m4)

    # ------------------------------------------------ M5 LP GRS: tablo -> .npy
    def m5():
        h = ham_klasor("M5")
        p = klasor("M5", "lp_grs")
        tab = h / "lpgrs_high1_elem_abundance_2deg.tab"
        satirlar = [s for s in tab.read_text(encoding="ascii").splitlines() if s.strip()]
        T = np.array([[float(x) for x in re.split(r"[,\s]+", s.strip()) if x] for s in satirlar])
        assert T.shape[0] == 11306, T.shape
        adlar = []
        for etk in sorted(h.glob("*.lbl")) + sorted(h.glob("*.fmt")):
            a = lbl_sutunlar(etk)
            if len(a) == T.shape[1]:
                adlar = a
                break
        npy_yaz(p, "lpgrs_tablo", T, {"kaynak": f"ay/{h.name}/{tab.name}", "sutun_adlari": adlar or "açık (etikette sütun sayısı tutmadı)",
                                      "veri_yok": "tanımlı değil; ham değerler aynen"})
        d3 = []
        i = rng.integers(0, len(satirlar), 1000)
        t = np.load(p / "lpgrs_tablo.npy")
        n_fark = sum(int((np.array([float(x) for x in re.split(r"[,\s]+", satirlar[k].strip()) if x]) != t[k]).sum()) for k in i)
        sinav("M5", "rastgele 1000 satir fark 0", f"{T.shape[0]} x {T.shape[1]}, {n_fark} farkli; sutun adlari {'var' if adlar else 'yok'}",
              n_fark == 0, d3)
        sinav("M5", "NaN sayisi = 0 (veri-yok tanimli degil)", f"{int(np.isnan(t).sum())}", int(np.isnan(t).sum()) == 0, d3)
        meta_yaz(p, "M5", {"crs": "Ay coğrafi, 2° eş-alanlı pikseller", "olcek": None, "ofset": None, "veri_yok_kurali": "yok",
                           "sutun_adlari": adlar, "dosyalar": "lpgrs_tablo.npy (11306 satır)"}, d3, D1, IND, D2)

    bolum("M5", m5)

    # ------------------------------------------------ M6 LP hidrojen: hydrogenhd -> .npy
    def m6():
        h = ham_klasor("M6")
        p = klasor("M6", "lp_hidrojen")
        kaynak = h / "hydrogenhd.txt"
        satirlar = [s for s in kaynak.read_text(encoding="ascii", errors="replace").splitlines()
                    if s.strip() and not s.lstrip().startswith("#")]

        def coz(s):
            q = [x.strip() for x in s.split(",")]
            return [float(q[0]), float(q[1]), float(q[2]), float(q[3]), float("nan") if "*" in q[4] else float(q[4])]
        T = np.array([coz(s) for s in satirlar])
        assert T.shape == (259200, 5), T.shape
        npy_yaz(p, "hidrojen_tablo", T, {"kaynak": f"ay/{h.name}/{kaynak.name}",
                                         "sutunlar": ["enlem_g", "enlem_k", "boylam_b", "boylam_d", "H_ppm"],
                                         "veri_yok": "'**********' hücresi -> NaN (1 hücre); 4180 negatif hücre kırpılmaz"})
        d3 = []
        t = np.load(p / "hidrojen_tablo.npy")
        i = rng.integers(0, len(satirlar), 1000)
        n_fark = sum(int((~esit(np.array(coz(satirlar[k])), t[k])).sum()) for k in i)
        sinav("M6", "rastgele 1000 satir fark 0", f"{n_fark} farkli", n_fark == 0, d3)
        n_yildiz = sum(1 for s in satirlar if "*" in s.split(",")[4])
        sinav("M6", "NaN sayisi = bozuk hucre sayisi", f"NaN {int(np.isnan(t).sum())}, '*' {n_yildiz}", int(np.isnan(t).sum()) == n_yildiz == 1, d3)
        sinav("M6", "negatif hucre sayisi korundu", f"{int((t[:, 4] < 0).sum())}", int((t[:, 4] < 0).sum()) == 4180, d3)
        meta_yaz(p, "M6", {"crs": "Ay coğrafi, 0,5°", "olcek": None, "ofset": None,
                           "veri_yok_kurali": "'**********' -> NaN; negatifler kırpılmaz (belge 10.2)",
                           "not": "hydrogenlow ve epis çapraz kontrol dosyaları temiz'e yazılmadı (puan yalnız hydrogenhd'yi kullanır)",
                           "dosyalar": "hidrojen_tablo.npy (259200 x 5)"}, d3, D1, IND, D2)

    bolum("M6", m6)

    # ------------------------------------------------ M7 PGDA: DN -> oran; kuzeydeki anormal piksel NaN
    def m7():
        h = ham_klasor("M7")
        p = klasor("M7", "pgda")
        d3, okuma = [], {}
        for ad in ("AVGVISIB_65S", "AVGVISIB_65N", "LPSR_65S", "LPSR_65N"):
            k = h / f"{ad}_240M_201608.TIF"
            lb = (h / f"{ad}_240M_201608.LBL").read_text(encoding="ascii", errors="replace")
            sf = float(re.search(r"SCALING_FACTOR\s*=\s*([-\d.Ee+]+)", lb).group(1))
            of = float(re.search(r"\bOFFSET\s*=\s*([-\d.Ee+]+)", lb).group(1))
            assert (sf, of) == ((0.00004, 0.0) if ad.startswith("AVG") else (0.000025, 0.5)), (ad, sf, of)
            ozel = []
            if ad.endswith("N"):
                import rasterio
                with rasterio.open(k) as s:
                    n = s.height
                e, b = 88.435, 106.602                                     # belge 10.2: anormal piksel
                rho = 2 * R_AY * math.tan(math.radians(90 - e) / 2)
                x, y = rho * math.sin(math.radians(b)), rho * math.cos(math.radians(b))   # belge 6.8: x = ρ sin λ, y = ρ cos λ
                yarim = n * 240 / 2
                ozel = [(int(math.floor((yarim - y) / 240)), int(math.floor((x + yarim) / 240)))]
            cikti = f"{ad.lower()}.tif"
            b = tif_cevir(k, p / cikti, sf, of)
            if ozel:
                import rasterio
                from rasterio.windows import Window
                r, c = ozel[0]
                with rasterio.open(k) as s:
                    pen = s.read(1, window=Window(c - 2, r - 2, 5, 5)).astype(float) * sf + of
                with rasterio.open(p / cikti, "r+") as t:
                    once = float(t.read(1, window=Window(c, r, 1, 1))[0, 0])
                    t.write(np.array([[np.nan]], dtype=np.float32), 1, window=Window(c, r, 1, 1))
                b["anormal_piksel"] = {"enlem": 88.435, "boylam": 106.602, "satir": r, "sutun": c, "ham_deger_olcekli": once,
                                       "5x5_komsu_ortanca": float(np.median(np.delete(pen.ravel(), 12)))}
                b["veri_yok_sayisi"] += 0 if np.isnan(once) else 1
                print(f"    {ad} anormal piksel ({r}, {c}): değer {once}, komşu ortancası {b['anormal_piksel']['5x5_komsu_ortanca']}", flush=True)
            f_, n_nan = tif_dogrula(k, p / cikti, b, rng, 1000, ozel_nan=ozel)
            okuma[cikti] = b
            sinav("M7", f"{cikti} rastgele piksel fark 0", f"1000 piksel{' + anormal piksel' if ozel else ''}, {f_} farkli", f_ == 0, d3)
            sinav("M7", f"{cikti} NaN sayisi = veri-yok sayisi", f"NaN {n_nan}, beklenen {b['veri_yok_sayisi']}", n_nan == b["veri_yok_sayisi"], d3)
            sinav("M7", f"{cikti} deger araligi [0, 1]", f"min {b['min']}, max {b['max']}", b["min"] >= 0 and b["max"] <= 1 + 1e-6, d3)
        meta_yaz(p, "M7", {"crs": "dosyadaki CRS korundu; belge 10.2: kuzey CRS'i yanlış — konum x = ρ sin λ, y = ρ cos λ "
                                  "(ρ = 2R tan(kolatitüd/2), R = 1737,4 km) ile hesaplanır; D2'de doğrulandı",
                           "cozunurluk": "240 m, 65° kutup", "olcek": {"AVGVISIB": 0.00004, "LPSR": 0.000025},
                           "ofset": {"AVGVISIB": 0.0, "LPSR": 0.5}, "olcek_kaynagi": "etiket SCALING_FACTOR / OFFSET",
                           "veri_yok_kurali": "dosya nodata -> NaN; anormal piksel 88,435°K 106,602°D (kuzey) -> NaN (belge 10.2)",
                           "okuma": okuma, "dosyalar": "avgvisib_65s.tif, avgvisib_65n.tif, lpsr_65s.tif, lpsr_65n.tif"}, d3, D1, IND, D2)

    bolum("M7", m7)

    # ------------------------------------------------ kayıt
    eski = []
    if D3CSV.exists():
        eski = [r for r in csv_oku(D3CSV) if not (r["govde"] == "ay" and r["set"] in calisan)]
    with open(D3CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["govde", "set", "test", "olculen", "durum", "zaman"])
        w.writeheader()
        w.writerows(eski + SATIRLAR)
    say = {}
    for s in SATIRLAR:
        say[s["durum"]] = say.get(s["durum"], 0) + 1
    boyut = sum(f.stat().st_size for f in (TEMIZ / "ay").rglob("*") if f.is_file()) / 1e9
    with open(KAYIT / "13_kasim_log.txt", "a", encoding="utf-8") as f:
        f.write(f"{datetime.now():%Y-%m-%d %H:%M:%S}  D3 ay {calisan}: {say}, {round((time.time() - t0) / 60, 1)} dk, temiz/ay {boyut:.2f} GB\n")
    print("=" * 60)
    print(f"D3 Ay {calisan}: {say} | {round((time.time() - t0) / 60, 1)} dk | temiz\\ay {boyut:.2f} GB | kayit/D3.csv")
    return say
'''
compile(kod, "d3_ay.py", "exec")
dosya = KOK / "kod" / "d3_ay.py"
dosya.write_text(kod, encoding="utf-8")
print("Yazıldı:", dosya, "\n")
g = runpy.run_path(str(dosya), run_name="d3_ay")
_ = g["main"]()

Yazıldı: C:\Users\USER\orbitrary\kod\d3_ay.py 


--- M1 başladı 02:03:23
[FAIL] M1 bolum calisamadi: AssertionError: Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif: dosyada ölçek/ofset 0.5/0.0
--- M1 bitti, 0.0 dk

--- M2 başladı 02:03:23
[FAIL] M2 bolum calisamadi: AssertionError: AREA_GEO toplamı / 4πR² (km²) = 1000000.7710813285: birim km² değil ya da kapsam eksik
--- M2 bitti, 0.0 dk

--- M3 başladı 02:03:24
[PASS] M3 ra_sam.tif rastgele piksel fark 0: 1000 piksel, 0 farkli
[PASS] M3 ra_sam.tif NaN sayisi = veri-yok sayisi: NaN 1392381, veri-yok 1392381
    ra_sam.tif: min 9.77237206711834e-08, max 0.9440608620643616
[PASS] M3 tbol_sam_K.tif rastgele piksel fark 0: 1000 piksel, 0 farkli
[PASS] M3 tbol_sam_K.tif NaN sayisi = veri-yok sayisi: NaN 4367, veri-yok 4367
    tbol_sam_K.tif: min 47.47753143310547, max 239.2853546142578
[PASS] M3 tbol gecerli < 250 K: max 239.2853546142578
--- M3 bitti, 2.2 dk

--- M4 başladı 02:05:34
[PASS] M4 PRP north talimat 6 sayilari D2 ile ayni: {'n_0':

In [27]:
# Developed with assistance from Claude (Anthropic).
# Adım D3 (Ay) yaması — M1/M7: dosyadaki GDAL ölçeği etiketle aynıysa kabul (ölçek bir kez uygulanır);
# M2: AREA_GEO birimi m² (toplamı 4πR² m²'ye eşit) -> alan_km2 = AREA_GEO / 1e6. Yalnız M1, M2, M7 yeniden çalışır.
from pathlib import Path
import runpy

KOK = Path.home() / "orbitrary"
dosya = KOK / "kod" / "d3_ay.py"
s = dosya.read_text(encoding="utf-8")

DEGISIM = [
    ('''        assert s.scales[0] == 1.0 and s.offsets[0] == 0.0, f"{kaynak.name}: dosyada ölçek/ofset {s.scales[0]}/{s.offsets[0]}"
''',
     '''        dosya_ol = (s.scales[0], s.offsets[0])
        # rasterio ham DN döndürür; ölçek yalnız burada, bir kez uygulanır. Dosyada ölçek varsa etiketteki değerle aynı olmalı.
        assert dosya_ol in ((1.0, 0.0), (olcek, ofset)), f"{kaynak.name}: dosyada ölçek/ofset {dosya_ol}, uygulanacak {(olcek, ofset)}"
'''),
    ('''                "sabit_veri_yok": list(sabit_yok), "veri_yok_sayisi": n_yok, "min": vmin, "max": vmax}''',
     '''                "sabit_veri_yok": list(sabit_yok), "veri_yok_sayisi": n_yok, "min": vmin, "max": vmax,
                "dosyadaki_gdal_olcek_ofset": list(dosya_ol)}'''),
    ('''        oran = float(df["AREA_GEO"].sum()) / (4 * math.pi * (R_AY / 1000) ** 2)
        assert 0.99 <= oran <= 1.01, f"AREA_GEO toplamı / 4πR² (km²) = {oran}: birim km² değil ya da kapsam eksik"
''',
     '''        oran = float(df["AREA_GEO"].sum()) / (4 * math.pi * R_AY ** 2)
        assert abs(oran - 1) <= 1e-3, f"AREA_GEO toplamı / 4πR² (m²) = {oran}: birim m² değil ya da kapsam eksik"
'''),
    ('''"alan_km2": df["AREA_GEO"].astype(float)},''', '''"alan_km2": df["AREA_GEO"].astype(float) / 1e6},'''),
    ('''a_ok = bool((geri["alan_km2"].to_numpy()[i] == df["AREA_GEO"].astype(float).to_numpy()[i]).all())''',
     '''a_ok = bool((geri["alan_km2"].to_numpy()[i] == df["AREA_GEO"].astype(float).to_numpy()[i] / 1e6).all())'''),
    ('''"alan_km2": "AREA_GEO (belge 10.2); toplam / 4πR² = %.6f -> birim km²" % oran,''',
     '''"alan_km2": "AREA_GEO (belge 10.2) birimi m² (toplam / 4πR² m² = %.6f); alan_km2 = AREA_GEO / 1e6" % oran,'''),
]
for eski, yeni in DEGISIM:
    assert s.count(eski) == 1, f"işaret bulunamadı: {eski[:70]!r}"
    s = s.replace(eski, yeni)
compile(s, "d3_ay.py", "exec")
dosya.write_text(s, encoding="utf-8")
print("Yamandı:", dosya, "\n")
g = runpy.run_path(str(dosya), run_name="d3_ay")
_ = g["main"](["M1", "M2", "M7"])

Yamandı: C:\Users\USER\orbitrary\kod\d3_ay.py 


--- M1 başladı 02:08:34
[PASS] M1 rastgele piksel fark 0: 1000 piksel, 0 farkli
[PASS] M1 NaN sayisi = veri-yok sayisi: NaN 0, veri-yok 0
[PASS] M1 uc degerler D2 ile ayni: min -9128.5, max 10781.5 (D2: -9128,5 / 10781,5)
--- M1 bitti, 4.8 dk

--- M2 başladı 02:13:24
[PASS] M2 rastgele 1000 poligon: birim ve alan_km2 ham ile ayni: 12247 kayit (ham 12247); birim True; alan True; AREA_GEO toplami / 4πR² = 1.000001
[PASS] M2 bos birim sayisi (kayit): 0
--- M2 bitti, 0.0 dk

--- M7 başladı 02:13:26
[PASS] M7 avgvisib_65s.tif rastgele piksel fark 0: 1000 piksel, 0 farkli
[PASS] M7 avgvisib_65s.tif NaN sayisi = veri-yok sayisi: NaN 0, beklenen 0
[PASS] M7 avgvisib_65s.tif deger araligi [0, 1]: min 0.0, max 0.9347599744796753
    AVGVISIB_65N anormal piksel (3266, 3399): değer 0.2800000011920929, komşu ortancası 0.28232
[PASS] M7 avgvisib_65n.tif rastgele piksel fark 0: 1000 piksel + anormal piksel, 0 farkli
[PASS] M7 avgvisib_65n.tif NaN sayis

In [28]:
# Developed with assistance from Claude (Anthropic).
# Adım D3 (Mars) — kod\d3_mars.py yazılır ve çalıştırılır (belge 17.3 D3, 10.4).
from pathlib import Path
import runpy

KOK = Path.home() / "orbitrary"
kod = r'''# Developed with assistance from Claude (Anthropic).
# d3_mars.py — Orbitrary v1.2, belge 17.3 D3 + 10.4: Mars setleri R1–R7 temiz\mars\<kod>_<ad>\ altına standart biçimde yazılır.
# Raster: fiziksel birim, float32 (GeoTIFF ya da .npy + .json), veri yok = NaN. Tablo: .npy. Vektör: GeoPackage. Her klasöre meta.json.
# TES TI (R4) kuzey üstte, boylam -180'den doğuya düzene çevrilir (albedo/DCI/TES ile aynı ızgara düzeni); dönüşüm testle doğrulanır.
# D3 testi (belge 17.3): rastgele 1000 piksel/satırda ham dosyadan yeniden hesaplanan değerle fark 0; NaN sayısı = veri-yok sayısı.
# Okuma kuralları d2_mars.py ile aynıdır (D2'de geçti). Ham veri (veri\) yalnız okunur.
import csv, json, math, re, shutil, time
from datetime import datetime
from pathlib import Path

import numpy as np

KOK = Path.home() / "orbitrary"
VERI, KAYIT, TEMIZ = KOK / "veri", KOK / "kayit", KOK / "temiz"
M = VERI / "mars"
D3CSV = KAYIT / "D3.csv"
TOHUM = 20261113
R_MARS = 3396190.0                     # m; SIM3292_README.txt

STATIK = {
    "R1": {"ad": "MGS MOLA DEM 463 m (128 ppd)", "doi": "10.1029/2000JE001364 (Smith ve ark. 2001)", "surum": "Edition 2003-03-21",
           "lisans": "CC0 (rapor Bölüm 4)", "birim": "m (areoid)"},
    "R2": {"ad": "Geologic Map of Mars SIM 3292", "doi": "10.3133/sim3292", "surum": "Tanaka ve ark. 2014, 1:20M",
           "lisans": "USGS yayını, kamu malı (rapor Bölüm 4)", "birim": "sınıf (Unit), km²"},
    "R3": {"ad": "TES mineral haritaları 4 ppd (Bandfield 2002)", "doi": "10.1029/2001JE001510 (makale)", "surum": "TES_mineral_maps_4_ppd",
           "lisans": "sayfada yok — 'açık' (rapor Bölüm 4)", "birim": "bolluk (oran)"},
    "R4": {"ad": "TES termal eylemsizlik 20 ppd (Putzig) + TES albedo 8 ppd", "doi": "Putzig ve ark. 2005, Icarus 173; Christensen ve ark. 2001",
           "surum": "ti_nmap2003.bin; global_albedo_8ppd", "lisans": "sayfada yok — 'açık' (rapor Bölüm 4)",
           "birim": "TI: J m⁻² K⁻¹ s⁻½; albedo: birimsiz"},
    "R5": {"ad": "TES Dust Cover Index 16 ppd", "doi": "10.1029/2001JE001580 (Ruff & Christensen 2002)", "surum": "dci_lo_ice_dust_16ppd_shifted",
           "lisans": "sayfada yok — 'açık' (rapor Bölüm 4)", "birim": "yayınım (1350–1400 cm⁻¹ ortalaması)"},
    "R6": {"ad": "Mars Odyssey GRS 5° x 5° yumuşatılmış", "doi": "ODY-M-GRS-5-ELEMENTS-V1.0 (PDS4'e taşınmış)", "surum": "V1.0",
           "lisans": "NASA PDS, kamuya açık (rapor Bölüm 4)", "birim": "ağırlık % (H2O, Fe, Si, Cl), ppm (K, Th) — PDS3 etiketleri"},
    "R7": {"ad": "SWIM4MIM Ci (0–1 m, 1–5 m, > 5 m)", "doi": "açık (rapor R7 künyesi)", "surum": "SWIM4MIM",
           "lisans": "sayfada yok — 'açık' (rapor Bölüm 4)", "birim": "tutarlılık indeksi -1…+1 (olasılık değil)"},
}

SATIRLAR = []

def csv_oku(yol):
    with open(yol, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def sinav(set_, test, olculen, ok, d3):
    durum = "PASS" if ok else "FAIL"
    d3.append({"test": test, "olculen": olculen, "durum": durum})
    SATIRLAR.append({"govde": "mars", "set": set_, "test": test, "olculen": olculen, "durum": durum,
                     "zaman": datetime.now().isoformat(timespec="seconds")})
    print(f"[{durum}] {set_} {test}: {olculen}", flush=True)


def ham_klasor(kod):
    k = [p for p in (VERI / "mars").glob(f"{kod}_*") if p.is_dir()]
    assert len(k) == 1, f"{kod}: ham klasör {k}"
    return k[0]


def klasor(kod, ad):
    p = TEMIZ / "mars" / f"{kod}_{ad}"
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True)
    return p


def meta_yaz(p, kod, ek, d3, D1, IND, D2):
    ind = [r for r in IND if r["set"] == kod and r["durum"] != "FAIL"]
    adres = sorted({r["adres"] for r in ind if r["adres"]})
    m = {"set": kod, **STATIK[kod],
         "kaynak_adresleri": adres if len(adres) <= 8 else adres[:5] + [f"... toplam {len(adres)} adres (kayit/indirme.csv)"],
         "indirme_tarihi": [min(r["zaman"] for r in ind), max(r["zaman"] for r in ind)] if ind else None,
         "ham_sha256": {r["hedef"]: r["sha256"] for r in D1 if r["set"] == kod},
         "gercek_yer_testleri_D2": [{k: r[k] for k in ("test", "nokta", "beklenen", "tolerans_kurali", "beklenen_kaynagi", "olculen", "durum")}
                                    for r in D2 if r["govde"] == "mars" and r["set"] == kod],
         "D3_testleri": d3, **ek, "uretim_zamani": datetime.now().isoformat(timespec="seconds"), "uretim_kodu": "kod/d3_mars.py"}
    (p / "meta.json").write_text(json.dumps(m, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


def esit(bek, tem):
    return (bek == tem) | (np.isnan(bek) & np.isnan(tem))


def npy_yaz(p, ad, v, bilgi):
    np.save(p / f"{ad}.npy", v)
    (p / f"{ad}.json").write_text(json.dumps({**bilgi, "sekil": list(v.shape), "dtype": str(v.dtype)},
                                             ensure_ascii=False, indent=2, default=str), encoding="utf-8")


# ---------------------------------------------------------------- GeoTIFF -> float32 GeoTIFF (NaN)
def _yok(a, v, nd, sabit_yok, ek_yok):
    y = np.zeros(a.shape, bool)
    if nd is not None:
        y |= np.isnan(a) if (isinstance(nd, float) and math.isnan(nd)) else (a == nd)
    for s in sabit_yok:
        y |= a == s
    if a.dtype.kind == "f":
        y |= ~np.isfinite(a)
    if ek_yok is not None:
        with np.errstate(invalid="ignore"):
            y |= ek_yok(v)
    return y


def tif_cevir(kaynak, hedef, olcek=1.0, ofset=0.0, sabit_yok=(), ek_yok=None, satir=256):
    import rasterio
    from rasterio.windows import Window
    with rasterio.open(kaynak) as s:
        assert s.count == 1, f"{kaynak.name}: {s.count} bant"
        dosya_ol = (s.scales[0], s.offsets[0])
        # rasterio ham DN döndürür; ölçek yalnız burada, bir kez uygulanır. Dosyada ölçek varsa etiketteki değerle aynı olmalı.
        assert dosya_ol in ((1.0, 0.0), (olcek, ofset)), f"{kaynak.name}: dosyada ölçek/ofset {dosya_ol}, uygulanacak {(olcek, ofset)}"
        nd = s.nodata
        prof = s.profile.copy()
        prof.update(driver="GTiff", dtype="float32", nodata=np.nan, count=1, compress="deflate", predictor=3,
                    tiled=True, blockxsize=512, blockysize=512, BIGTIFF="YES")
        n_yok, vmin, vmax = 0, np.inf, -np.inf
        with rasterio.open(hedef, "w", **prof) as t:
            for r0 in range(0, s.height, satir):
                w = Window(0, r0, s.width, min(satir, s.height - r0))
                a = s.read(1, window=w)
                v = (a.astype(np.float64) * olcek + ofset).astype(np.float32)
                yok = _yok(a, v, nd, sabit_yok, ek_yok)
                v[yok] = np.nan
                n_yok += int(yok.sum())
                if (~yok).any():
                    vmin, vmax = min(vmin, float(np.nanmin(v))), max(vmax, float(np.nanmax(v)))
                t.write(v, 1, window=w)
        return {"ham_dtype": s.dtypes[0], "ham_nodata": nd, "boyut": [s.width, s.height], "crs": str(s.crs),
                "transform": list(s.transform)[:6], "piksel": list(s.res), "olcek": olcek, "ofset": ofset,
                "sabit_veri_yok": list(sabit_yok), "veri_yok_sayisi": n_yok, "min": vmin, "max": vmax,
                "dosyadaki_gdal_olcek_ofset": list(dosya_ol)}


def tif_dogrula(kaynak, hedef, bilgi, rng, n, sabit_yok=(), ek_yok=None, ozel_nan=()):
    import rasterio
    from rasterio.windows import Window
    with rasterio.open(kaynak) as s, rasterio.open(hedef) as t:
        assert (s.width, s.height, s.transform) == (t.width, t.height, t.transform), f"{hedef.name}: geometri farklı"
        assert t.dtypes[0] == "float32"
        rr = list(rng.integers(0, s.height, n)) + [r for r, c in ozel_nan]
        cc = list(rng.integers(0, s.width, n)) + [c for r, c in ozel_nan]
        ham = np.array([s.read(1, window=Window(int(c), int(r), 1, 1))[0, 0] for r, c in zip(rr, cc)])
        tem = np.array([t.read(1, window=Window(int(c), int(r), 1, 1))[0, 0] for r, c in zip(rr, cc)], dtype=np.float32)
        bek = (ham.astype(np.float64) * bilgi["olcek"] + bilgi["ofset"]).astype(np.float32)
        bek[_yok(ham, bek, s.nodata, sabit_yok, ek_yok)] = np.nan
        if ozel_nan:
            bek[-len(ozel_nan):] = np.nan
        n_fark = int((~esit(bek, tem)).sum())
        n_nan = 0
        for r0 in range(0, t.height, 512):
            n_nan += int(np.isnan(t.read(1, window=Window(0, r0, t.width, min(512, t.height - r0)))).sum())
    return n_fark, n_nan


def main(setler=None):
    t0 = time.time()
    D1 = csv_oku(KAYIT / "D1_dosyalar.csv")
    IND = csv_oku(KAYIT / "indirme.csv")
    D2 = csv_oku(KAYIT / "D2.csv")
    secili = set(setler) if setler else set(STATIK)
    rng = np.random.default_rng(TOHUM)
    calisan = []

    def bolum(kod, f):
        if kod not in secili:
            return
        calisan.append(kod)
        t1 = time.time()
        print(f"\n--- {kod} başladı {datetime.now():%H:%M:%S}", flush=True)
        try:
            f()
        except Exception as e:
            sinav(kod, "bolum calisamadi", f"{type(e).__name__}: {e}", False, [])
        print(f"--- {kod} bitti, {round((time.time() - t1) / 60, 1)} dk", flush=True)

    # ------------------------------------------------ R1 MOLA: int16 m -> float32 GeoTIFF (konum kuralı belge 10.3)
    def r1():
        h = ham_klasor("R1")
        p = klasor("R1", "mola")
        k = h / "Mars_MGS_MOLA_DEM_mosaic_global_463m.tif"
        d3 = []
        b = tif_cevir(k, p / "mola_m.tif", 1.0, 0.0)
        f_, n_nan = tif_dogrula(k, p / "mola_m.tif", b, rng, 1000)
        sinav("R1", "rastgele piksel fark 0", f"1000 piksel, {f_} farkli", f_ == 0, d3)
        sinav("R1", "NaN sayisi = veri-yok sayisi", f"NaN {n_nan}, veri-yok {b['veri_yok_sayisi']}", n_nan == b["veri_yok_sayisi"], d3)
        sinav("R1", "uc degerler D2 ile ayni", f"min {b['min']}, max {b['max']} (D2: -8201 / 21241)", b["min"] == -8201 and b["max"] == 21241, d3)
        meta_yaz(p, "R1", {"crs": b["crs"], "cozunurluk": "128 ppd (463 m)", "olcek": 1.0, "ofset": 0.0,
                           "konum_kurali": "transform kullanılmaz: satır = ⌊(90 − φ)·128⌋, sütun = ⌊(λ + 180)·128⌋ (belge 10.3; D2 Jezero ile doğrulandı)",
                           "veri_yok_kurali": "dosya nodata -> NaN", "okuma": b, "dosyalar": "mola_m.tif"}, d3, D1, IND, D2)

    bolum("R1", r1)

    # ------------------------------------------------ R2 SIM 3292 -> GeoPackage (birim, alan_km2)
    def r2():
        import pyogrio, geopandas as gpd
        p = klasor("R2", "sim3292")
        shp = M / "R2_sim3292/acik/SIM3292_MarsGlobalGeologicGIS_20M/SIM3292_Shapefiles/SIM3292_Global_Geology.shp"
        df = pyogrio.read_dataframe(shp, columns=["Unit", "SphArea_km"])
        oran = float(df["SphArea_km"].sum()) / (4 * math.pi * (R_MARS / 1000) ** 2)
        assert len(df) == 1311 and df["Unit"].nunique() == 44 and abs(oran - 1) <= 1e-4, (len(df), df["Unit"].nunique(), oran)
        out = gpd.GeoDataFrame({"birim": df["Unit"].astype("string"), "alan_km2": df["SphArea_km"].astype(float)},
                               geometry=df.geometry, crs=df.crs)
        gp = p / "sim3292.gpkg"
        pyogrio.write_dataframe(out, gp, layer="sim3292", driver="GPKG")
        geri = pyogrio.read_dataframe(gp, layer="sim3292")
        i = rng.integers(0, len(df), 1000)
        b_ok = bool((geri["birim"].astype("string").iloc[i].fillna("<bos>").to_numpy()
                     == df["Unit"].astype("string").iloc[i].fillna("<bos>").to_numpy()).all())
        a_ok = bool((geri["alan_km2"].to_numpy()[i] == df["SphArea_km"].astype(float).to_numpy()[i]).all())
        d3 = []
        sinav("R2", "rastgele 1000 poligon: birim ve alan_km2 ham ile ayni",
              f"{len(geri)} kayit, {geri['birim'].nunique()} birim; birim {b_ok}; alan {a_ok}; SphArea_km toplami / 4πR² = {oran:.6f}",
              len(geri) == len(df) and b_ok and a_ok, d3)
        meta_yaz(p, "R2", {"crs": str(df.crs), "olcek": None, "ofset": None, "veri_yok_kurali": "vektör",
                           "alan_km2": "SphArea_km (README: geodesic area, km²); toplam / 4πR² = %.6f" % oran,
                           "kayit_sayisi": len(df), "birim_sayisi": int(df["Unit"].nunique()),
                           "dosyalar": "sim3292.gpkg (birim, alan_km2)"}, d3, D1, IND, D2)

    bolum("R2", r2)

    # ------------------------------------------------ R3 TES mineral: 13 bant -> 13 .npy (-1 = yok)
    def r3():
        p = klasor("R3", "tes_mineral")
        hdr = (M / "R3_tes_mineral/TES_mineral_maps_4_ppd.envi.hdr").read_text(encoding="ascii", errors="replace")
        hh = {k: int(re.search(r"(?im)^\s*" + k + r"\s*=\s*(\d+)", hdr).group(1)) for k in ("samples", "lines", "bands", "data type", "byte order")}
        assert hh == {"samples": 1440, "lines": 720, "bands": 13, "data type": 4, "byte order": 1}, hh
        kaynak = M / "R3_tes_mineral/TES_mineral_maps_4_ppd.envi"
        A = np.fromfile(kaynak, dtype=">f4").reshape(13, 720, 1440)
        adlar = ["quartz", "k_feldspar", "plagioclase", "amphibole", "low_ca_pyroxene", "high_ca_pyroxene", "olivine",
                 "hematite", "sulfates", "carbonates", "sheet_silicates_high_si_glass", "surface_dust", "rms_error"]
        GT = [-180.0, 0.25, 0.0, 90.0, 0.0, -0.25]
        d3, n_fark, nan_hata = [], 0, []
        for j, ad in enumerate(adlar):
            ham = A[j]
            v = ham.astype(np.float32)
            yok = (ham == -1) | ~np.isfinite(ham)
            v[yok] = np.nan
            dosya = f"b{j + 1:02d}_{ad}"
            npy_yaz(p, dosya, v, {"kaynak": "mars/R3_tes_mineral/TES_mineral_maps_4_ppd.envi", "bant": j + 1, "bilesen": ad,
                                  "geotransform": GT, "veri_yok": "-1 -> NaN (belge 10.3)", "dtype_ham": ">f4"})
            # bağımsız okuma: aynı dosyadan yalnız bu bandın baytları offset ile okunur
            bag = np.fromfile(kaynak, dtype=">f4", count=720 * 1440, offset=j * 720 * 1440 * 4).reshape(720, 1440)
            rr, cc = rng.integers(0, 720, 1000), rng.integers(0, 1440, 1000)
            h = bag[rr, cc].astype(np.float32)
            h[(bag[rr, cc] == -1) | ~np.isfinite(bag[rr, cc])] = np.nan
            t = np.load(p / f"{dosya}.npy")
            n_fark += int((~esit(h, t[rr, cc])).sum())
            if int(np.isnan(t).sum()) != int(((bag == -1) | ~np.isfinite(bag)).sum()):
                nan_hata.append(dosya)
        sinav("R3", "rastgele piksel fark 0", f"13 bant x 1000 piksel, {n_fark} farkli", n_fark == 0, d3)
        sinav("R3", "NaN sayisi = veri-yok (-1) sayisi", f"tutmayan bant {nan_hata}", not nan_hata, d3)
        meta_yaz(p, "R3", {"crs": "Mars 'ocentric, simple cylindrical", "geotransform": GT,
                           "geometri_kaynagi": "dosyalarda yok; ASU Mars Global Data Sets (TES Low-Calcium Pyroxene): "
                                               "Simple cylindrical, -180E to 180E, 90N to -90N; D2 hematit testiyle doğrulandı",
                           "bant_sirasi_kaynagi": "ASU ürün sayfası (rapor R3) = belge D11", "olcek": 1.0, "ofset": 0.0,
                           "bayt_sirasi": ".envi >f4 (hdr byte order 1); D2'de .isis <f4 ile birebir aynı",
                           "veri_yok_kurali": "-1 ve sonlu olmayan -> NaN", "dosyalar": "b01_quartz.npy … b13_rms_error.npy"}, d3, D1, IND, D2)

    bolum("R3", r3)

    # ------------------------------------------------ R4 TI (yeniden düzenlenir) + albedo
    def r4():
        p = klasor("R4", "tes_ti_albedo")
        d3 = []
        kaynak = M / "R4_tes_ti_albedo/ti_nmap2003.bin"
        bas = kaynak.open("rb").read(7200).decode("ascii", errors="replace")
        assert "west" in bas.lower() and "-90" in bas
        assert kaynak.stat().st_size == 7200 + 7200 * 3600 * 2
        ham = np.fromfile(kaynak, dtype=">i2", offset=7200).reshape(3600, 7200)      # ilk satır güney, boylam batıya 0-360
        yeni = np.roll(ham[::-1, ::-1], -3600, axis=1)                               # -> kuzey üstte, boylam -180'den doğuya
        v = yeni.astype(np.float32)
        v[yeni == 0] = np.nan                                                         # 0 = yok (başlık)
        GT_TI = [-180.0, 0.05, 0.0, 90.0, 0.0, -0.05]
        npy_yaz(p, "ti", v, {"kaynak": "mars/R4_tes_ti_albedo/ti_nmap2003.bin", "geotransform": GT_TI, "birim": "J m-2 K-1 s-1/2",
                             "veri_yok": "ham 0 -> NaN", "duzen": "ham: ilk satır -90, boylam 0–360 batı; temiz: ilk satır +90, boylam -180'den doğuya"})
        # bağımsız test: rastgele ham piksel (i, j) -> enlem/boylam -> temiz satır/sütun formülle hesaplanır
        i, j = rng.integers(0, 3600, 1000), rng.integers(0, 7200, 1000)
        enlem = -90 + (i + 0.5) / 20
        boylam_d = ((-(j + 0.5) / 20) + 180) % 360 - 180
        r2_, c2_ = np.floor((90 - enlem) * 20).astype(int), np.floor((boylam_d + 180) * 20).astype(int)
        hv = ham[i, j].astype(np.float32)
        hv[ham[i, j] == 0] = np.nan
        t = np.load(p / "ti.npy")
        n_fark = int((~esit(hv, t[r2_, c2_])).sum())
        n_nan = int(np.isnan(t).sum())
        sinav("R4", "TI rastgele piksel fark 0 (yeniden düzenleme dahil)", f"1000 piksel, {n_fark} farkli", n_fark == 0, d3)
        sinav("R4", "TI NaN sayisi = veri-yok (0) sayisi", f"NaN {n_nan}, 0 {int((ham == 0).sum())}", n_nan == int((ham == 0).sum()), d3)
        # albedo: etiketten (PC_REAL = <f4, kuzey üstte, -180'den doğuya); veri-yok tanımı yok -> sonlu olmayan ve <= 0 NaN (D2 ile aynı proje kararı)
        lbl = (M / "R4_tes_ti_albedo/global_albedo_8ppd.lbl").read_text(encoding="ascii", errors="replace")
        for anahtar, bek in (("LINES", "1440"), ("LINE_SAMPLES", "2880"), ("SAMPLE_TYPE", "PC_REAL"), ("SAMPLE_BITS", "32"),
                             ("MAXIMUM_LATITUDE", "90.0"), ("WESTERNMOST_LONGITUDE", "-180.0")):
            assert re.search(r"(?m)^\s*" + anahtar + r"\s*=\s*\"?" + re.escape(bek), lbl), anahtar
        akaynak = M / "R4_tes_ti_albedo/global_albedo_8ppd.img"
        a = np.fromfile(akaynak, dtype="<f4").reshape(1440, 2880)
        va = a.astype(np.float32)
        ayok = ~np.isfinite(a) | (a <= 0)
        va[ayok] = np.nan
        npy_yaz(p, "albedo", va, {"kaynak": "mars/R4_tes_ti_albedo/global_albedo_8ppd.img", "geotransform": [-180.0, 0.125, 0.0, 90.0, 0.0, -0.125],
                                  "birim": "bolometrik albedo", "veri_yok": "sonlu olmayan ve <= 0 -> NaN (etikette veri-yok yok; proje kararı, D2 ile aynı)"})
        bag = np.fromfile(akaynak, dtype="<f4").reshape(1440, 2880)
        rr, cc = rng.integers(0, 1440, 1000), rng.integers(0, 2880, 1000)
        hb = bag[rr, cc].astype(np.float32)
        hb[~np.isfinite(bag[rr, cc]) | (bag[rr, cc] <= 0)] = np.nan
        ta = np.load(p / "albedo.npy")
        sinav("R4", "albedo rastgele piksel fark 0", f"1000 piksel, {int((~esit(hb, ta[rr, cc])).sum())} farkli", bool(esit(hb, ta[rr, cc]).all()), d3)
        sinav("R4", "albedo NaN sayisi = veri-yok sayisi", f"NaN {int(np.isnan(ta).sum())}, beklenen {int(ayok.sum())}",
              int(np.isnan(ta).sum()) == int(ayok.sum()), d3)
        meta_yaz(p, "R4", {"crs": "Mars 'ocentric, simple cylindrical", "geotransform": {"ti": GT_TI, "albedo": [-180.0, 0.125, 0.0, 90.0, 0.0, -0.125]},
                           "olcek": 1.0, "ofset": 0.0, "ti_okuma": "ham >i2, 7200 bayt başlık, ilk satır güney, boylam 0–360 batı (başlık + rapor R4); "
                           "temizde kuzey üstte, -180'den doğuya", "ti_baslik": bas.strip()[:600],
                           "veri_yok_kurali": "TI 0 -> NaN; albedo sonlu olmayan ve <= 0 -> NaN",
                           "eksik": "interpolasyon maskesi nmsk2003.bin indirilemedi (belge 13.3 açık kontrol)",
                           "dosyalar": "ti.npy (3600 x 7200), albedo.npy (1440 x 2880)"}, d3, D1, IND, D2)

    bolum("R4", r4)

    # ------------------------------------------------ R5 DCI: 0,85 = yok
    def r5():
        p = klasor("R5", "tes_dci")
        kaynak = M / "R5_tes_dci/dci_lo_ice_dust_16ppd_shifted.pds"
        etk = kaynak.open("rb").read(1024).decode("ascii", errors="replace")
        assert re.search(r"CORE_ITEMS\s*=\s*\(\s*1\s*,\s*5760\s*,\s*2880\s*\)", etk) and "PC_REAL" in etk
        assert re.search(r"\^QUBE\s*=\s*3", etk) and re.search(r"RECORD_BYTES\s*=\s*512", etk)
        d = np.fromfile(kaynak, dtype="<f4", offset=1024).reshape(2880, 5760)
        v = d.astype(np.float32)
        yok = (d == np.float32(0.85)) | ~np.isfinite(d)
        v[yok] = np.nan
        GT = [-180.0, 1 / 16, 0.0, 90.0, 0.0, -1 / 16]
        npy_yaz(p, "dci", v, {"kaynak": "mars/R5_tes_dci/dci_lo_ice_dust_16ppd_shifted.pds", "geotransform": GT,
                              "veri_yok": "0,85 -> NaN (belge 10.3, veriden)"})
        d3 = []
        bag = np.memmap(kaynak, dtype="<f4", mode="r", offset=1024, shape=(2880, 5760))
        rr, cc = rng.integers(0, 2880, 1000), rng.integers(0, 5760, 1000)
        h = np.asarray(bag[rr, cc]).astype(np.float32)
        h[(h == np.float32(0.85)) | ~np.isfinite(h)] = np.nan
        t = np.load(p / "dci.npy")
        sinav("R5", "rastgele piksel fark 0", f"1000 piksel, {int((~esit(h, t[rr, cc])).sum())} farkli", bool(esit(h, t[rr, cc]).all()), d3)
        sinav("R5", "NaN sayisi = veri-yok (0,85) sayisi", f"NaN {int(np.isnan(t).sum())}, 0,85 {int(yok.sum())}", int(np.isnan(t).sum()) == int(yok.sum()), d3)
        meta_yaz(p, "R5", {"crs": "Mars 'ocentric, simple cylindrical", "geotransform": GT, "olcek": 1.0, "ofset": 0.0,
                           "okuma": "PDS3 QUBE: CORE_ITEMS (1,5760,2880), PC_REAL (<f4), veri 1025. bayttan (^QUBE = 3 x 512)",
                           "geometri_kaynagi": "etikette yok; rapor R5 (ikincil kayıt): -180 -> 180 doğu, 90 -> -90; D2 DCI-albedo korelasyonu -0,84",
                           "veri_yok_kurali": "0,85 -> NaN", "dosyalar": "dci.npy (2880 x 5760)"}, d3, D1, IND, D2)

    bolum("R5", r5)

    # ------------------------------------------------ R6 Odyssey GRS: 6 element tablosu (PDS4 alan konumları)
    def r6():
        p = klasor("R6", "odyssey_grs")
        d3, n_fark, nan_hata, bilgi = [], 0, [], {}
        for el in ("h2o", "fe", "si", "k", "th", "cl"):
            xml = (M / f"R6_odyssey_grs/pds4_etiket/{el}_sr_5x5.xml").read_text(encoding="utf-8", errors="replace")
            rl = int(re.search(r"<record_length[^>]*>(\d+)<", xml).group(1))
            nrec = int(re.search(r"<records>(\d+)<", xml).group(1))
            alan = [(re.search(r"<name>(.*?)</name>", b).group(1).strip(),
                     int(re.search(r"<field_location[^>]*>(\d+)<", b).group(1)),
                     int(re.search(r"<field_length[^>]*>(\d+)<", b).group(1)))
                    for b in re.findall(r"<Field_Character>(.*?)</Field_Character>", xml, re.S)]
            ham = (M / f"R6_odyssey_grs/{el}_sr_5x5.tab").read_bytes()
            assert len(ham) == rl * nrec, (el, len(ham), rl, nrec)
            sat = [ham[k * rl:(k + 1) * rl] for k in range(nrec)]
            T = np.array([[float(s[l - 1:l - 1 + n]) for (_, l, n) in alan] for s in sat])
            v = T.copy()
            v[np.isclose(T, 9999.999)] = np.nan
            v[:, :2] = T[:, :2]                                                     # enlem/boylam sütunlarına dokunulmaz
            lbl = (M / f"R6_odyssey_grs/{el}_sr_5x5.lbl").read_text(encoding="ascii", errors="replace")
            birimler = re.findall(r"UNIT\s*=\s*\"([^\"]+)\"", lbl)
            npy_yaz(p, f"{el}_tablo", v, {"kaynak": f"mars/R6_odyssey_grs/{el}_sr_5x5.tab", "sutunlar": [a[0] for a in alan],
                                          "birimler_pds3": birimler, "veri_yok": "9999.999 -> NaN",
                                          "okuma": f"PDS4 etiketi: record_length {rl}, {nrec} kayıt, alan konumları"})
            t = np.load(p / f"{el}_tablo.npy")
            i = rng.integers(0, nrec, 200)
            for k in i:
                s = ham[k * rl:(k + 1) * rl].decode("ascii")
                h = np.array([float(x) for x in s.split()])                          # bağımsız: boşlukla bölme
                h[2:][np.isclose(h[2:], 9999.999)] = np.nan
                n_fark += int((~esit(h, t[k])).sum()) if h.shape == t[k].shape else 1
            if int(np.isnan(t).sum()) != int(np.isclose(T[:, 2:], 9999.999).sum()):
                nan_hata.append(el)
            bilgi[el] = {"kayit": nrec, "record_length": rl, "sutunlar": [a[0] for a in alan], "birimler_pds3": birimler,
                         "nan": int(np.isnan(t).sum())}
        sinav("R6", "rastgele satir fark 0 (6 element x 200 satir, boslukla bolerek bagimsiz okuma)", f"{n_fark} farkli", n_fark == 0, d3)
        sinav("R6", "NaN sayisi = 9999.999 sayisi", f"tutmayan {nan_hata}", not nan_hata, d3)
        meta_yaz(p, "R6", {"crs": "Mars, 5° x 5° hücre merkezleri (derece, boylam doğuya)", "olcek": None, "ofset": None,
                           "okuma": "PDS4 etiketleri (pds4_etiket/*.xml): 72/66 bayt kayıt; PDS3 .lbl RECORD_BYTES (71/65) eski",
                           "veri_yok_kurali": "9999.999 -> NaN (belge 10.3); H2O yalnız 57,5°G–57,5°K (D2)", "puan": "puana girmez (çapraz kontrol)",
                           "elementler": bilgi, "dosyalar": "<element>_tablo.npy (h2o, fe, si, k, th, cl)"}, d3, D1, IND, D2)

    bolum("R6", r6)

    # ------------------------------------------------ R7 SWIM: 3 GeoTIFF, -30 = yok
    def r7():
        p = klasor("R7", "swim")
        d3, okuma = [], {}
        for ad in ("SWIM4MIM_Ci_0_1", "SWIM4MIM_Ci_1_5", "SWIM4MIM_Ci_5"):
            k = M / "R7_swim" / f"{ad}.tif"
            b = tif_cevir(k, p / f"{ad.lower()}.tif", 1.0, 0.0, sabit_yok=(-30,))
            f_, n_nan = tif_dogrula(k, p / f"{ad.lower()}.tif", b, rng, 1000, sabit_yok=(-30,))
            okuma[ad] = b
            sinav("R7", f"{ad} rastgele piksel fark 0", f"1000 piksel, {f_} farkli", f_ == 0, d3)
            sinav("R7", f"{ad} NaN sayisi = veri-yok sayisi", f"NaN {n_nan}, veri-yok {b['veri_yok_sayisi']}", n_nan == b["veri_yok_sayisi"], d3)
            sinav("R7", f"{ad} deger araligi [-1, 1]", f"min {b['min']}, max {b['max']}", b["min"] >= -1 and b["max"] <= 1, d3)
        meta_yaz(p, "R7", {"crs": okuma["SWIM4MIM_Ci_0_1"]["crs"], "cozunurluk": "20 ppd, 60°G–60°K", "olcek": 1.0, "ofset": 0.0,
                           "veri_yok_kurali": "-30 (GDAL_NODATA) -> NaN", "puan_katmani": "Ci 0–1 m (belge 10.3); 1–5 m ve > 5 m bilgi için",
                           "okuma": okuma, "dosyalar": "swim4mim_ci_0_1.tif, swim4mim_ci_1_5.tif, swim4mim_ci_5.tif"}, d3, D1, IND, D2)

    bolum("R7", r7)

    # ------------------------------------------------ kayıt
    eski = []
    if D3CSV.exists():
        eski = [r for r in csv_oku(D3CSV) if not (r["govde"] == "mars" and r["set"] in calisan)]
    with open(D3CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["govde", "set", "test", "olculen", "durum", "zaman"])
        w.writeheader()
        w.writerows(eski + SATIRLAR)
    say = {}
    for s in SATIRLAR:
        say[s["durum"]] = say.get(s["durum"], 0) + 1
    boyut = sum(f.stat().st_size for f in (TEMIZ / "mars").rglob("*") if f.is_file()) / 1e9
    with open(KAYIT / "13_kasim_log.txt", "a", encoding="utf-8") as f:
        f.write(f"{datetime.now():%Y-%m-%d %H:%M:%S}  D3 mars {calisan}: {say}, {round((time.time() - t0) / 60, 1)} dk, temiz/mars {boyut:.2f} GB\n")
    print("=" * 60)
    print(f"D3 Mars {calisan}: {say} | {round((time.time() - t0) / 60, 1)} dk | temiz\\mars {boyut:.2f} GB | kayit/D3.csv")
    return say
'''
compile(kod, "d3_mars.py", "exec")
dosya = KOK / "kod" / "d3_mars.py"
dosya.write_text(kod, encoding="utf-8")
print("Yazıldı:", dosya, "\n")
g = runpy.run_path(str(dosya), run_name="d3_mars")
_ = g["main"]()

Yazıldı: C:\Users\USER\orbitrary\kod\d3_mars.py 


--- R1 başladı 02:24:05
[PASS] R1 rastgele piksel fark 0: 1000 piksel, 0 farkli
[PASS] R1 NaN sayisi = veri-yok sayisi: NaN 0, veri-yok 0
[PASS] R1 uc degerler D2 ile ayni: min -8201.0, max 21241.0 (D2: -8201 / 21241)
--- R1 bitti, 1.3 dk

--- R2 başladı 02:25:21
[PASS] R2 rastgele 1000 poligon: birim ve alan_km2 ham ile ayni: 1311 kayit, 44 birim; birim True; alan True; SphArea_km toplami / 4πR² = 0.999999
--- R2 bitti, 0.0 dk

--- R3 başladı 02:25:22
[PASS] R3 rastgele piksel fark 0: 13 bant x 1000 piksel, 0 farkli
[PASS] R3 NaN sayisi = veri-yok (-1) sayisi: tutmayan bant []
--- R3 bitti, 0.0 dk

--- R4 başladı 02:25:22
[PASS] R4 TI rastgele piksel fark 0 (yeniden düzenleme dahil): 1000 piksel, 0 farkli
[PASS] R4 TI NaN sayisi = veri-yok (0) sayisi: NaN 2728379, 0 2728379
[PASS] R4 albedo rastgele piksel fark 0: 1000 piksel, 0 farkli
[PASS] R4 albedo NaN sayisi = veri-yok sayisi: NaN 0, beklenen 0
--- R4 bitti, 0.0 dk

--- R5 başlad

In [29]:
# Developed with assistance from Claude (Anthropic).
# Gün sonu raporu: kayit\13_kasim_veri.md yazılır; sonuna D2.csv / D3.csv / temiz\ sayımları dosyalardan eklenir.
from pathlib import Path
import csv
from collections import Counter
from datetime import datetime

KOK = Path.home() / "orbitrary"
rapor = r'''# 13 Kasım — Veri günü raporu (D0–D3)

Orbitrary · NASA Space Apps 2026 · 13–14 Kasım gecesi · belge Bölüm 17.1–17.3
Hazırlayan: İrem (Claude yardımıyla). Bütün sayılar `kayit\` dosyalarından ve koşu çıktılarından.

## 1. Özet

| Adım | Sonuç |
|---|---|
| D0 ortam | conda `orbitrary` (conda-forge): Python 3.12, GDAL 3.13.3, rasterio 1.5.2, numpy 2.5.4, pandas 3.1.1, earthaccess 0.19.0, gudhi 3.13.0 → `kayit\ortam.json` |
| İndirme | 409 dosyadan 408. Ham 36,92 GB; zip'ler açılınca 42,79 GB. Üretici sağlaması olan her dosya tuttu. Eksik: R4 `nmsk2003.bin` (se.psi.edu iki denemede de açılmadı) |
| D1 bütünlük | 76/76 PASS |
| D2 Dünya | 18 PASS, 1 UYARI (ASTER en küçük yayınım 0,604; talimat 7) |
| D2 Ay | 14 PASS, 1 FAIL (PGDA kuzey soğuk/gölge örtüşmesi %88 < %90) |
| D2 Mars | 13/13 PASS (ilk koşuda R3 ve R5 etiketi okunamadı; okuma düzeltildi, eşik değişmedi) |
| D3 Dünya | 10/10 set PASS; `temiz\dunya` 22,55 GB |
| D3 Ay | 7/7 set PASS; `temiz\ay` 9,57 GB |
| D3 Mars | 7/7 set PASS; `temiz\mars` 1,09 GB |

D3 testi her sette: rastgele en az 1000 piksel/satırda ham dosyadan bağımsız yeniden hesaplanan değerle fark 0; NaN sayısı = veri-yok sayısı. Her klasörde `meta.json` (kaynak, DOI, sürüm, lisans, indirme tarihi, ham SHA-256, CRS, ölçek/ofset, veri-yok kuralı, D2 ve D3 testleri).

İlk koşularda kodun kendi kontrollerinin yakaladıkları (hepsi düzeltildi, kayıtta duruyor): E4 NDVI'de fill özniteliği yok; E9'da veri-yok tanımı yok; M1 ve M7'de ölçek TIF içinde de yazılı (etiketle aynı, bir kez uygulandı); M2 `AREA_GEO` birimi m² (km² değil).

## 2. MAKALEYİ ETKİLER — Emir ve Sude için

1. **R6 Mars Odyssey GRS (belge 10.3, 17.3 D1).** PDS bu seti 2026'da PDS4'e taşımış; 2013 tarihli MD5 listesi artık dosyalarla tutmuyor. Bütünlük PDS4 etiketleriyle doğrulandı (12/12). Veri sürümü 1.0; kayıt uzunluğu 72/66 bayt (eski PDS3 etiketinde 71/65). Yöntemde "PDS4 sürümü kullanıldı" denmeli.
2. **R4 Putzig TI interpolasyon maskesi (belge 13.3).** İndirilemedi. Açık kontrol açık kalıyor; Limitations'a bir cümle.
3. **E4 ASTER GED yayınımı (belge 10.1, 17.3 D2).** Ölçülen aralık 0,604–1,000 (belge 0,61 diyordu). Ofset dosyada 0,49; kuvars testi geçti. Talimat 7 gereği UYARI; makalede aralık 0,604–1,000.
4. **Diviner ↔ PGDA kalıcı gölge (belge 13.3, 17.3 D2).** En soğuk 100 PRP üçgeninin (|enlem| ≥ 85°) LPSR = 1'e düşme oranı güneyde %98, kuzeyde %88. Önceden konan %90 eşiği yüzünden kuzey FAIL; eşik değişmedi. Okuma hatası değil: aynı dönüşüm güneyde %98 veriyor ve belgedeki kuzey anormal pikseli (88,435°K 106,602°D) aynı formülle tam yerinde bulundu (LPSR 0,5, komşuları 0). Limitations'a bir cümle.
5. **E4 ASTER NDVI veri-yok kuralı (belge 10.1).** Dosyada veri-yok değeri tanımlı değil; ham 0–999, negatif yok; NDVI ofseti 0. Yayınımın 5 bandının da boş olduğu pikseller "veri yok" (ürünün kendi maskesi). Kara pikselinde NDVI 0 değer olarak kaldı ve bayraklandı (bitki yok ya da kırpılmış negatif). Bayrak sayısı ocakta ~4,0 milyon, temmuzda ~2,7 milyon (kuzey kışında kar/buz) — 0'ların gerçek "bitki yok" gibi davrandığını gösteriyor.
6. **E9 Global Aridity Index v3.1 veri-yok kuralı (belge 10.1).** Readme: değerler × 0,0001; veri-yok tanımlı değil. Okyanus 0 olarak saklanmış (ham 0'ın 550,9 milyonu su, 4,9 milyonu kara). Ham 0 ve MODIS su %100 → veri yok. Karadaki 0 (ör. Atacama) değer olarak kaldı, bayrak 1. 1,2 milyon pikselde uint16 tavanı 65535 (AI 6,5535), çoğu Grönland ve 40–60°G buz alanlarında → bayrak 2 "kırpılmış". Limitations'a bir cümle.

## 3. Belge 13.3 açık kontrollerin durumu

| Açık kontrol | Durum |
|---|---|
| TES bant 3/5/6 sırası | Kapandı: ASU ürün sayfasındaki liste = belge D11. Bant 8 (hematit) ekvator maksimumu Opportunity iniş noktasına 120 km (Meridiani) |
| PGDA kuzey soğuk/gölge oranı | Kapandı: kuzey 0,88, güney 0,98 (madde 4) |
| Putzig TI interpolasyon maskesi | Açık: dosya indirilemedi (madde 2) |
| GLiM pb sınıf kapsamı | Sude'de |

## 4. Talimat 6 ve 7

**Talimat 6 (PRP ice_depth).** Negatif değerler yalnız {−1, −999}; −1 olan bütün üçgenlerde temp_max ≤ 101,3 K (iki yarıda da en büyük 101,299 K). D2 ve D3'te aynı sayılar:

| | 0 | −1 | pozitif | −999 |
|---|---|---|---|---|
| Kuzey | 14 457 | 65 481 | 989 573 | 1 810 489 |
| Güney | 18 375 | 100 319 | 913 867 | 1 847 439 |

temiz'de: −999 → NaN + bayrak 1 (sansürlü, > 2,87 m); −1 değer olarak kalır + bayrak 2.

**Talimat 7 (ASTER ofseti).** `Offset` HDF5 özniteliğinde 0,49 (dosyadan, assert edildi). Kuvars testi PASS. En küçük yayınım 0,604 → UYARI, iş sürdü.

## 5. Okuma notları (iç; makaleye gerekmez)

- TES mineral ve DCI dosyalarında harita konumu yok; ASU ürün tanımından alındı (simple cylindrical, −180 → 180 doğu, 90 → −90). Hematit (Meridiani) ve DCI–albedo (r = −0,84) testleriyle doğrulandı. TES .envi (>f4) ile .isis (<f4) birebir aynı.
- TES TI temiz'de kuzey üstte, −180'den doğuya düzene çevrildi (ham: ilk satır güney, boylam batıya); dönüşüm 1000 noktada test edildi.
- M7 anormal piksel yalnız LPSR'de anormal; belge ürün belirtmediği için iki üründe de NaN yapıldı (AVGVISIB'de 1 piksel, değeri 0,280 / komşu ortancası 0,282).
- R1 MOLA: transform kullanılmaz; konum belgedeki satır/sütun kuralıyla (meta.json'da).
- E1 ETOPO'da veri-yok yok (0); E7/E8 permafrost 0–1; E10 kil 8–796 ‰, silt 3–900 ‰.

## 6. Dosyalar

- `kod\`: d0_ortam.py, liste_denetim.py, indir_hepsi.py, d1_butunluk.py, d2_dunya.py, d2_ay.py, d2_mars.py, d3_dunya.py, d3_ay.py, d3_mars.py, 13_kasim_veri.ipynb
- `kayit\`: ortam.json, rapor_sha256.txt, liste_denetim.json, indirme.csv, indirme_ozet.json, E1_karolar.csv, D1.json, D1_dosyalar.csv, D2.csv, D2_okuma_dunya/ay/mars.json, D3.csv, R6_inceleme.json, R6_satir_sonu.json, R6_karar.json, 13_kasim_log.txt, bu rapor
- `temiz\<govde>\<kod>_<ad>\` + `meta.json` (24 set). 14 Kasım kodu yalnız `temiz\`'i okur.
'''

# ---- dosyalardan sayım (raporla tutmalı)
def say(yol):
    with open(yol, encoding="utf-8", newline="") as f:
        return Counter((r["govde"], r["durum"]) for r in csv.DictReader(f))
d2, d3 = say(KOK / "kayit/D2.csv"), say(KOK / "kayit/D3.csv")
metalar = sorted(p.parent.relative_to(KOK / "temiz") for p in (KOK / "temiz").rglob("meta.json"))
boyut = {g: sum(f.stat().st_size for f in (KOK / "temiz" / g).rglob("*") if f.is_file()) / 1e9 for g in ("dunya", "ay", "mars")}
ek = ["", f"## 7. Dosyalardan sayım ({datetime.now():%Y-%m-%d %H:%M})", ""]
ek.append("| Gövde | D2 | D3 | temiz (GB) |")
ek.append("|---|---|---|---|")
for g in ("dunya", "ay", "mars"):
    f2 = ", ".join(f"{k[1]} {v}" for k, v in sorted(d2.items()) if k[0] == g)
    f3 = ", ".join(f"{k[1]} {v}" for k, v in sorted(d3.items()) if k[0] == g)
    ek.append(f"| {g} | {f2} | {f3} | {boyut[g]:.2f} |")
ek.append("")
ek.append(f"meta.json sayısı: {len(metalar)} — " + ", ".join(str(m).replace(chr(92), '/') for m in metalar))
metin = rapor + "\n".join(ek) + "\n"
(KOK / "kayit" / "13_kasim_veri.md").write_text(metin, encoding="utf-8")
with open(KOK / "kayit" / "13_kasim_log.txt", "a", encoding="utf-8") as f:
    f.write(f"{datetime.now():%Y-%m-%d %H:%M:%S}  gün sonu raporu yazıldı: kayit/13_kasim_veri.md; meta.json {len(metalar)}\n")
print("\n".join(ek))
assert len(metalar) == 24, f"meta.json {len(metalar)} (beklenen 24)"
assert all(k[1] == "PASS" for k in d3), f"D3.csv'de PASS olmayan satır var: {d3}"
print("\nYazıldı:", KOK / "kayit" / "13_kasim_veri.md")


## 7. Dosyalardan sayım (2026-11-14 02:37)

| Gövde | D2 | D3 | temiz (GB) |
|---|---|---|---|
| dunya | PASS 18, UYARI 1 | PASS 27 | 22.55 |
| ay | FAIL 1, PASS 14 | PASS 41 | 9.57 |
| mars | PASS 13 | PASS 23 | 1.09 |

meta.json sayısı: 24 — ay/M1_lola, ay/M2_jeoloji, ay/M3_diviner_ghrm, ay/M4_diviner_prp_gcp, ay/M5_lp_grs, ay/M6_lp_hidrojen, ay/M7_pgda, dunya/E10_soilgrids, dunya/E1_etopo, dunya/E2_mcd12c1, dunya/E3_glim, dunya/E4_aster_ged, dunya/E5_myd11c3, dunya/E6_mcd43c3, dunya/E7_permafrost_kuzey, dunya/E8_permafrost_antarktika, dunya/E9_aridity, mars/R1_mola, mars/R2_sim3292, mars/R3_tes_mineral, mars/R4_tes_ti_albedo, mars/R5_tes_dci, mars/R6_odyssey_grs, mars/R7_swim

Yazıldı: C:\Users\USER\orbitrary\kayit\13_kasim_veri.md
